# Laboratorio GPT-2: ¿podemos hacer que un modelo escriba un poco más como el autor que elegimos?

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## El desafío

Vas a tomar un modelo que ya sabe escribir en español, elegir los textos de un autor (por defecto, cuentos de Horacio Quiroga) y ajustar el modelo con ellos. Después vas a comparar lo que escribe antes y después, y a revisar qué evidencia tenés de que algo cambió. No sabemos de antemano si va a pasar: la pregunta está abierta, y el laboratorio sirve para probarla.

Lo que se hace se llama **ajuste fino** (*fine-tuning*): seguir entrenando un modelo ya entrenado (`DeepESP/gpt2-spanish`). No se entrena un modelo desde cero ni se configura un GPT dentro de ChatGPT.

## El recorrido

| Estación | Qué hacés | Qué vas a ver |
|---|---|---|
| **1. Probá el modelo base** | Das un comienzo de frase y movés los controles de la generación | Cómo continúa el texto y qué cambia con cada control |
| **2. Elegí y revisá los textos** | Mirás cuánto pesa cada texto, sacás o sumás documentos y probás limpiezas | El corpus antes y después de tus decisiones |
| **3. Mirá cómo el modelo lee** | Recorrés un bloque de texto convertido en números | Qué recibe realmente el modelo al ajustarse |
| **4. Configurá el ajuste** | Movés los controles del entrenamiento | Cuántos pasos, bloques y cuánto tiempo implican tus decisiones |
| **5. Observá el ajuste** | Lanzás el ajuste y lo mirás mientras corre | La curva de la pérdida y lo que escribe el modelo en el camino |
| **6. Compará y examiná la evidencia** | Ponés lado a lado el modelo base y el ajustado | Qué semejanzas aparecen y qué evidencia las respalda |
| **7. Guardá el experimento** | Dejás todo registrado | Una carpeta que otra persona puede abrir y repetir |

## Cómo se trabaja

- **Primero probás, después leés.** Cada estación empieza con un desafío corto y un panel con controles. Las explicaciones vienen después, sobre lo que acabás de ver.
- **Celdas de preparación.** Están marcadas como **Preparación**. Hay que ejecutarlas, pero no hace falta leerlas: su código aparece plegado si tu entorno lo permite, y se despliega con la flecha de la celda.
- **El código también está.** Todo lo que hacés con un panel se puede hacer con código: cada estación termina con su «Vía con código».
- **«Ejecutar todo» no entrena.** Las operaciones largas están detrás de cuatro banderas `PERMITIR_...` que arrancan en `False`. Con ese valor el cuaderno prepara todo, deja los paneles listos y no escribe experimentos.

### Al terminar este cuaderno vas a poder:

1. Generar texto con un modelo causal y explicar qué cambian la temperatura, `top_k`, `top_p` y la semilla.
2. Decidir qué textos entran al corpus, probar limpiezas y comparar el antes y el después.
3. Mostrar con tokens reales cómo se arman los bloques con los que se ajusta el modelo.
4. Configurar, lanzar y detener un ajuste, y leer sus curvas sin confundir pérdida con calidad literaria.
5. Comparar el modelo base con el ajustado, con la misma entrada y la misma configuración, y registrar qué observás y qué interpretás.
6. Guardar un experimento y reconstruir la comparación sin depender de los controles de esta sesión.

## ◈ Marco contextual

**Analogía.** Pensá en alguien que habla español con soltura porque leyó muchísimo, y a quien le pedimos que pase unas semanas leyendo solo cuentos de un autor. Cuando vuelva a escribir, puede que se le noten giros, palabras y ritmos de esas lecturas. Un ajuste fino hace algo parecido: el modelo sigue entrenándose, pero con un corpus más acotado. La analogía tiene un límite: el modelo no «lee» ni «entiende»; modifica sus *pesos* (los números donde guarda lo que aprendió) para que ciertas secuencias de tokens le resulten más probables.

**Bajada tecnológica.** Se llama *ajuste fino* (*fine-tuning*) de un modelo de lenguaje causal. Es la técnica detrás de muchos modelos «de dominio» (jurídicos, médicos, de estilo) y una de las etapas de los asistentes actuales. Se hace con la librería `transformers` de Hugging Face sobre PyTorch. En la unidad de transfer learning viste la adaptación de dominio y, en la introducción a los LLMs, cómo se genera texto; acá abrimos cada paso.

## ◈ Microglosario

| Término | Qué significa en este cuaderno |
|---|---|
| **Token** | Cada pieza en que se corta un texto para que el modelo lo lea: una palabra, un pedazo de palabra o un signo |
| **Corpus** | El conjunto de textos con el que ajustás el modelo. Lo que aprenda va a salir de ahí |
| **Ajuste fino** | Seguir entrenando un modelo ya entrenado con un corpus propio |
| **Bloque** | Un tramo de largo fijo de tokens consecutivos: lo que se le da al modelo en cada paso |
| **Pérdida** | Cuánto «le sorprendió» al modelo el texto real, en promedio. Menor pérdida significa que lo predijo mejor, no que el texto sea mejor |
| **Checkpoint** | Una copia de los pesos del modelo guardada en un momento del ajuste, para poder volver a usarla |

Cada término se vuelve a explicar en la estación donde lo usás.

---
## Preparación general

**Preparación.** Ejecutá las tres celdas que siguen: importan las herramientas, fijan los parámetros del cuaderno y cargan el modelo base. La primera vez, el modelo se descarga (unos 250 MB). Dos corridas con la misma semilla pueden dar textos distintos si cambian las versiones de las librerías o el dispositivo, por eso la primera celda las muestra.

In [ ]:
# Librerías de la biblioteca estándar de Python.
import copy
import dataclasses
import difflib
import gc
import hashlib
import json
import math
import random
import re
import threading
import time
from pathlib import Path

# Librerías de análisis y de modelos.
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

# Librerías de interfaz.
import ipywidgets as widgets
from IPython.display import HTML, Image, clear_output, display

# laboratorio.py y utils.py están en esta carpeta. No contienen algoritmos de la unidad:
# laboratorio.py arma controles y archivos; utils.py dibuja.
import laboratorio
import utils

print(f"torch {torch.__version__} · transformers {transformers.__version__} · pandas {pd.__version__}")

### Parámetros del cuaderno

Están todos en la celda siguiente, a la vista. Los que más vas a tocar son las cuatro banderas `PERMITIR_...`: encienden operaciones largas y por eso arrancan apagadas.

| Parámetro | Qué es | Qué cambia si lo modificás |
|---|---|---|
| `MODELO_BASE` y `REVISIÓN_DEL_MODELO` | El modelo de partida y la versión exacta de sus pesos | Otro modelo puede tener otro tokenizador, y entonces las métricas dejan de ser comparables |
| `DISPOSITIVO` | Dónde se calcula: `mps` (GPU de Mac), `cuda` (GPU NVIDIA) o `cpu` | La velocidad y pequeñas diferencias numéricas; no lo que el modelo calcula en principio |
| `RAÍZ_DE_EXPERIMENTOS` | La carpeta donde queda todo lo que guardás | Dónde se buscan y se guardan los experimentos |
| `PERMITIR_ACTIVIDAD_DE_GENERACIÓN` | La actividad de la estación 1 (decenas de generaciones seguidas) | En `True`, esa celda genera cada vez que se ejecuta |
| `PERMITIR_ENSAYO_DE_TIEMPO` | Un entrenamiento de prueba (estación 4) para medir cuánto tarda tu máquina | En `True`, entrena unos pocos microlotes con una copia del modelo |
| `PERMITIR_AJUSTE_POR_CÓDIGO` | La vía con código de la estación 5, que entrena de verdad | En `True`, entrena y escribe en `experimentos/` (minutos y unos 2 GB en disco) |
| `PERMITIR_ANALIZAR_CHECKPOINTS` | Las vías con código de las estaciones 6 y 7, que cargan checkpoints | En `True`, cargan modelos (unos 500 MB cada uno) y la exportación escribe `FICHA.md` |
| `EXPERIMENTO_A_USAR` | El experimento con el que trabajan las vías con código de las estaciones 5 a 7 | `None` toma el más reciente que terminó; si no hay ninguno, esas celdas avisan y siguen |

Los paneles no necesitan ninguna bandera: funcionan con las cuatro en `False`.

In [ ]:
MODELO_BASE = "DeepESP/gpt2-spanish"
REVISIÓN_DEL_MODELO = "1b935e39cf9893108bd2f4fb5317f48ae1c3ab5e"
DISPOSITIVO = torch.accelerator.current_accelerator() or torch.device("cpu")
RAÍZ_DE_EXPERIMENTOS = Path("experimentos")

# Operaciones largas u opcionales. Con False, «Ejecutar todo» las salta y el cuaderno termina en pocos minutos.
PERMITIR_ACTIVIDAD_DE_GENERACIÓN = False # estación 1: decenas de generaciones seguidas
PERMITIR_ENSAYO_DE_TIEMPO = False         # estación 4: entrena unos pocos microlotes con una copia del modelo
PERMITIR_AJUSTE_POR_CÓDIGO = False        # estación 5: entrena de verdad (minutos y unos 2 GB en disco)
PERMITIR_ANALIZAR_CHECKPOINTS = False     # estaciones 6 y 7: cargan checkpoints, generan texto y escriben FICHA.md

# Experimento con el que trabajan las vías con código de las estaciones 5 a 7. None = el más reciente que terminó.
EXPERIMENTO_A_USAR = None

print(f"Modelo base: {MODELO_BASE}")
print(f"Dispositivo: {DISPOSITIVO}")

**Mirá el resultado.** Tiene que aparecer el dispositivo elegido. Con `cpu` todo funciona igual, solo que el ajuste tarda más (en la estación 4 podés medir cuánto). La línea de `DISPOSITIVO` solo le pregunta a PyTorch qué acelerador hay; para forzar la CPU, reemplazala por `torch.device("cpu")`.

### Cargar el modelo base y su tokenizador

El **tokenizador** corta los textos en tokens y el **modelo** son los pesos ya entrenados. Los dos se traen de Hugging Face, o de la caché local si ya los bajaste (ver `ANTES_DE_LA_CLASE.md`). Si es la primera vez, requiere conexión.

In [ ]:
tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE, revision=REVISIÓN_DEL_MODELO)

modelo_base = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    revision=REVISIÓN_DEL_MODELO,
    dtype=torch.float32,
)
modelo_base = modelo_base.to(DISPOSITIVO)
modelo_base.eval()  # modo de uso: apaga el dropout (el apagado aleatorio de neuronas)

cantidad_de_parámetros = 0
for parámetro in modelo_base.parameters():
    cantidad_de_parámetros += parámetro.numel()
print(f"Parámetros del modelo: {utils.con_miles(cantidad_de_parámetros)}")
print(f"Vocabulario del tokenizador: {utils.con_miles(len(tokenizador))} tokens")
print(f"Largo máximo de contexto: {modelo_base.config.n_positions} tokens")
print(f"Token de fin de texto: {tokenizador.eos_token!r} (id {tokenizador.eos_token_id})")

**Mirá el resultado.** Unos 124 millones de parámetros (los pesos), un vocabulario de 50.257 tokens y un contexto de hasta 1024 tokens: ni lo que escribas ni lo que genere el modelo pueden pasar de ahí.

<details>
<summary>Profundización: el aviso de la carga, <code>float32</code> y lo que sabemos del modelo</summary>

- **El aviso `UNEXPECTED ... attn.masked_bias`.** Es normal: es un dato que el checkpoint original guardó y la versión actual de `transformers` ya no usa. En nuestras pruebas no afectó al modelo (la estación 3 comprueba que la pérdida calculada a mano coincide con la del modelo), pero es una comprobación para este modelo y esta versión, no una garantía general.
- **Por qué `float32`.** El checkpoint está guardado en `float16` y, con la versión 5 de `transformers`, se carga así si no se pide otro formato. Entrenar en `float16` con el optimizador que usamos produjo valores `nan` en una prueba nuestra (la pérdida deja de ser un número en el primer paso). Por eso lo pedimos explícitamente en `float32`.
- **Qué sabemos del modelo.** Según su ficha (`docs/MODELO.md`), se entrenó desde cero con unos 11,5 GB de textos en español (Wikipedia y libros), con licencia MIT. No sabemos qué libros concretos leyó, y la ficha advierte que el corpus no fue filtrado.
</details>

**Preparación.** La celda siguiente deja lista la **bitácora**: un registro donde, cuando quieras, anotás qué viste y qué interpretás. Va a aparecer en varios momentos del recorrido; anotar nunca es obligatorio.

In [ ]:
bitácora_general = laboratorio.Bitácora(RAÍZ_DE_EXPERIMENTOS / "bitácora.jsonl")


def nombres_de_experimentos():
    """Nombres de las carpetas de experimentos guardados, del más nuevo al más viejo."""
    nombres = []
    for carpeta in laboratorio.listar_experimentos(RAÍZ_DE_EXPERIMENTOS):
        nombres.append(carpeta.name)
    return nombres


def armar_bitácora_visual(estación_inicial="7-guardado", lugar="7"):
    """Panel de la bitácora: agrega entradas y lista todas las guardadas.

    `estación_inicial` es la estación que aparece elegida; `lugar` distingue los paneles que se muestran en
    distintos puntos del recorrido, para que mostrar uno no apague a los otros.
    """
    estación = laboratorio.nueva_estación(f"bitácora-{lugar}")
    cuál = estación.agregar(widgets.Dropdown(options=["1-explorar", "2-corpus", "3-bloques", "4-diseño", "5-entrenamiento",
                                                       "6a-comparador", "6b-siguiente-token", "6c-evaluación", "7-guardado"], value=estación_inicial))
    laboratorio.control(cuál, "Estación", "estación", ancho="24em", tipo="registro",
                        ayuda="A qué parte del recorrido corresponde lo que anotás.")
    tipo = estación.agregar(widgets.Dropdown(options=list(laboratorio.TIPOS_DE_ANOTACIÓN), value="observado"))
    laboratorio.control(tipo, "Tipo", "tipo", ancho="24em", tipo="registro",
                        ayuda="Observado: lo que ves. Interpretación: lo que creés que lo explica. Limitación: lo que lo debilita. Hipótesis pendiente: lo que habría que probar.",
                        prueba="anotá primero lo que ves y, aparte, lo que creés que lo explica.")
    opciones_de_experimento = [("(ninguno)", None)]
    for nombre_del_experimento in nombres_de_experimentos():
        opciones_de_experimento.append((nombre_del_experimento, nombre_del_experimento))
    experimento = estación.agregar(widgets.Dropdown(options=opciones_de_experimento, value=None))
    laboratorio.control(experimento, "Experimento al que se refiere", "experimento", ancho="36em", tipo="registro",
                        ayuda="Si elegís uno, la anotación queda asociada a él y la estación 7 la copia a su carpeta.")
    predicción = estación.agregar(widgets.Textarea(rows=2, layout=widgets.Layout(width="100%")))
    predicción.description = "Predicción"
    texto = estación.agregar(widgets.Textarea(rows=3, layout=widgets.Layout(width="100%")))
    texto.description = "Texto"
    botón_guardar = estación.agregar(laboratorio.botón("Guardar en la bitácora", "Queda en bitácora.jsonl", "success"))
    lista = estación.agregar(widgets.HTML())

    def mostrar():
        filas = []
        for entrada in bitácora_general.leer():
            filas.append([entrada["hora"][11:], entrada["estación"], entrada["tipo"], entrada["texto"]])
        lista.value = utils.html_tabla(["hora", "estación", "tipo", "texto"], filas)

    def al_guardar(_):
        referidos = []
        if experimento.value:
            referidos.append(experimento.value)
        try:
            bitácora_general.anotar(cuál.value, tipo.value, texto.value, predicción=predicción.value, experimentos=referidos)
        except ValueError as error:
            lista.value = utils.html_aviso(str(error), "error")
            return
        texto.value = ""
        mostrar()

    estación.al_hacer_clic(botón_guardar, al_guardar)
    mostrar()
    return widgets.VBox([laboratorio.fila_de_control(cuál), laboratorio.fila_de_control(tipo), laboratorio.fila_de_control(experimento),
                         predicción, texto, botón_guardar, lista])

---
## Estación 1. Probá cómo escribe el modelo base

**Desafío.** Escribí el comienzo de una frase, pedile al modelo que la siga y mirá qué hace. Después cambiá **un solo control** y probá de nuevo: ¿qué cambia en el texto?

**Preparación · estación 1.** Ejecutá la celda siguiente: define `generar` (la función que llama al modelo) y arma el panel. Su código aparece plegado si tu entorno lo permite (desplegalo con la flecha si querés leerlo).

In [ ]:
ejecución_actual = {"hilo": None, "parar": threading.Event(), "estado": "pendiente", "resultado": None, "error": None}


def exigir_dispositivo_libre():
    """Levanta un RuntimeError si un ajuste corre en otro hilo.

    La GPU no tolera que dos hilos la usen a la vez: en una prueba en el navegador, calcular candidatos
    del siguiente token mientras corría un ajuste cerró el kernel (error de memoria en `mps`). Por eso
    generar, medir y cargar modelos esperan a que el ajuste termine o se detenga.
    """
    hilo = ejecución_actual["hilo"]
    if hilo is not None and hilo.is_alive() and threading.current_thread() is not hilo:
        raise RuntimeError("Hay un ajuste en curso. Usar un modelo ahora puede colgar o cerrar el kernel: "
                           "esperá a que termine o apretá Detener (estación 5).")


def validar_configuración_de_generación(comienzo, modo, temperatura, top_k, top_p, max_new_tokens, n_muestras):
    """Revisa los parámetros de generación y levanta un ValueError claro si alguno no tiene sentido."""
    if modo not in ("muestreo", "determinista"):
        raise ValueError(f"El modo tiene que ser 'muestreo' o 'determinista', no {modo!r}.")
    if modo == "muestreo" and not temperatura > 0:
        raise ValueError("La temperatura tiene que ser mayor que 0 (con 0 se divide por cero).")
    if modo == "muestreo" and not 0 < top_p <= 1:
        raise ValueError("top_p tiene que estar entre 0 (sin incluirlo) y 1.")
    if modo == "muestreo" and top_k < 0:
        raise ValueError("top_k no puede ser negativo (0 significa «sin límite»).")
    if max_new_tokens < 1:
        raise ValueError("max_new_tokens tiene que ser al menos 1.")
    if n_muestras < 1:
        raise ValueError("Pedí al menos una muestra.")


def generar(modelo, tokenizador, comienzo, n_muestras=3, max_new_tokens=60, modo="muestreo",
            temperatura=0.8, top_k=50, top_p=0.95, semilla=42, procedencia="base"):
    """Genera `n_muestras` continuaciones de `comienzo` y devuelve una lista de diccionarios.

    Cada diccionario trae la entrada, la continuación y la configuración efectiva: lo que
    realmente se usó, no lo que se pidió. En modo determinista hay una sola muestra,
    porque todas serían iguales, y temperatura, top_k, top_p y semilla no se usan.
    """
    exigir_dispositivo_libre()
    validar_configuración_de_generación(comienzo, modo, temperatura, top_k, top_p, max_new_tokens, n_muestras)
    if modo == "determinista":
        n_muestras = 1

    # Un comienzo vacío no se puede tokenizar a nada: arrancamos con el token de fin de texto.
    if comienzo == "":
        identificadores = [tokenizador.eos_token_id]
    else:
        identificadores = tokenizador(comienzo)["input_ids"]
    largo_de_la_entrada = len(identificadores)
    if largo_de_la_entrada + max_new_tokens > modelo.config.n_positions:
        raise ValueError(
            f"La entrada ({largo_de_la_entrada} tokens) más max_new_tokens ({max_new_tokens}) "
            f"supera el contexto del modelo ({modelo.config.n_positions})."
        )

    entrada = torch.tensor([identificadores], device=modelo.device)
    muestras = []
    for número in range(n_muestras):
        semilla_de_la_muestra = semilla + número  # cada muestra se puede repetir por separado
        argumentos = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizador.eos_token_id)
        if modo == "muestreo":
            torch.manual_seed(semilla_de_la_muestra)
            argumentos.update(do_sample=True, temperature=temperatura, top_k=top_k, top_p=top_p)
        else:
            argumentos.update(do_sample=False)
        with torch.no_grad():
            salida = modelo.generate(entrada, **argumentos)
        tokens_nuevos = salida[0][largo_de_la_entrada:].tolist()
        terminó_en_fin_de_texto = tokenizador.eos_token_id in tokens_nuevos
        muestras.append({
            "entrada": comienzo,
            "continuación": tokenizador.decode(tokens_nuevos),
            "tokens_nuevos": len(tokens_nuevos),
            "terminó_en_fin_de_texto": terminó_en_fin_de_texto,
            "configuración": {
                "modelo": procedencia,
                "modo": modo,
                "temperatura": temperatura if modo == "muestreo" else None,
                "top_k": top_k if modo == "muestreo" else None,
                "top_p": top_p if modo == "muestreo" else None,
                "max_new_tokens": max_new_tokens,
                "semilla": semilla_de_la_muestra if modo == "muestreo" else None,
                "dispositivo": str(modelo.device),
                "torch": torch.__version__,
                "transformers": transformers.__version__,
            },
        })
    return muestras


def describir_diferencias(configuración_a, configuración_b):
    """Devuelve las filas (parámetro, A, B, ¿cambió?) y los avisos de una comparación de dos corridas."""
    filas = []
    cantidad_de_cambios = 0
    for nombre in configuración_a:
        valor_a = configuración_a[nombre]
        valor_b = configuración_b.get(nombre)
        cambió = valor_a != valor_b
        if cambió:
            cantidad_de_cambios += 1
        filas.append([nombre, str(valor_a), str(valor_b), "cambió" if cambió else "igual"])
    avisos = []
    if cantidad_de_cambios > 1:
        avisos.append("Cambió más de un parámetro: si ves una diferencia, no sabés a cuál atribuirla.")
    avisos.append("Pocas muestras no prueban un efecto general: sirven para formular una hipótesis, no para confirmarla.")
    return filas, avisos


registro_de_corridas = laboratorio.RegistroJSONL(RAÍZ_DE_EXPERIMENTOS / "exploración" / "corridas.jsonl")
corridas_de_esta_sesión = []  # cada elemento: {"número", "entrada", "muestras"}


def armar_estación_1():
    """Crea los controles de la estación 1 y los conecta. Devuelve el panel de tres pestañas."""
    estación = laboratorio.nueva_estación("1-explorar")

    comienzo = estación.agregar(widgets.Textarea(value="Una mañana de otoño, el jaguar", rows=3))
    modo = estación.agregar(widgets.ToggleButtons(options=["muestreo", "determinista"], value="muestreo"))
    temperatura = estación.agregar(widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.05, continuous_update=False))
    top_k = estación.agregar(widgets.IntSlider(value=50, min=0, max=200, continuous_update=False))
    top_p = estación.agregar(widgets.FloatSlider(value=0.95, min=0.05, max=1.0, step=0.05, continuous_update=False))
    máximo = estación.agregar(widgets.IntSlider(value=40, min=5, max=200, continuous_update=False))
    cantidad = estación.agregar(widgets.IntSlider(value=3, min=1, max=6, continuous_update=False))
    semilla = estación.agregar(widgets.IntText(value=42))
    laboratorio.control(comienzo, "Comienzo del texto", "comienzo", tipo="experimento",
                        ayuda="Lo que escribís vos; el modelo sigue desde ahí.",
                        prueba="dejalo fijo y cambiá un solo control: así comparás desde el mismo punto de partida.")
    laboratorio.control(modo, "Cómo elige", "modo", ancho="34em", tipo="experimento",
                        ayuda="«muestreo» sortea entre los candidatos; «determinista» elige siempre el más probable.",
                        prueba="generá dos veces en determinista y mirá si el texto cambia.")
    laboratorio.control(temperatura, "Temperatura", "temperatura", tipo="experimento",
                        ayuda="Reparte o concentra las probabilidades antes de sortear.",
                        prueba="0,3 y 1,5 con la misma semilla; mirá cuánto se parecen las muestras entre sí.")
    laboratorio.control(top_k, "Candidatos máximos (0 = sin límite)", "top_k", tipo="experimento",
                        ayuda="Deja en juego solo a los k tokens más probables.",
                        prueba="poné 1 y generá varias muestras: ¿siguen siendo distintas?")
    laboratorio.control(top_p, "Probabilidad acumulada", "top_p", tipo="experimento",
                        ayuda="Deja en juego al grupo más chico de candidatos cuya probabilidad suma p.",
                        prueba="bajá a 0,3 y compará con 1,0.")
    laboratorio.control(máximo, "Tokens nuevos como máximo", "max_new_tokens", tipo="experimento",
                        ayuda="Cuánto escribe el modelo (un token es una pieza de texto, no siempre una palabra).",
                        prueba="subí a 120 y mirá si el tema se sostiene hasta el final.")
    laboratorio.control(cantidad, "Cantidad de muestras", "n_muestras", tipo="experimento",
                        ayuda="Cuántas continuaciones del mismo comienzo; una sola puede ser casualidad.",
                        prueba="pedí 5 y marcá qué se repite entre ellas y qué cambia.")
    laboratorio.control(semilla, "Semilla del sorteo", "semilla", tipo="experimento",
                        ayuda="Fija el sorteo: misma semilla y mismos controles, mismo texto (en esta máquina).",
                        prueba="cambiá solo la semilla y compará con lo que cambió al mover la temperatura.")

    botón_generar = estación.agregar(laboratorio.botón("Generar", "Llama al modelo con esta configuración", "primary"))
    botón_comparar = estación.agregar(laboratorio.botón("Comparar las dos últimas corridas", "Tabla de diferencias entre configuraciones"))
    botón_restaurar = estación.agregar(laboratorio.botón("Volver a los valores iniciales", "Solo mueve los controles; las corridas quedan"))
    corrida_anterior = estación.agregar(widgets.Dropdown(options=[]))
    laboratorio.control(corrida_anterior, "Corrida", "número", ancho="22em", tipo="vista",
                        ayuda="Cuál de las corridas de esta sesión querés recuperar.",
                        prueba="elegí la 1 y apretá «Usar la configuración de esa corrida».")
    botón_cargar = estación.agregar(laboratorio.botón("Usar la configuración de esa corrida", "Pone en los controles lo que se usó en esa corrida"))
    estado = estación.agregar(widgets.HTML(utils.html_estado("pendiente")))
    resultados = estación.agregar(widgets.VBox([]))
    inspección = estación.agregar(widgets.HTML("<i>Todavía no hay corridas.</i>"))
    código = estación.agregar(widgets.HTML())

    estación.recordar_valores_iniciales(modo, comienzo, temperatura, top_k, top_p, máximo, cantidad, semilla)

    def actualizar_coherencia(_=None):
        es_muestreo = modo.value == "muestreo"
        for control in (temperatura, top_k, top_p, semilla, cantidad):
            control.disabled = not es_muestreo
        if not es_muestreo:
            cantidad.value = 1

    def actualizar_código(_=None):
        argumentos = dict(comienzo=comienzo.value, n_muestras=cantidad.value, max_new_tokens=máximo.value,
                          modo=modo.value, temperatura=temperatura.value, top_k=top_k.value,
                          top_p=top_p.value, semilla=semilla.value)
        llamada = laboratorio.texto_de_llamada("generar", dict(modelo=laboratorio.Código("modelo_base"),
                                                              tokenizador=laboratorio.Código("tokenizador"), **argumentos), "muestras")
        código.value = f"<pre style='font-family:{utils.FUENTE_CODIGO}'>{utils.escapar(llamada)}</pre>"

    def actualizar_corridas_anteriores():
        números = []
        for corrida in corridas_de_esta_sesión:
            números.append(corrida["número"])
        corrida_anterior.options = números
        if números:
            corrida_anterior.value = números[-1]

    def al_restaurar(_):
        estación.restaurar_valores_iniciales()
        actualizar_coherencia()
        actualizar_código()

    def al_cargar_configuración(_):
        for corrida in corridas_de_esta_sesión:
            if corrida["número"] != corrida_anterior.value:
                continue
            configuración = corrida["muestras"][0]["configuración"]
            comienzo.value = corrida["entrada"]
            modo.value = configuración["modo"]
            máximo.value = configuración["max_new_tokens"]
            if configuración["modo"] == "muestreo":
                temperatura.value = configuración["temperatura"]
                top_k.value = configuración["top_k"]
                top_p.value = configuración["top_p"]
                cantidad.value = len(corrida["muestras"])
                semilla.value = configuración["semilla"]
        actualizar_coherencia()
        actualizar_código()

    def al_generar(_):
        with laboratorio.Ocupado([botón_generar, botón_comparar], estado, "Generando", utils.html_estado) as ocupado:
            nuevas = generar(modelo=modelo_base, tokenizador=tokenizador, comienzo=comienzo.value,
                             n_muestras=cantidad.value, max_new_tokens=máximo.value, modo=modo.value,
                             temperatura=temperatura.value, top_k=top_k.value, top_p=top_p.value,
                             semilla=semilla.value)
            número = len(corridas_de_esta_sesión) + 1
            corridas_de_esta_sesión.append({"número": número, "entrada": comienzo.value, "muestras": nuevas})
            registro_de_corridas.agregar({"número": número, "muestras": nuevas})
            actualizar_corridas_anteriores()
            tarjetas = [widgets.HTML(f"<b>Corrida {número}</b>")]
            for muestra in nuevas:
                tarjetas.append(widgets.HTML(utils.html_texto_con_continuación(muestra["entrada"], muestra["continuación"], "base")))
            resultados.children = tuple(tarjetas) + tuple(resultados.children)
            configuración = nuevas[0]["configuración"]
            inspección.value = utils.html_tabla(["parámetro", "valor efectivo"], utils.filas_de_diccionario(configuración))
        if ocupado.error is not None:
            resultados.children = (widgets.HTML(utils.html_aviso(str(ocupado.error), "error")),) + tuple(resultados.children)

    def al_comparar(_):
        if len(corridas_de_esta_sesión) < 2:
            inspección.value = utils.html_aviso("Hacen falta al menos dos corridas.")
            return
        configuración_a = corridas_de_esta_sesión[-2]["muestras"][0]["configuración"]
        configuración_b = corridas_de_esta_sesión[-1]["muestras"][0]["configuración"]
        filas, avisos = describir_diferencias(configuración_a, configuración_b)
        entradas_distintas = corridas_de_esta_sesión[-2]["entrada"] != corridas_de_esta_sesión[-1]["entrada"]
        if entradas_distintas:
            avisos.insert(0, "Los comienzos son distintos.")
        inspección.value = (utils.html_tabla(["parámetro", "corrida anterior", "última", ""], filas)
                            + utils.html_avisos(avisos))

    for control in (modo, temperatura, top_k, top_p, máximo, cantidad, semilla, comienzo):
        estación.observar(control, actualizar_código)
    estación.observar(modo, actualizar_coherencia)
    estación.al_hacer_clic(botón_generar, al_generar)
    estación.al_hacer_clic(botón_comparar, al_comparar)
    estación.al_hacer_clic(botón_restaurar, al_restaurar)
    estación.al_hacer_clic(botón_cargar, al_cargar_configuración)
    actualizar_corridas_anteriores()
    actualizar_coherencia()
    actualizar_código()

    filas_de_controles = []
    for control_de_la_estación in (comienzo, modo, temperatura, top_k, top_p, máximo, cantidad, semilla):
        filas_de_controles.append(laboratorio.fila_de_control(control_de_la_estación))
    volver_a_una_anterior = laboratorio.sección_plegable("Volver a una configuración anterior", widgets.VBox([laboratorio.fila_de_control(corrida_anterior), botón_cargar]))
    explorar = widgets.VBox(filas_de_controles + [widgets.HBox([botón_generar, botón_comparar, botón_restaurar]),
                                                  volver_a_una_anterior, estado, resultados])
    return laboratorio.panel_de_tres_niveles(explorar, widgets.VBox([inspección]), widgets.VBox([código]))

### Panel: generar y comparar

Cada control lleva una etiqueta que dice qué mueve: **Experimento** (cambia lo que se genera), **Vista** (solo cambia cómo se ve), **Datos** (cambia lo que se prepara) o **Registro** (solo anota lo que pensás). Al mover un control no se genera nada: solo el botón **Generar** llama al modelo.

1. Apretá **Generar** con los valores iniciales. Cada corrida queda arriba de la anterior.
2. Cambiá solo la **temperatura** (por ejemplo, 0,3 y 1,5) y generá de nuevo.
3. Apretá **Comparar las dos últimas corridas**: la tabla marca qué cambió y qué no.
4. Pasá a **determinista** y generá dos veces. ¿Qué pasa con el texto?

Si querés volver atrás, **Volver a los valores iniciales** restaura los controles y deja las corridas donde están; y «Volver a una configuración anterior» carga la de una corrida que ya hiciste. Las pestañas **Inspeccionar** y **Modificar** muestran la configuración que se usó y la llamada de Python equivalente.

### ✎ Antes de ejecutar

¿Qué esperás que cambie más el texto: mover la temperatura o cambiar la semilla?

In [ ]:
display(armar_estación_1())

### Qué hay detrás del texto que apareció

El mismo comienzo da textos distintos porque el modelo no «elige una palabra»: en cada paso reparte probabilidad entre **todos** los tokens de su vocabulario (50.257) y después una regla sortea uno. Eso se repite hasta completar el texto. Un **token** es una pieza de texto: una palabra, un pedazo de palabra o un signo.

Cómo reparte, en dos pasos: el modelo le da un puntaje a cada token (los *logits*: a mayor puntaje, más candidato) y la *softmax* convierte esos puntajes en probabilidades que suman 1. La **temperatura** divide los puntajes antes de la softmax. Mirá un ejemplo con tres candidatos inventados:

In [ ]:
# Puntajes INVENTADOS para tres candidatos: sirven para ver el mecanismo, no vienen del modelo.
puntajes = torch.tensor([2.0, 1.0, 0.0])

tabla_del_ejemplo = []
for temperatura in (0.5, 1.0, 2.0):
    puntajes_ajustados = puntajes / temperatura                  # la temperatura divide los puntajes
    probabilidades = torch.softmax(puntajes_ajustados, dim=0)    # softmax: los pasa a probabilidades que suman 1
    probabilidades = probabilidades.tolist()
    tabla_del_ejemplo.append({
        "temperatura": temperatura,
        "salió": round(probabilidades[0], 3),
        "dormía": round(probabilidades[1], 3),
        "rugió": round(probabilidades[2], 3),
    })

display(pd.DataFrame(tabla_del_ejemplo))

**Mirá la tabla.** Con temperatura 0,5, «salió» se queda con la mayor parte de la probabilidad (0,867); con 2,0 baja a 0,506 y los otros dos candidatos ganan lugar. Con los puntajes reales del modelo el mecanismo es el mismo, pero entre miles de candidatos.

### Qué controla cada parámetro

| Control | Qué controla | Qué podés probar y mirar | Qué no significa |
|---|---|---|---|
| `temperatura` | Cuánto se reparten las probabilidades antes de sortear | Bajala y mirá si las muestras se parecen más entre sí; subila y mirá si aparecen palabras raras | No es «creatividad»: solo cambia el reparto |
| `top_k` | Cuántos candidatos quedan en juego: los `k` más probables | Poné 1 y generá varias muestras | No elige `k` candidatos al azar |
| `top_p` | Deja en juego al grupo más chico de candidatos cuya probabilidad suma `p` | Bajalo a 0,3 y compará con 1,0 | No es una probabilidad mínima por candidato |
| `semilla` | Fija el sorteo | Repetí una corrida con la misma semilla y mirá si el texto coincide | No hace al texto mejor ni peor: es otra tirada |
| determinista (`do_sample=False`) | Elige siempre el candidato más probable | Generá dos veces | No es «el mejor texto» |

Los filtros se aplican en este orden: primero la temperatura, después `top_k`, después `top_p`. En la estación 6B lo vas a ver paso a paso.

### Ahora cambiá otra cosa

Con el mismo comienzo:

- Cambiá **solo la semilla**. ¿Cambia más o menos que cuando moviste la temperatura?
- Pedí **5 muestras** y marcá qué se repite entre ellas.
- Poné `top_k` en 1 y generá varias muestras: ¿siguen siendo distintas?

Una sola muestra puede ser casualidad: conviene mirar varias antes de sacar una conclusión.

### Actividad opcional: ¿la temperatura cambia la repetición?

Para saber si un control tiene efecto hay que cambiar **uno solo** y mirar **varias** muestras: con una por valor no se distingue el efecto del control del azar del sorteo.

**Hipótesis para probar** (editá la tuya): *«Con temperaturas más altas, el texto repite menos trigramas de palabras»*. Un *trigrama* es un trío de palabras seguidas. La actividad genera muchas muestras por cada valor de temperatura, con las mismas semillas, y mide dos cosas simples de repetición (no de calidad):

- `palabras_distintas`: qué proporción de las palabras de la continuación no se repite.
- `trigramas_repetidos`: qué proporción de los tríos de palabras aparece más de una vez.

**Preparación · actividad.** Ejecutá la celda siguiente: define las dos funciones de la actividad. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def medidas_de_repetición(texto):
    """Devuelve (palabras_distintas, trigramas_repetidos) de un texto. Son medidas de repetición, no de calidad."""
    palabras = re.findall(r"[a-záéíóúüñ]+", texto.lower())
    if len(palabras) < 3:
        return None, None
    proporción_distintas = len(set(palabras)) / len(palabras)
    trigramas = []
    for posición in range(len(palabras) - 2):
        trigramas.append(tuple(palabras[posición:posición + 3]))
    repetidos = len(trigramas) - len(set(trigramas))
    return proporción_distintas, repetidos / len(trigramas)


def experimento_de_una_variable(modelo, tokenizador, comienzo, parámetro, valores, configuración_fija, n_muestras, semilla):
    """Genera `n_muestras` por cada valor de `parámetro` y devuelve una fila por muestra.

    `configuración_fija` es un diccionario con los demás parámetros de `generar`.
    Todos los valores usan las mismas semillas, así la diferencia entre valores no viene de otro sorteo.
    """
    filas = []
    for valor in valores:
        argumentos = dict(configuración_fija)
        argumentos[parámetro] = valor
        muestras_del_valor = generar(modelo, tokenizador, comienzo, n_muestras=n_muestras, semilla=semilla, **argumentos)
        for muestra in muestras_del_valor:
            distintas, repetidos = medidas_de_repetición(muestra["continuación"])
            filas.append({
                parámetro: valor,
                "semilla": muestra["configuración"]["semilla"],
                "palabras_distintas": distintas,
                "trigramas_repetidos": repetidos,
                "continuación": muestra["continuación"],
            })
    return pd.DataFrame(filas)

**Advertencia:** son tres valores de temperatura por 15 muestras cada uno, o sea 45 generaciones de 40 tokens; en CPU puede llevar varios minutos. Por eso la celda **no corre con «Ejecutar todo»**: poné `PERMITIR_ACTIVIDAD_DE_GENERACIÓN = True` en los parámetros del cuaderno.

### ✎ Antes de ejecutar

¿La media de `trigramas_repetidos` va a subir, bajar o quedar igual entre 0,3 y 1,4?

In [ ]:
PARÁMETRO_A_VARIAR = "temperatura"
VALORES_A_PROBAR = [0.3, 0.8, 1.4]
MUESTRAS_POR_VALOR = 15
COMIENZO_DE_LA_ACTIVIDAD = "Una mañana de otoño, el jaguar"
SEMILLA_DE_LA_ACTIVIDAD = 40

configuración_fija = dict(max_new_tokens=40, modo="muestreo", temperatura=0.8, top_k=50, top_p=0.95)

if PERMITIR_ACTIVIDAD_DE_GENERACIÓN:
    resultados_de_la_actividad = experimento_de_una_variable(
        modelo=modelo_base,
        tokenizador=tokenizador,
        comienzo=COMIENZO_DE_LA_ACTIVIDAD,
        parámetro=PARÁMETRO_A_VARIAR,
        valores=VALORES_A_PROBAR,
        configuración_fija=configuración_fija,
        n_muestras=MUESTRAS_POR_VALOR,
        semilla=SEMILLA_DE_LA_ACTIVIDAD,
    )

    resumen = resultados_de_la_actividad.groupby(PARÁMETRO_A_VARIAR)[["palabras_distintas", "trigramas_repetidos"]].mean()
    display(resumen)
else:
    resultados_de_la_actividad = None
    cantidad_de_generaciones = len(VALORES_A_PROBAR) * MUESTRAS_POR_VALOR
    print(f"Actividad omitida: son {cantidad_de_generaciones} generaciones. "
          "Poné PERMITIR_ACTIVIDAD_DE_GENERACIÓN = True en los parámetros del cuaderno para ejecutarla.")

**Mirá la tabla.** Cada fila es el promedio de las muestras de un valor. Un promedio esconde la dispersión; en la celda siguiente se dibuja cada muestra.

In [ ]:
if resultados_de_la_actividad is not None:
    figura = utils.figura_de_puntos_por_valor(
        tabla=resultados_de_la_actividad,
        columna_valor=PARÁMETRO_A_VARIAR,
        columna_medida="trigramas_repetidos",
        título="Trigramas de palabras repetidos, por temperatura",
        etiqueta_y="Proporción de trigramas repetidos",
    )
    display(Image(data=utils.figura_a_png(figura)))
else:
    print('Sin resultados que graficar: la actividad no se ejecutó.')

**Mirá el gráfico.** Cada punto es una muestra y la raya es la media de ese valor.

- **Observación verificable:** la media de cada valor y cuánto se separan los puntos dentro de un mismo valor.
- **Interpretación posible:** si los puntos de dos valores se superponen mucho, la diferencia entre medias puede ser azar del sorteo.
- **Límite:** un solo comienzo y 40 tokens por muestra no permiten generalizar, y estas medidas no dicen si el texto es «mejor».

### ✎ Para pensar

¿Tu apuesta se cumplió? ¿Con qué datos lo afirmás?

**Anotá (opcional).** Si querés dejar registrado lo que viste, usá el panel de abajo. Elegí el tipo de anotación: una **observación** (lo que ves), una **interpretación** (lo que creés que lo explica), una **limitación** (lo que debilita tu conclusión) o una **hipótesis pendiente** (lo que habría que probar).

In [ ]:
display(armar_bitácora_visual(estación_inicial="1-explorar", lugar="1"))

<details>
<summary>Profundización: qué hace <code>generar</code></summary>

`generar` hace cuatro cosas, en este orden: valida los parámetros, convierte el comienzo en identificadores de tokens, llama a `modelo.generate` una vez por muestra (cada una con su semilla: `semilla + número de muestra`) y devuelve el texto junto con la configuración efectiva.

- En modo `determinista` ignora temperatura, `top_k`, `top_p` y semilla, y los registra como `None`: lo que se registra es lo que se usó.
- Guarda por separado la entrada y la continuación, para no confundir lo que escribiste con lo que agregó el modelo.
- `exigir_dispositivo_libre` hace que `generar` (y toda función que use un modelo) se niegue mientras un ajuste corre en otro hilo: usar la GPU desde dos hilos cerró el kernel en una prueba en el navegador.
- `describir_diferencias` compara dos configuraciones parámetro por parámetro y avisa si cambió más de una.
</details>

### Vía con código (alternativa al panel)

Generar dos continuaciones del mismo comienzo con muestreo. Los parámetros están en MAYÚSCULAS para que los edites.

Antes de ejecutar, apostá: si ejecutás la celda dos veces sin tocar nada, ¿esperás el mismo texto? ¿Y si cambiás solo la semilla?

In [ ]:
COMIENZO = "Una mañana de otoño, el jaguar"
MODO = "muestreo"
TEMPERATURA = 1.0
TOP_K = 50
TOP_P = 0.95
MAX_NEW_TOKENS = 200
N_MUESTRAS = 2
SEMILLA = 40

muestras = generar(
    modelo=modelo_base,
    tokenizador=tokenizador,
    comienzo=COMIENZO,
    n_muestras=N_MUESTRAS,
    max_new_tokens=MAX_NEW_TOKENS,
    modo=MODO,
    temperatura=TEMPERATURA,
    top_k=TOP_K,
    top_p=TOP_P,
    semilla=SEMILLA,
)

for muestra in muestras:
    semilla_usada = muestra["configuración"]["semilla"]
    display(HTML(utils.html_texto_con_continuación(muestra["entrada"], muestra["continuación"], actor="base")))
    print(f"semilla de esta muestra: {semilla_usada} · tokens nuevos: {muestra['tokens_nuevos']}")

**Mirá el resultado.** Cada tarjeta separa la **entrada** (gris) de la **continuación** (con borde de color y el rótulo «Modelo base»). Con la misma semilla y en esta máquina deberías obtener los mismos textos al volver a ejecutar la celda; con otra semilla, no. Las muestras difieren entre sí porque cada una usa una semilla distinta, y dos muestras no alcanzan para saber cómo se comporta el modelo en general.

---
## Estación 2. Elegí y revisá los textos

**Desafío.** El modelo solo puede aprender de lo que pongas en el **corpus**. Mirá qué hay, cuánto pesa cada texto y qué problemas traen; después decidí qué entra, qué sale y qué se limpia.

Acá «limpiar» quiere decir transformar los textos del corpus, que son los datos del experimento. No tiene que ver con revisar o corregir las explicaciones de este cuaderno.

**Preparación · estación 2 (lectura de textos).** Ejecutá la celda siguiente: define cómo se leen los archivos `.txt`, `.pdf` y `.epub`, cómo se revisa un texto y cómo se arma un documento. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def leer_txt(ruta):
    """Lee un archivo de texto y devuelve (texto, advertencias).

    Prueba UTF-8; si falla, usa latin-1 y lo avisa. No quita nada: si el archivo trae una marca
    BOM al principio, se conserva en `texto` y se ve como una advertencia.
    """
    advertencias = []
    contenido = Path(ruta).read_bytes()
    try:
        texto = contenido.decode("utf-8")
    except UnicodeDecodeError:
        texto = contenido.decode("latin-1")
        advertencias.append("No era UTF-8: se leyó como latin-1. Revisá tildes y eñes.")
    return texto, advertencias


def leer_pdf(ruta):
    """Extrae el texto de un PDF con la librería `pypdf` y devuelve (texto, advertencias).

    Un PDF escaneado (una foto de la página) no tiene texto que extraer: da páginas vacías.
    """
    try:
        from pypdf import PdfReader
    except ImportError as error:
        raise ImportError("Para leer PDF falta la librería pypdf. Instalala con: uv pip install pypdf") from error
    lector = PdfReader(str(ruta))
    páginas = []
    páginas_sin_texto = 0
    for página in lector.pages:
        texto_de_la_página = página.extract_text() or ""
        if not texto_de_la_página.strip():
            páginas_sin_texto += 1
        páginas.append(texto_de_la_página)
    advertencias = []
    if páginas_sin_texto:
        advertencias.append(f"{páginas_sin_texto} de {len(páginas)} páginas no tienen texto extraíble (¿PDF escaneado? haría falta OCR).")
    return "\n\n".join(páginas), advertencias


def leer_epub(ruta):
    """Extrae el texto de un EPUB con `ebooklib` y `beautifulsoup4` y devuelve (texto, advertencias)."""
    try:
        import ebooklib
        from bs4 import BeautifulSoup
        from ebooklib import epub
    except ImportError as error:
        raise ImportError("Para leer EPUB faltan librerías. Instalalas con: uv pip install ebooklib beautifulsoup4") from error
    libro = epub.read_epub(str(ruta))
    capítulos = []
    for ítem in libro.get_items():
        if ítem.get_type() == ebooklib.ITEM_DOCUMENT:
            sopa = BeautifulSoup(ítem.get_body_content(), "html.parser")
            capítulos.append(sopa.get_text("\n"))
    advertencias = []
    if not capítulos:
        advertencias.append("El EPUB no tiene capítulos de texto.")
    return "\n\n".join(capítulos), advertencias


LECTORES = {".txt": leer_txt, ".pdf": leer_pdf, ".epub": leer_epub}


def revisar_texto(texto):
    """Busca señales de problemas en un texto y devuelve una lista de advertencias en español."""
    advertencias = []
    if not texto.strip():
        advertencias.append("Documento vacío.")
        return advertencias
    if len(texto) < 500:
        advertencias.append(f"Documento muy corto ({len(texto)} caracteres).")
    if "﻿" in texto:
        advertencias.append("Tiene una marca BOM (carácter invisible al principio del archivo).")
    cantidad_de_reemplazos = texto.count("�")
    if cantidad_de_reemplazos:
        advertencias.append(f"Tiene {cantidad_de_reemplazos} caracteres de reemplazo (�): la codificación se rompió al leer.")
    líneas_con_texto = []
    for línea in texto.split("\n"):
        if línea.strip():
            líneas_con_texto.append(línea)
    cortas = 0
    for línea in líneas_con_texto:
        if len(línea.strip()) < 25:
            cortas += 1
    if len(líneas_con_texto) >= 30 and cortas / len(líneas_con_texto) > 0.4:
        advertencias.append("Más del 40 % de las líneas son muy cortas: posible corte de líneas de una extracción de PDF.")
    cortadas_por_guion = len(re.findall(r"\w-\n\w", texto))
    if cortadas_por_guion > 5:
        advertencias.append(f"{cortadas_por_guion} palabras parecen cortadas con guion a fin de línea.")
    letras = len(re.findall(r"[A-Za-záéíóúüñÁÉÍÓÚÜÑ]", texto))
    if letras / len(texto) < 0.6:
        advertencias.append("Menos del 60 % de los caracteres son letras: ¿tablas, números de página, código?")
    return advertencias


def documento_desde_texto(identificador, texto, advertencias_de_lectura=None, **metadatos):
    """Arma un documento (diccionario) a partir de un texto y sus metadatos."""
    documento = {
        "id": identificador,
        "título": metadatos.get("título", identificador),
        "autoría": metadatos.get("autoría", "sin dato"),
        "fecha": metadatos.get("fecha", "sin dato"),
        "género": metadatos.get("género", "sin dato"),
        "fuente": metadatos.get("fuente", "sin dato"),
        "condiciones_de_uso": metadatos.get("condiciones_de_uso", "sin dato: confirmá antes de compartir"),
        "grupo": metadatos.get("grupo", ""),
        "archivo": metadatos.get("archivo", ""),
        "texto_original": texto,
        "texto": texto,
        "incluido": True,
        "transformaciones": [],
    }
    documento["advertencias"] = list(advertencias_de_lectura or []) + revisar_texto(texto)
    return documento


def documento_desde_archivo(ruta, identificador=None, **metadatos):
    """Lee un .txt, .pdf o .epub y lo convierte en documento. Falla con un mensaje claro si no puede."""
    ruta = Path(ruta)
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo: {ruta}")
    extensión = ruta.suffix.lower()
    if extensión not in LECTORES:
        raise ValueError(f"No sé leer archivos {extensión!r}. Formatos: {', '.join(LECTORES)}.")
    texto, advertencias = LECTORES[extensión](ruta)
    return documento_desde_texto(identificador or ruta.stem, texto, advertencias, archivo=ruta.name, **metadatos)


def cargar_corpus_de_quiroga(ruta="datos/quiroga/corpus.jsonl"):
    """Carga los 26 cuentos de Horacio Quiroga (Wikisource) como documentos."""
    documentos = []
    with open(ruta, encoding="utf-8") as archivo:
        for línea in archivo:
            registro = json.loads(línea)
            documentos.append(documento_desde_texto(
                registro["id"], registro["texto"],
                título=registro["titulo"], autoría=registro["autor"],
                fecha="1918 (según Wikisource; sin verificar la primera edición)",
                género="cuento", grupo=registro["grupo"],
                fuente=f"Wikisource, revisión {registro['revid']}: {registro['fuente_url']}",
                condiciones_de_uso="Dominio público (el autor murió en 1937). La transcripción es una edición colaborativa.",
            ))
    return documentos


def cargar_corpus_de_textos_propios(carpeta="datos/textos"):
    """Carga los .txt de una carpeta; si hay un manifiesto.csv, lo usa para los metadatos."""
    carpeta = Path(carpeta)
    manifiesto = {}
    ruta_del_manifiesto = carpeta / "manifiesto.csv"
    if ruta_del_manifiesto.exists():
        tabla = pd.read_csv(ruta_del_manifiesto, encoding="utf-8")
        for _, fila in tabla.iterrows():
            manifiesto[fila["archivo"]] = fila
    documentos = []
    for ruta in sorted(carpeta.glob("*.txt")):
        ficha = manifiesto.get(ruta.name)
        metadatos = {}
        if ficha is not None:
            metadatos = dict(
                título=ruta.stem, autoría=ficha["autoría"], fecha=str(ficha["fecha_de_publicación_aprox"]),
                género=ficha["género"], fuente=ficha["nota_de_procedencia"],
                condiciones_de_uso=ficha["condiciones_de_uso"], grupo="textos propios",
            )
        identificador = ficha["id"] if ficha is not None else ruta.stem
        documentos.append(documento_desde_archivo(ruta, identificador=identificador, **metadatos))
    return documentos


TEXTO_DE_PRÁCTICA = (
    "\ufeffLa panadería de la calle Sarmiento abría a las cinco de la mañana. Desde la vereda se olía el pan caliente\r\n"
    "y se oía el ruido de las bandejas contra el mostrador de mármol. Don Aurelio, que llevaba cuarenta años\r\n"
    "amasando, decía que el secreto no estaba en la harina sino en la paciencia de esperar que la masa creciera sola.\r\n"
    "\n\n\n"
    "Esa mañana llegó un cliente que nadie conocía. Pidió medio kilo de factu-\n"
    "ras, pagó con un billete viejo y se quedó mirando el reloj de la pared. Aurelio le preguntó si esperaba a alguien.  \n"
    "El hombre sonrió y contestó que esperaba que el reloj, por fin, diera la hora correcta.\n"
)


def documento_de_práctica():
    """Un texto INVENTADO con los problemas típicos, para probar limpiezas. Nace fuera del corpus (`incluido=False`)."""
    documento = documento_desde_texto(
        "práctica", TEXTO_DE_PRÁCTICA,
        título="Texto de práctica (inventado)", autoría="escrito para este cuaderno", fecha="2026",
        género="práctica", grupo="práctica", fuente="Escrito para practicar limpiezas; no es un texto real.",
        condiciones_de_uso="Libre.",
    )
    documento["incluido"] = False
    return documento

### Cargar el corpus de trabajo

Elegí con qué textos trabajás con `CORPUS_ELEGIDO`:

- `"quiroga"`: 26 cuentos de Horacio Quiroga (dominio público), divididos en adulto e infantil.
- `"textos"`: 4 textos con derechos vigentes, solo para uso local.
- `"ambos"`: los dos conjuntos.

La celda también suma un **texto de práctica inventado**, con los problemas típicos de formato. Queda fuera del corpus (`incluido=False`): el modelo no lo ve. Sirve para probar limpiezas con cambios visibles.

**Advertencia:** los textos de `datos/textos` tienen derechos vigentes; son para trabajo local y de aula, y no deben publicarse ni subirse a servicios externos.

Para sumar tus propios archivos, usá `documento_desde_archivo("mis_textos/cuento.txt", autoría="...")` y agregalos a `documentos`.

In [ ]:
CORPUS_ELEGIDO = "quiroga"

documentos = []
if CORPUS_ELEGIDO in ("quiroga", "ambos"):
    documentos = documentos + cargar_corpus_de_quiroga()
if CORPUS_ELEGIDO in ("textos", "ambos"):
    documentos = documentos + cargar_corpus_de_textos_propios()

print(f"Documentos del corpus: {len(documentos)}")
con_advertencias = 0
for documento in documentos:
    if documento["advertencias"]:
        con_advertencias += 1
print(f"Documentos con advertencias: {con_advertencias}")

# Texto inventado para practicar limpiezas. Queda fuera del corpus (incluido=False): el modelo no lo ve.
documentos = documentos + [documento_de_práctica()]
print("Texto de práctica agregado, fuera del corpus.")

**Mirá el resultado.** Cuántos documentos tiene el corpus y cuántos traen advertencias. En el corpus de Quiroga no esperamos advertencias graves: es texto ya limpio de Wikisource.

**Preparación · estación 2 (mesa de trabajo).** Ejecutá la celda siguiente: define la tabla de documentos, las transformaciones, las versiones y el panel. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def documento_con_id(documentos, identificador):
    """Devuelve el documento con ese `id`, o levanta un error que lista los ids que existen."""
    for documento in documentos:
        if documento["id"] == identificador:
            return documento
    ids = []
    for otro in documentos:
        ids.append(otro["id"])
    raise KeyError(f"No hay un documento con id {identificador!r}. Ids disponibles: {ids}")


def tabla_de_documentos(documentos, tokenizador):
    """Una fila por documento: tamaños, participación entre los incluidos y cantidad de advertencias."""
    filas = []
    total_de_tokens = 0
    for documento in documentos:
        cantidad_de_tokens = len(tokenizador(documento["texto"])["input_ids"])
        documento["_tokens"] = cantidad_de_tokens
        if documento["incluido"]:
            total_de_tokens += cantidad_de_tokens
    for documento in documentos:
        participación = documento["_tokens"] / total_de_tokens if documento["incluido"] and total_de_tokens else 0.0
        filas.append({
            "id": documento["id"],
            "título": documento["título"],
            "grupo": documento["grupo"],
            "incluido": documento["incluido"],
            "caracteres": len(documento["texto"]),
            "palabras": len(documento["texto"].split()),
            "tokens": documento["_tokens"],
            "participación": utils.en_porcentaje(participación),
            "advertencias": len(documento["advertencias"]),
        })
    return pd.DataFrame(filas)


def detectar_duplicados_exactos(documentos):
    """Agrupa los documentos incluidos cuyo texto es idéntico, carácter por carácter. Devuelve una lista de grupos de ids."""
    ids_por_huella = {}
    for documento in documentos:
        if not documento["incluido"]:
            continue
        huella = laboratorio.huella_de_texto(documento["texto"])
        ids_por_huella.setdefault(huella, []).append(documento["id"])
    grupos = []
    for ids in ids_por_huella.values():
        if len(ids) > 1:
            grupos.append(ids)
    return grupos


def quitar_marca_bom(texto):
    nuevo = texto.replace("﻿", "")
    return nuevo, texto.count("﻿")


def unificar_saltos_de_línea(texto):
    cambios = texto.count("\r\n") + texto.replace("\r\n", "").count("\r")
    nuevo = texto.replace("\r\n", "\n").replace("\r", "\n")
    return nuevo, cambios


def recortar_espacios_al_final_de_línea(texto):
    nuevo, cambios = re.subn(r"[ \t]+(?=\n)", "", texto)
    return nuevo, cambios


def reducir_líneas_en_blanco(texto):
    nuevo, cambios = re.subn(r"\n{3,}", "\n\n", texto)
    return nuevo, cambios


def unir_palabras_cortadas_por_guion(texto):
    nuevo, cambios = re.subn(r"(\w)-\n(\w)", r"\1\2", texto)
    return nuevo, cambios


def unir_líneas_de_un_párrafo(texto):
    nuevo, cambios = re.subn(r"(?<=\S)[ \t]*\n(?=\S)", " ", texto)
    return nuevo, cambios


TRANSFORMACIONES = {
    "quitar_marca_BOM": quitar_marca_bom,
    "unificar_saltos_de_línea": unificar_saltos_de_línea,
    "recortar_espacios_al_final_de_línea": recortar_espacios_al_final_de_línea,
    "reducir_líneas_en_blanco": reducir_líneas_en_blanco,
    "unir_palabras_cortadas_por_guion": unir_palabras_cortadas_por_guion,
    "unir_líneas_de_un_párrafo": unir_líneas_de_un_párrafo,
}


def aplicar_pasos(texto, pasos):
    """Aplica `pasos` (nombres de TRANSFORMACIONES) a un texto y devuelve (texto nuevo, registro). No toca ningún documento."""
    registro = []
    for paso in pasos:
        if paso not in TRANSFORMACIONES:
            raise ValueError(f"Transformación desconocida: {paso!r}. Opciones: {list(TRANSFORMACIONES)}")
        caracteres_antes = len(texto)
        texto, cambios = TRANSFORMACIONES[paso](texto)
        registro.append({"paso": paso, "cambios": cambios, "caracteres_antes": caracteres_antes, "caracteres_después": len(texto)})
    return texto, registro


def probar_transformaciones(documento, pasos):
    """Ensayo de una limpieza: NO cambia el documento y devuelve qué pasaría.

    Parte del texto original y devuelve el texto resultante, el registro de cada paso y los primeros
    pares (antes, después) que cambiarían respecto del texto que el corpus tiene ahora.
    """
    texto, registro = aplicar_pasos(documento["texto_original"], pasos)
    pares = líneas_que_cambiaron(documento["texto"], texto, máximo=4)
    return {"texto": texto, "registro": registro, "pares": pares, "cambia": texto != documento["texto"]}


def aplicar_transformaciones(documento, pasos):
    """Aplica `pasos` (nombres de TRANSFORMACIONES) al texto ORIGINAL de un documento.

    Parte siempre de `texto_original`, así aplicar dos veces no acumula cambios. Guarda en el
    documento qué se aplicó y cuántos cambios hizo cada paso. Devuelve ese registro.
    """
    texto, registro = aplicar_pasos(documento["texto_original"], pasos)
    documento["texto"] = texto
    documento["transformaciones"] = registro
    documento["advertencias"] = revisar_texto(texto)
    return registro


def líneas_que_cambiaron(antes, después, máximo=6):
    """Pares (línea antes, línea después) de los primeros cambios, para ver el efecto sin leer todo."""
    líneas_antes = antes.split("\n")
    líneas_después = después.split("\n")
    comparador = difflib.SequenceMatcher(a=líneas_antes, b=líneas_después, autojunk=False)
    pares = []
    for etiqueta, a1, a2, b1, b2 in comparador.get_opcodes():
        if etiqueta == "equal":
            continue
        pares.append(("\n".join(líneas_antes[a1:a2]), "\n".join(líneas_después[b1:b2])))
        if len(pares) >= máximo:
            break
    return pares


def participación_por_grupo(tabla):
    """Una fila por grupo (por ejemplo, adulto e infantil) con sus documentos incluidos, sus tokens y su parte del corpus."""
    tokens_por_grupo = {}
    documentos_por_grupo = {}
    total_de_tokens = 0
    for fila in tabla.to_dict("records"):
        if not fila["incluido"]:
            continue
        grupo = fila["grupo"] or "sin grupo"
        tokens_por_grupo[grupo] = tokens_por_grupo.get(grupo, 0) + int(fila["tokens"])
        documentos_por_grupo[grupo] = documentos_por_grupo.get(grupo, 0) + 1
        total_de_tokens += int(fila["tokens"])
    filas = []
    for grupo in tokens_por_grupo:
        parte = tokens_por_grupo[grupo] / total_de_tokens if total_de_tokens else 0.0
        filas.append([grupo, str(documentos_por_grupo[grupo]), utils.con_miles(tokens_por_grupo[grupo]), utils.en_porcentaje(parte)])
    return filas


def ejemplos_de_problemas(texto, máximo=3):
    """Dónde están los problemas que se pueden limpiar: lista de (problema, fragmento del texto).

    `revisar_texto` avisa que algo anda mal; esta función muestra un ejemplo concreto de cada problema.
    """
    ejemplos = []
    if "\ufeff" in texto:
        ejemplos.append(("Marca BOM (carácter invisible)", "al principio: " + repr(texto[:12])))
    posición = texto.find("\ufffd")
    if posición >= 0:
        ejemplos.append(("Carácter de reemplazo: la codificación se rompió", repr(texto[max(0, posición - 25):posición + 25])))
    if "\r" in texto:
        posición = texto.index("\r")
        ejemplos.append(("Saltos de línea de Windows (\\r\\n)", repr(texto[max(0, posición - 25):posición + 2])))
    coincidencia = re.search(r"[ \t]+\n", texto)
    if coincidencia:
        ejemplos.append(("Espacios al final de una línea", repr(texto[max(0, coincidencia.start() - 25):coincidencia.end()])))
    coincidencia = re.search(r"\n{3,}", texto)
    if coincidencia:
        ejemplos.append(("Tres o más saltos de línea seguidos", repr(texto[max(0, coincidencia.start() - 25):coincidencia.end() + 10])))
    for cortada in re.findall(r"\w+-\n\w+", texto)[:máximo]:
        ejemplos.append(("Palabra cortada con guion a fin de línea", repr(cortada)))
    return ejemplos


def tomar_versión(documentos, nombre):
    """Foto del corpus en este momento: documentos incluidos, huella de cada texto y huella general."""
    fichas = []
    for documento in documentos:
        pasos_aplicados = []
        for transformación in documento["transformaciones"]:
            pasos_aplicados.append(transformación["paso"])
        fichas.append({
            "id": documento["id"],
            "incluido": documento["incluido"],
            "huella_del_texto": laboratorio.huella_de_texto(documento["texto"]),
            "huella_del_original": laboratorio.huella_de_texto(documento["texto_original"]),
            "caracteres": len(documento["texto"]),
            "transformaciones": pasos_aplicados,
        })
    incluidos = []
    for ficha in fichas:
        if ficha["incluido"]:
            incluidos.append(f"{ficha['id']}:{ficha['huella_del_texto']}")
    huella_general = laboratorio.huella_de_texto("\n".join(sorted(incluidos)))
    return {"nombre": nombre, "hora": laboratorio.ahora(), "huella": huella_general, "documentos": fichas}


def comparar_versiones(versión_a, versión_b):
    """Tabla con una fila por documento: si se agregó, se quitó, se modificó o quedó igual."""
    por_id_a = {}
    for ficha in versión_a["documentos"]:
        if ficha["incluido"]:
            por_id_a[ficha["id"]] = ficha
    por_id_b = {}
    for ficha in versión_b["documentos"]:
        if ficha["incluido"]:
            por_id_b[ficha["id"]] = ficha
    filas = []
    for identificador in sorted(set(por_id_a) | set(por_id_b)):
        ficha_a = por_id_a.get(identificador)
        ficha_b = por_id_b.get(identificador)
        if ficha_a is None:
            estado = "agregado"
        elif ficha_b is None:
            estado = "quitado"
        elif ficha_a["huella_del_texto"] != ficha_b["huella_del_texto"]:
            estado = "modificado"
        else:
            estado = "igual"
        filas.append({
            "id": identificador,
            "estado": estado,
            "caracteres_A": ficha_a["caracteres"] if ficha_a else None,
            "caracteres_B": ficha_b["caracteres"] if ficha_b else None,
        })
    return pd.DataFrame(filas)


versiones_del_corpus = []  # las versiones que se guardan con el panel


def armar_estación_2():
    """Mesa de trabajo del corpus: ver cuánto pesa cada texto, cambiar la composición y probar limpiezas antes de aceptarlas.

    Trabaja sobre la lista global `documentos` y guarda sus versiones en `versiones_del_corpus`.
    """
    estación = laboratorio.nueva_estación("2-corpus")

    opciones_de_documentos = []
    for documento in documentos:
        opciones_de_documentos.append((documento["id"] + " · " + documento["título"], documento["id"]))
    selector = estación.agregar(widgets.Dropdown(options=opciones_de_documentos))
    laboratorio.control(selector, "Documento", "ID_DEL_DOCUMENTO", ancho="40em", tipo="vista",
                        ayuda="Con cuál de los textos trabajás en las pestañas Inspeccionar y Modificar.",
                        prueba="elegí «Texto de práctica (inventado)» para ver una limpieza con cambios.")
    incluir = estación.agregar(widgets.Checkbox(value=True, description="Incluido en el corpus", indent=False))
    laboratorio.control(incluir, "Incluido en el corpus", "incluido", tipo="datos",
                        ayuda="Si lo destildás, el documento sale del corpus (no se borra).",
                        prueba="destildá el más largo, apretá «Actualizar tabla» y mirá cómo se reparte el peso.")
    pasos = estación.agregar(widgets.SelectMultiple(options=list(TRANSFORMACIONES), rows=6))
    laboratorio.control(pasos, "Transformaciones (Ctrl/Cmd para varias)", "PASOS_A_APLICAR", tipo="datos",
                        ayuda="Qué le hacés al texto de este documento. Siempre se parte del texto original.",
                        prueba="elegí una sola, mirá el antes y el después, y recién ahí decidí.")

    botón_tabla = estación.agregar(laboratorio.botón("Actualizar tabla y gráfico", "Cuenta los tokens de cada documento"))
    botón_versión = estación.agregar(laboratorio.botón("Guardar versión del corpus", "Calcula la huella del corpus actual"))
    botón_comparar_versiones = estación.agregar(laboratorio.botón("Comparar las dos últimas versiones", "Qué documentos se agregaron, quitaron o cambiaron"))
    botón_probar = estación.agregar(laboratorio.botón("Probar la limpieza", "Muestra el antes y el después; no cambia el corpus", "primary"))
    botón_aceptar = estación.agregar(laboratorio.botón("Aceptar el cambio", "Aplica la limpieza probada a este documento; el original se conserva", "success"))
    botón_descartar = estación.agregar(laboratorio.botón("Descartar la prueba", "Todo sigue como estaba"))
    botón_volver = estación.agregar(laboratorio.botón("Volver al texto original", "Deshace las limpiezas aceptadas de este documento"))
    botón_aceptar.disabled = True
    botón_descartar.disabled = True
    estado = estación.agregar(widgets.HTML(utils.html_estado("pendiente")))
    tabla_html = estación.agregar(widgets.HTML())
    gráfico = estación.agregar(widgets.Image(format="png", layout=widgets.Layout(max_width="100%")))
    grupos_html = estación.agregar(widgets.HTML())
    versiones_vista = estación.agregar(widgets.HTML())
    inspección = estación.agregar(widgets.HTML())
    vista_original = estación.agregar(widgets.HTML())
    vista_prueba = estación.agregar(widgets.HTML())
    código = estación.agregar(widgets.HTML())
    ensayo = {"documento": None, "pasos": []}  # la prueba pendiente de aceptar o descartar

    def documento_actual():
        return documento_con_id(documentos, selector.value)

    def borrar_la_prueba():
        ensayo["documento"] = None
        ensayo["pasos"] = []
        botón_aceptar.disabled = True
        botón_descartar.disabled = True
        vista_prueba.value = ""

    def dibujar_documento(_=None):
        documento = documento_actual()
        incluir.value = documento["incluido"]
        metadatos = []
        for campo in ("título", "autoría", "fecha", "género", "fuente", "condiciones_de_uso", "grupo"):
            metadatos.append([campo, str(documento[campo])])
        problemas = []
        for problema, fragmento in ejemplos_de_problemas(documento["texto"]):
            problemas.append([problema, fragmento])
        if problemas:
            tabla_de_problemas = utils.html_tabla(["problema que se puede limpiar", "ejemplo en el texto"], problemas)
        else:
            tabla_de_problemas = utils.html_aviso("No se encontraron problemas de formato en el texto que el corpus tiene ahora.")
        transformaciones = []
        for transformación in documento["transformaciones"]:
            transformaciones.append([transformación["paso"], str(transformación["cambios"])])
        if not transformaciones:
            transformaciones.append(["ninguna: el corpus tiene el texto original", "0"])
        inspección.value = (utils.html_tabla(["metadato", "valor"], metadatos)
                            + tabla_de_problemas
                            + utils.html_avisos(documento["advertencias"])
                            + utils.html_tabla(["limpieza aceptada", "cambios"], transformaciones))
        vista = "<b>Texto original</b> (primeros 700 caracteres)" + utils.texto_con_espacios_visibles(documento["texto_original"][:700])
        if documento["texto"] != documento["texto_original"]:
            vista += "<b>Texto que hoy tiene el corpus</b> (primeros 700 caracteres)" + utils.texto_con_espacios_visibles(documento["texto"][:700])
        vista_original.value = vista

    def dibujar_código(_=None):
        documento = documento_actual()
        argumentos = dict(documento=laboratorio.Código(f'documento_con_id(documentos, "{documento["id"]}")'), pasos=list(pasos.value))
        prueba = laboratorio.texto_de_llamada("probar_transformaciones", argumentos, "ensayo")
        aceptar = laboratorio.texto_de_llamada("aplicar_transformaciones", argumentos, "registro")
        texto = prueba + "\n\n# Para aceptar el cambio:\n" + aceptar
        código.value = f"<pre style='font-family:{utils.FUENTE_CODIGO}'>{utils.escapar(texto)}</pre>"

    def al_cambiar_documento(_=None):
        borrar_la_prueba()
        dibujar_documento()
        dibujar_código()

    def al_cambiar_pasos(_=None):
        borrar_la_prueba()
        dibujar_código()

    def al_cambiar_inclusión(cambio):
        documento_actual()["incluido"] = cambio["new"]

    def al_actualizar_tabla(_):
        with laboratorio.Ocupado([botón_tabla, botón_versión, botón_comparar_versiones, botón_probar], estado, "Contando tokens", utils.html_estado):
            tabla = tabla_de_documentos(documentos, tokenizador)
            ordenada = tabla.sort_values("tokens", ascending=False)
            visibles = ordenada[["id", "título", "grupo", "incluido", "palabras", "tokens", "participación", "advertencias"]]
            tabla_html.value = utils.html_tabla(list(visibles.columns), visibles.astype(str).values.tolist())
            incluidos = ordenada[ordenada["incluido"]]
            if len(incluidos) == 0:
                raise ValueError("No hay documentos incluidos: tildá al menos uno.")
            figura = utils.figura_de_participación(incluidos["id"].tolist(), incluidos["tokens"].tolist())
            gráfico.value = utils.figura_a_png(figura)
            grupos_html.value = utils.html_tabla(["grupo", "documentos incluidos", "tokens", "parte del corpus"], participación_por_grupo(tabla))

    def al_probar(_):
        if not pasos.value:
            vista_prueba.value = utils.html_aviso("Elegí al menos una transformación para probarla.")
            return
        documento = documento_actual()
        resultado = probar_transformaciones(documento, list(pasos.value))
        ensayo["documento"] = documento["id"]
        ensayo["pasos"] = list(pasos.value)
        filas = []
        for registro in resultado["registro"]:
            filas.append([registro["paso"], str(registro["cambios"]), f"{registro['caracteres_antes']} → {registro['caracteres_después']}"])
        vista_prueba.value = (utils.html_aviso("Es solo una prueba: el corpus todavía no cambió. Aceptá o descartá.")
                              + utils.html_tabla(["paso", "cambios", "caracteres (antes → después)"], filas)
                              + utils.html_comparación_de_textos(resultado["pares"]))
        botón_aceptar.disabled = not resultado["cambia"]
        botón_descartar.disabled = False

    def al_aceptar(_):
        if ensayo["documento"] != selector.value:
            borrar_la_prueba()
            vista_prueba.value = utils.html_aviso("La prueba era de otro documento. Probá de nuevo.", "error")
            return
        documento = documento_actual()
        aplicar_transformaciones(documento, ensayo["pasos"])
        borrar_la_prueba()
        dibujar_documento()
        vista_prueba.value = utils.html_aviso("Cambio aceptado. El texto original se conserva y podés volver a él. "
                                              "Actualizá la tabla para ver el nuevo peso.", "ok")

    def al_descartar(_):
        borrar_la_prueba()
        vista_prueba.value = utils.html_aviso("Prueba descartada: el corpus sigue igual.")

    def al_volver_al_original(_):
        documento = documento_actual()
        documento["texto"] = documento["texto_original"]
        documento["transformaciones"] = []
        documento["advertencias"] = revisar_texto(documento["texto"])
        borrar_la_prueba()
        dibujar_documento()
        vista_prueba.value = utils.html_aviso("El documento volvió a su texto original.", "ok")

    def al_guardar_versión(_):
        nombre = f"versión {len(versiones_del_corpus) + 1}"
        versiones_del_corpus.append(tomar_versión(documentos, nombre))
        estado.value = utils.html_estado("completado", f"{nombre}: huella {versiones_del_corpus[-1]['huella'][:12]}")

    def al_comparar_versiones(_):
        if len(versiones_del_corpus) < 2:
            versiones_vista.value = utils.html_aviso("Guardá al menos dos versiones para compararlas.")
            return
        anterior = versiones_del_corpus[-2]
        última = versiones_del_corpus[-1]
        comparación = comparar_versiones(anterior, última)
        filas = []
        con_diferencias = 0
        for registro in comparación.to_dict("records"):
            if registro["estado"] != "igual":
                con_diferencias += 1
            fila = [registro["id"], registro["estado"]]
            for clave in ("caracteres_A", "caracteres_B"):
                valor = registro[clave]
                fila.append("—" if pd.isna(valor) else str(int(valor)))
            filas.append(fila)
        titular = f"{anterior['nombre']} → {última['nombre']}: {con_diferencias} documento(s) con diferencias"
        versiones_vista.value = f"<b>{utils.escapar(titular)}</b>" + utils.html_tabla(["id", "estado", "caracteres A", "caracteres B"], filas)

    estación.observar(selector, al_cambiar_documento)
    estación.observar(incluir, al_cambiar_inclusión)
    estación.observar(pasos, al_cambiar_pasos)
    estación.al_hacer_clic(botón_tabla, al_actualizar_tabla)
    estación.al_hacer_clic(botón_probar, al_probar)
    estación.al_hacer_clic(botón_aceptar, al_aceptar)
    estación.al_hacer_clic(botón_descartar, al_descartar)
    estación.al_hacer_clic(botón_volver, al_volver_al_original)
    estación.al_hacer_clic(botón_versión, al_guardar_versión)
    estación.al_hacer_clic(botón_comparar_versiones, al_comparar_versiones)
    dibujar_documento()
    dibujar_código()

    explorar = widgets.VBox([widgets.HBox([botón_tabla, botón_versión, botón_comparar_versiones]), estado,
                             tabla_html, gráfico, grupos_html, versiones_vista])
    inspeccionar = widgets.VBox([inspección, vista_original])
    modificar = widgets.VBox([laboratorio.fila_de_control(pasos),
                              widgets.HBox([botón_probar, botón_aceptar, botón_descartar, botón_volver]),
                              vista_prueba, código])
    arriba = widgets.VBox([laboratorio.fila_de_control(selector), laboratorio.fila_de_control(incluir)])
    return widgets.VBox([arriba, laboratorio.panel_de_tres_niveles(explorar, inspeccionar, modificar)])

### Mesa de trabajo del corpus

Arriba del panel elegís un **documento** y si está **incluido** en el corpus; las pestañas muestran cosas distintas sobre él.

1. **Explorar → Actualizar tabla y gráfico.** Antes de apretar, apostá: ¿qué cuento pesa más, uno para adultos o uno de la selva? Después mirá también el reparto entre grupos.
2. **Cambiá la composición.** Destildá **Incluido** en el documento más largo, actualizá y mirá cómo se reparte el peso entre los que quedan.
3. **Inspeccionar.** Elegí el «Texto de práctica» y mirá su texto original y los problemas concretos que tiene, con ejemplos.
4. **Modificar → Probar la limpieza.** Elegí una transformación y apretá **Probar la limpieza**: ves el antes y el después, y el corpus todavía no cambia. Después **Aceptar el cambio** o **Descartar la prueba**. **Volver al texto original** deshace lo aceptado.
5. **Guardar versión del corpus** antes y después de tus cambios.

In [ ]:
display(armar_estación_2())

### Qué miraste en el corpus

- **El peso.** La tabla cuenta *tokens* (las piezas en que el modelo corta el texto), no caracteres, porque es lo que el modelo realmente lee. La parte del corpus que representa cada documento es su participación. Con el corpus de Quiroga, `adulto_09` tiene 10.674 de 94.899 tokens (11,2 %): si un cuento pesa tanto, el modelo ajustado puede reflejar sobre todo a ese cuento. El peso mide tamaño, no importancia literaria.
- **Los problemas.** Un PDF mal extraído puede meter encabezados y números de página como si fueran literatura; un archivo mal leído rompe tildes y eñes. Los avisos usan criterios nuestros, discutibles: si tu corpus es poesía, las líneas cortas son normales.
- **Limpiar no es mejorar.** Cada transformación hace algo concreto y tiene un riesgo. Probala, mirá qué cambia y decidí. Ninguna toca tildes ni eñes, y el original siempre se conserva.

| Transformación | Qué hace | Riesgo |
|---|---|---|
| `quitar_marca_BOM` | Saca el carácter invisible `U+FEFF` del principio | Ninguno relevante |
| `unificar_saltos_de_línea` | Pasa `\r\n` y `\r` a `\n` | Ninguno relevante |
| `recortar_espacios_al_final_de_línea` | Saca los espacios al final de cada línea | Ninguno relevante |
| `reducir_líneas_en_blanco` | Tres o más saltos seguidos pasan a dos | Pierde separaciones de énfasis |
| `unir_palabras_cortadas_por_guion` | `pa-\nlabra` pasa a `palabra` | Une mal compuestos reales (`franco-\nargentino`) |
| `unir_líneas_de_un_párrafo` | Un salto simple entre líneas con texto pasa a espacio | Rompe versos y diálogos en líneas cortas |

Los cuentos de Quiroga ya vienen limpios: lo comprobamos y las seis transformaciones dan cero cambios en los 26. Es un resultado válido (ese documento ya cumplía lo que el paso buscaba). Para ver cambios, probá con el texto de práctica o cargá `CORPUS_ELEGIDO = "ambos"`: los textos propios traen una marca BOM y saltos de línea de Windows.

### ✎ Para pensar

¿Qué harías con el cuento más largo: dejarlo, recortarlo o excluirlo? ¿Qué efecto esperás de cada decisión?

### Ahora compará versiones

Una **versión** del corpus lleva una **huella**: un código (SHA-256) que depende del texto exacto de cada documento incluido. Si cambia una coma, cambia la huella. Los experimentos guardan la huella, no el texto, para saber con qué corpus se hicieron.

Guardá una versión, excluí un documento (o aceptá una limpieza en uno incluido), guardá otra y apretá **Comparar las dos últimas versiones**: la tabla dice qué documentos se agregaron, se quitaron o se modificaron.

<details>
<summary>Profundización: cómo es un documento y cómo se leen los archivos</summary>

En el cuaderno, un documento es un diccionario con su texto y sus metadatos:

| Campo | Qué guarda |
|---|---|
| `id` | Identificador corto y único |
| `título`, `autoría`, `fecha`, `género` | Metadatos para citar y para agrupar |
| `fuente`, `condiciones_de_uso` | De dónde viene y bajo qué condiciones se puede usar |
| `texto_original` | Lo que se extrajo del archivo, sin tocar |
| `texto` | Lo que usa el resto del cuaderno, después de las transformaciones aceptadas |
| `incluido` | Si entra o no al corpus |
| `advertencias` | Problemas detectados al leer |

Cada formato tiene su lector (`leer_txt`, `leer_pdf`, `leer_epub`) y los tres devuelven lo mismo: el texto y una lista de advertencias. Si falta una librería opcional (`pypdf`, `ebooklib`), el error dice qué instalar. `revisar_texto` no corrige nada: solo avisa. `aplicar_transformaciones` parte siempre de `texto_original`, así que aplicarla dos veces no acumula cambios; `probar_transformaciones` hace lo mismo sin guardar nada, y es lo que usa el botón **Probar la limpieza**.
</details>

### Vía con código (alternativa al panel)

**Ver el peso de cada documento.** Lo mismo que el botón **Actualizar tabla y gráfico**, sin controles: los ocho documentos con más tokens y los duplicados exactos (dos textos con una coma de diferencia no son duplicados).

In [ ]:
tabla = tabla_de_documentos(documentos, tokenizador)
display(tabla.sort_values("tokens", ascending=False).head(8))
print(f"Duplicados exactos: {detectar_duplicados_exactos(documentos) or 'ninguno'}")

**Mirá el resultado.** La tabla está ordenada por tokens. Fijate qué documento tiene la mayor participación y cuántas veces supera al más corto.

**Probar una limpieza.** Aplicar tres transformaciones a un documento de los textos propios (que sí traen una marca BOM y saltos de línea de más) y ver el antes y el después. Elegí otro con `ID_DEL_DOCUMENTO`. Esta prueba no toca el corpus de trabajo `documentos`.

Antes de ejecutar, apostá: los textos propios empiezan con una marca BOM y tienen saltos de línea de Windows. ¿Qué paso va a hacer más cambios? ¿Alguno debería hacer cero?

In [ ]:
# Para ver efectos usamos los textos propios, que sí traen una marca BOM y saltos de línea de más.
# (Los cuentos de Quiroga ya vienen limpios de Wikisource: casi todos los pasos harían cero cambios.)
documentos_para_probar = cargar_corpus_de_textos_propios()
ID_DEL_DOCUMENTO = documentos_para_probar[0]["id"]
PASOS_A_APLICAR = ["quitar_marca_BOM", "unificar_saltos_de_línea", "reducir_líneas_en_blanco"]

documento_elegido = documento_con_id(documentos_para_probar, ID_DEL_DOCUMENTO)

registro = aplicar_transformaciones(documento_elegido, PASOS_A_APLICAR)
display(pd.DataFrame(registro))

pares = líneas_que_cambiaron(documento_elegido["texto_original"], documento_elegido["texto"])
print(f"Líneas que cambiaron (primeras): {len(pares)}")
for antes, después in pares[:2]:
    print(f"  antes:   {antes[:80]!r}")
    print(f"  después: {después[:80]!r}")

**Mirá el resultado.** La tabla dice cuántos cambios hizo cada paso y cuántos caracteres había antes y después.

- **Observación verificable:** un paso con cero cambios indica que ese documento ya cumplía lo que el paso buscaba.
- **Interpretación posible:** si un paso cambia mucho en un documento y nada en otro, es probable que vengan de fuentes con formatos distintos.
- **Límite:** esta lista de pasos es chica a propósito. Una limpieza real depende de cada fuente y se revisa a ojo.

**Comparar dos versiones.** Guardar una versión del corpus tal cual está, otra con un documento recortado, y comparar sus huellas. El documento vuelve a su texto completo al terminar.

Antes de ejecutar, apostá: ¿las dos huellas van a coincidir? ¿Cuántos documentos va a marcar la tabla como `modificado`?

In [ ]:
# Versión A: el corpus de trabajo tal cual está.
versión_a = tomar_versión(documentos, "tal cual")

# Versión B: el mismo corpus, pero con un documento reducido a sus primeros 3.000 caracteres.
documento_recortado = documentos[0]
texto_completo = documento_recortado["texto"]
documento_recortado["texto"] = texto_completo[:3000]
versión_b = tomar_versión(documentos, "con un documento recortado")
documento_recortado["texto"] = texto_completo  # volvemos al texto completo

comparación = comparar_versiones(versión_a, versión_b)
print(f"Huella A: {versión_a['huella'][:16]}…")
print(f"Huella B: {versión_b['huella'][:16]}…")
print(comparación["estado"].value_counts().to_string())

**Mirá el resultado.** Las huellas tienen que ser distintas: en la versión B un documento perdió texto. La huella cambia aunque se modifique un solo carácter, pero solo identifica el contenido, no su calidad.

---
## Estación 3. Mirá cómo el modelo lee los textos

**Desafío.** El modelo no lee cuentos: lee listas de números. Elegí un bloque y seguí un texto desde que son letras hasta que son tokens, y fijate cuál es exactamente la tarea de predicción.

Para llegar ahí hay cuatro pasos: **partir** los documentos en entrenamiento y validación, **tokenizar** cada uno (texto → lista de números), **armar bloques** y **verificar** que no haya fuga entre particiones. La celda de preparación los hace por vos; el panel te deja mirar el resultado.

**Preparación · estación 3.** Ejecutá la celda siguiente: define esos cuatro pasos, las huellas que los atan y el panel. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def huella_del_tokenizador(un_tokenizador):
    """Huella del vocabulario y del token de fin de texto. Si difiere, los números de dos modelos no son comparables."""
    descripción = {
        "vocabulario": un_tokenizador.get_vocab(),
        "fin_de_texto": un_tokenizador.eos_token_id,
    }
    return laboratorio.huella_de_json(descripción)


def huella_de_la_partición(partición):
    """Identifica quién quedó de cada lado y con qué contenido, junto con el modo y la semilla."""
    descripción = {"modo": partición["modo"], "semilla": partición["semilla"]}
    for lado in ("entrenamiento", "validación"):
        fichas = []
        for documento in partición[lado]:
            fichas.append([documento["id"], laboratorio.huella_de_texto(documento["texto"])])
        descripción[lado] = fichas
    return laboratorio.huella_de_json(descripción)


def calcular_identidades(documentos, un_tokenizador, partición, bloques_de_entrenamiento, bloques_de_validación):
    """Las huellas de cada paso de la preparación y una huella general que las reúne."""
    identidades = {
        "corpus": tomar_versión(documentos, "identidades")["huella"],
        "partición": huella_de_la_partición(partición),
        "tokenizador": huella_del_tokenizador(un_tokenizador),
        "bloques_de_entrenamiento": laboratorio.huella_de_json(bloques_de_entrenamiento["bloques"]),
        "bloques_de_validación": laboratorio.huella_de_json(bloques_de_validación["bloques"]),
        "largo_de_bloque": bloques_de_entrenamiento["largo_de_bloque"],
    }
    identidades["preparación"] = laboratorio.huella_de_json(identidades)
    return identidades


def motivos_de_desactualización(preparación, documentos, un_tokenizador):
    """Compara las huellas registradas con lo que hay ahora. Devuelve la lista de motivos (vacía si todo coincide)."""
    guardadas = preparación["identidades"]
    actuales = calcular_identidades(documentos, un_tokenizador, preparación["partición"],
                                    preparación["bloques_de_entrenamiento"], preparación["bloques_de_validación"])
    explicaciones = {
        "corpus": "El corpus cambió después de preparar los datos (estación 2).",
        "partición": "La partición no coincide con la registrada.",
        "tokenizador": "El tokenizador no es el que se usó para armar los bloques.",
        "bloques_de_entrenamiento": "Los bloques de entrenamiento no coinciden con los registrados.",
        "bloques_de_validación": "Los bloques de validación no coinciden con los registrados.",
    }
    motivos = []
    for clave, explicación in explicaciones.items():
        if actuales[clave] != guardadas[clave]:
            motivos.append(explicación)
    return motivos


def exigir_datos_vigentes(preparación, documentos, un_tokenizador):
    """Levanta un ValueError si los datos preparados ya no corresponden al corpus y al tokenizador actuales."""
    motivos = motivos_de_desactualización(preparación, documentos, un_tokenizador)
    if motivos:
        raise ValueError("Los datos preparados están desactualizados. " + " ".join(motivos)
                         + " Volvé a ejecutar la estación 3 (Preparar datos).")


def partir_documentos(documentos, modo="por_documento", ids_de_validación=None, proporción_de_validación=0.1, semilla=42):
    """Separa los documentos incluidos en entrenamiento y validación y devuelve un diccionario.

    Cada elemento de las listas es {"id", "texto"}. En modo "por_documento" ningún documento
    aparece en las dos listas. En modo "por_párrafo" cada documento se reparte entre las dos.
    """
    incluidos = []
    for documento in documentos:
        if documento["incluido"]:
            incluidos.append(documento)
    azar = random.Random(semilla)
    advertencias = []
    entrenamiento = []
    validación = []

    if modo == "por_documento":
        if len(incluidos) < 3:
            raise ValueError(
                f"Hay {len(incluidos)} documentos incluidos: con menos de 3 no se puede partir por documento. "
                "Probá modo='por_párrafo' (con sus límites) o sumá documentos."
            )
        if ids_de_validación is None:
            cantidad = max(1, round(proporción_de_validación * len(incluidos)))
            ids_de_validación = []
            for sorteado in azar.sample(incluidos, cantidad):
                ids_de_validación.append(sorteado["id"])
        ids_incluidos = []
        for documento in incluidos:
            ids_incluidos.append(documento["id"])
        for identificador in ids_de_validación:
            if identificador not in ids_incluidos:
                raise ValueError(f"El documento de validación {identificador!r} no está entre los incluidos.")
        for documento in incluidos:
            ficha = {"id": documento["id"], "texto": documento["texto"]}
            if documento["id"] in ids_de_validación:
                validación.append(ficha)
            else:
                entrenamiento.append(ficha)
        if not entrenamiento:
            raise ValueError("Todos los documentos quedaron en validación: no queda nada para entrenar.")
    elif modo == "por_párrafo":
        advertencias.append(
            "Partición por párrafo: los mismos textos aparecen en entrenamiento y validación. "
            "La pérdida de validación es más optimista que con documentos no vistos."
        )
        for documento in incluidos:
            párrafos = []
            for párrafo in documento["texto"].split("\n\n"):
                if párrafo.strip():
                    párrafos.append(párrafo)
            cantidad_de_párrafos = len(párrafos)
            cantidad = max(1, round(proporción_de_validación * cantidad_de_párrafos))
            posiciones_de_validación = set(azar.sample(range(cantidad_de_párrafos), min(cantidad, cantidad_de_párrafos)))
            párrafos_de_entrenamiento = []
            párrafos_de_validación = []
            posición = 0
            for párrafo in párrafos:
                if posición in posiciones_de_validación:
                    párrafos_de_validación.append(párrafo)
                else:
                    párrafos_de_entrenamiento.append(párrafo)
                posición += 1
            if párrafos_de_entrenamiento:
                entrenamiento.append({"id": documento["id"], "texto": "\n\n".join(párrafos_de_entrenamiento)})
            if párrafos_de_validación:
                validación.append({"id": documento["id"] + "#validación", "texto": "\n\n".join(párrafos_de_validación)})
    else:
        raise ValueError(f"Modo de partición desconocido: {modo!r}. Usá 'por_documento' o 'por_párrafo'.")

    return {"modo": modo, "entrenamiento": entrenamiento, "validación": validación, "advertencias": advertencias,
            "semilla": semilla}


def tokenizar_documentos(lista_de_documentos, tokenizador):
    """Convierte cada {"id", "texto"} en {"id", "ids"}: la lista de identificadores de tokens del texto.

    El tokenizador de este modelo no agrega tokens especiales por su cuenta: el token de fin de
    texto lo ponemos nosotros al armar los bloques.
    """
    tokenizados = []
    for documento in lista_de_documentos:
        identificadores = tokenizador(documento["texto"])["input_ids"]
        tokenizados.append({"id": documento["id"], "ids": identificadores})
    return tokenizados


def armar_bloques(documentos_tokenizados, largo_de_bloque, id_de_fin_de_texto):
    """Corta la cadena de todos los documentos (cada uno seguido de fin de texto) en bloques de largo fijo.

    Devuelve un diccionario con:
      - "bloques": lista de listas de identificadores, todas de largo `largo_de_bloque`;
      - "origen": para cada bloque, una lista de tramos (id del documento, desde, hasta) dentro del bloque;
      - "tokens_totales": cuántos tokens tenía la cadena completa;
      - "tokens_descartados": los que sobran al final y no alcanzan para un bloque entero.
    """
    if largo_de_bloque < 8:
        raise ValueError("El largo de bloque tiene que ser de al menos 8 tokens.")
    cadena = []
    dueños = []  # de qué documento viene cada token de la cadena
    for documento in documentos_tokenizados:
        for identificador in documento["ids"]:
            cadena.append(identificador)
            dueños.append(documento["id"])
        cadena.append(id_de_fin_de_texto)
        dueños.append(documento["id"])  # el fin de texto se atribuye al documento que cierra

    cantidad_de_bloques = len(cadena) // largo_de_bloque
    bloques = []
    origen = []
    for número in range(cantidad_de_bloques):
        desde = número * largo_de_bloque
        hasta = desde + largo_de_bloque
        bloques.append(cadena[desde:hasta])
        tramos = []
        inicio_del_tramo = 0
        for posición in range(1, largo_de_bloque + 1):
            termina_tramo = posición == largo_de_bloque or dueños[desde + posición] != dueños[desde + inicio_del_tramo]
            if termina_tramo:
                tramos.append((dueños[desde + inicio_del_tramo], inicio_del_tramo, posición))
                inicio_del_tramo = posición
        origen.append(tramos)
    return {
        "bloques": bloques,
        "origen": origen,
        "largo_de_bloque": largo_de_bloque,
        "tokens_totales": len(cadena),
        "tokens_descartados": len(cadena) - cantidad_de_bloques * largo_de_bloque,
    }


def buscar_fuga_entre_particiones(bloques_de_entrenamiento, bloques_de_validación, largo_de_ventana=16):
    """Busca repeticiones entre las dos particiones y devuelve un diccionario de diagnóstico.

    - "bloques_idénticos": bloques que están enteros en las dos particiones. Esto sí es un error.
    - "ventanas_compartidas": tramos de `largo_de_ventana` tokens seguidos que aparecen en las dos.
      Puede pasar con frases hechas y no es un error por sí solo: es una señal para mirar.
    """
    bloques_de_entrenamiento_como_tuplas = set()
    ventanas_de_entrenamiento = set()
    for bloque in bloques_de_entrenamiento["bloques"]:
        bloques_de_entrenamiento_como_tuplas.add(tuple(bloque))
        for posición in range(len(bloque) - largo_de_ventana + 1):
            ventanas_de_entrenamiento.add(tuple(bloque[posición:posición + largo_de_ventana]))
    idénticos = 0
    ventanas_compartidas = set()
    for bloque in bloques_de_validación["bloques"]:
        if tuple(bloque) in bloques_de_entrenamiento_como_tuplas:
            idénticos += 1
        for posición in range(len(bloque) - largo_de_ventana + 1):
            ventana = tuple(bloque[posición:posición + largo_de_ventana])
            if ventana in ventanas_de_entrenamiento:
                ventanas_compartidas.add(ventana)
    return {"bloques_idénticos": idénticos, "ventanas_compartidas": len(ventanas_compartidas),
            "largo_de_ventana": largo_de_ventana}


def preparar_datos(documentos, tokenizador, largo_de_bloque=256, modo_de_partición="por_documento",
                   ids_de_validación=None, proporción_de_validación=0.1, semilla=42):
    """Hace los cuatro pasos de la estación 3 (partir, tokenizar, armar bloques, verificar) y los devuelve juntos.

    Incluye `identidades`: las huellas del corpus, la partición, el tokenizador y los bloques.
    """
    partición = partir_documentos(documentos, modo_de_partición, ids_de_validación, proporción_de_validación, semilla)
    tokenizados_de_entrenamiento = tokenizar_documentos(partición["entrenamiento"], tokenizador)
    tokenizados_de_validación = tokenizar_documentos(partición["validación"], tokenizador)
    bloques_de_entrenamiento = armar_bloques(tokenizados_de_entrenamiento, largo_de_bloque, tokenizador.eos_token_id)
    bloques_de_validación = armar_bloques(tokenizados_de_validación, largo_de_bloque, tokenizador.eos_token_id)
    if not bloques_de_entrenamiento["bloques"] or not bloques_de_validación["bloques"]:
        raise ValueError(
            "Alguna partición quedó sin bloques completos: el texto es más corto que un bloque. "
            "Bajá el largo de bloque o sumá texto."
        )
    verificación = buscar_fuga_entre_particiones(bloques_de_entrenamiento, bloques_de_validación)
    if verificación["bloques_idénticos"]:
        raise ValueError(f"{verificación['bloques_idénticos']} bloques están en entrenamiento y en validación: hay fuga.")
    identidades = calcular_identidades(documentos, tokenizador, partición, bloques_de_entrenamiento, bloques_de_validación)
    return {
        "partición": partición,
        "bloques_de_entrenamiento": bloques_de_entrenamiento,
        "bloques_de_validación": bloques_de_validación,
        "verificación": verificación,
        "largo_de_bloque": largo_de_bloque,
        "identidades": identidades,
    }


def piezas_de_los_tokens(identificadores, tokenizador):
    """El texto de cada token por separado, para mostrarlo. Un carácter partido en bytes se marca como tal."""
    piezas = []
    for identificador in identificadores:
        piezas.append(tokenizador.decode([identificador]))
    return piezas


def ventana_de_posiciones(largo, posición, ancho=96):
    """Rango (desde, hasta) de posiciones a mostrar, centrado en `posición` y sin pasarse de los bordes."""
    desde = max(0, min(posición - ancho // 2, largo - ancho))
    return desde, min(largo, desde + ancho)


def armar_estación_3():
    """Vistas enlazadas de texto, tokens y bloques. Guarda el resultado en la variable global `preparación`."""
    estación = laboratorio.nueva_estación("3-bloques")

    opciones_de_largo = [64, 128, 256, 512]
    if preparación["largo_de_bloque"] not in opciones_de_largo:
        opciones_de_largo.append(preparación["largo_de_bloque"])  # un largo propio también tiene que poder elegirse
    largo = estación.agregar(widgets.Dropdown(options=opciones_de_largo, value=preparación["largo_de_bloque"]))
    laboratorio.control(largo, "Largo de bloque (tokens)", "LARGO_DE_BLOQUE", ancho="22em", tipo="datos",
                        ayuda="Cuántos tokens tiene cada bloque. Más largo: más contexto, pero cada paso cuesta más memoria y tiempo.",
                        prueba="poné 64, apretá «Preparar datos» y mirá cuántos bloques hay ahora.")
    modo = estación.agregar(widgets.Dropdown(options=["por_documento", "por_párrafo"], value=MODO_DE_PARTICIÓN))
    laboratorio.control(modo, "Cómo partir", "MODO_DE_PARTICIÓN", ancho="22em", tipo="datos",
                        ayuda="«por_documento»: cada texto queda entero de un lado. «por_párrafo»: se reparte, y la validación sale más optimista.",
                        prueba="cambiá a «por_párrafo» y mirá el aviso del resumen.")
    botón_preparar = estación.agregar(laboratorio.botón("Preparar datos", "Parte, tokeniza y arma bloques con estos valores", "primary"))
    estado = estación.agregar(widgets.HTML(utils.html_estado("completado", "datos de la celda anterior")))
    partición = estación.agregar(widgets.ToggleButtons(options=["entrenamiento", "validación"], value="entrenamiento"))
    laboratorio.control(partición, "Conjunto", "partición", ancho="34em", tipo="vista",
                        ayuda="Los bloques con los que se ajusta el modelo (entrenamiento) o los reservados para medir (validación).",
                        prueba="pasá de uno a otro y mirá cómo cambia el documento de origen.")
    bloque = estación.agregar(widgets.IntSlider(value=0, min=0, max=0, continuous_update=False))
    laboratorio.control(bloque, "Bloque", "número de bloque", tipo="vista",
                        ayuda="Cuál de los bloques mirás. No cambia los datos.",
                        prueba="buscá uno cuyo origen tenga dos documentos: ahí está el fin de texto.")
    posición = estación.agregar(widgets.IntSlider(value=10, min=0, max=preparación["largo_de_bloque"] - 2, continuous_update=False))
    laboratorio.control(posición, "Posición t (contexto hasta t, objetivo t+1)", "t", tipo="vista",
                        ayuda="En qué punto del bloque mirás qué ve el modelo y qué intenta predecir.",
                        prueba="movela y fijate que el objetivo es siempre el token que sigue.")
    resumen = estación.agregar(widgets.HTML())
    vista_texto = estación.agregar(widgets.HTML())
    vista_tokens = estación.agregar(widgets.HTML())
    vista_par = estación.agregar(widgets.HTML())
    vista_origen = estación.agregar(widgets.HTML())
    código = estación.agregar(widgets.HTML())

    def datos_de_la_partición():
        clave = "bloques_de_entrenamiento" if partición.value == "entrenamiento" else "bloques_de_validación"
        return preparación[clave]

    def dibujar(_=None):
        datos = datos_de_la_partición()
        bloque.max = max(0, len(datos["bloques"]) - 1)
        posición.max = max(0, datos["largo_de_bloque"] - 2)
        identificadores = datos["bloques"][bloque.value]
        piezas = piezas_de_los_tokens(identificadores, tokenizador)
        desde, hasta = ventana_de_posiciones(len(identificadores), posición.value)
        resumen.value = (f"<b>{len(datos['bloques'])}</b> bloques de <b>{datos['largo_de_bloque']}</b> tokens · "
                         f"tokens descartados al final: <b>{datos['tokens_descartados']}</b> · "
                         f"modo: <b>{preparación['partición']['modo']}</b>"
                         + utils.html_avisos(preparación["partición"]["advertencias"]))
        vista_texto.value = utils.texto_con_espacios_visibles(tokenizador.decode(identificadores))
        fichas = utils.html_tokens(piezas[desde:hasta], identificadores[desde:hasta],
                                   resaltar={posición.value - desde, posición.value + 1 - desde})
        vista_tokens.value = f"<div style='font-size:.85em'>Tokens {desde} a {hasta - 1} de {len(identificadores) - 1} (borde grueso: contexto actual y objetivo)</div>" + fichas
        contexto = piezas[max(0, posición.value - 7): posición.value + 1]
        objetivo = piezas[posición.value + 1]
        piezas_del_contexto = []
        for pieza in contexto:
            piezas_del_contexto.append(utils.pieza_visible(pieza))
        texto_del_contexto = " ".join(piezas_del_contexto)
        vista_par.value = utils.html_tabla(
            ["qué ve el modelo en la posición t", "token objetivo (posición t+1)"],
            [[texto_del_contexto, utils.pieza_visible(objetivo)]],
        )
        filas = []
        for documento_de_origen, desde_el_tramo, hasta_el_tramo in datos["origen"][bloque.value]:
            filas.append([documento_de_origen, f"{desde_el_tramo} a {hasta_el_tramo - 1}"])
        vista_origen.value = utils.html_tabla(["documento de origen", "posiciones dentro del bloque"], filas)
        llamada = laboratorio.texto_de_llamada("preparar_datos", dict(
            documentos=laboratorio.Código("documentos"), tokenizador=laboratorio.Código("tokenizador"),
            largo_de_bloque=largo.value, modo_de_partición=modo.value), "preparación")
        código.value = f"<pre style='font-family:{utils.FUENTE_CODIGO}'>{utils.escapar(llamada)}</pre>"

    def al_preparar(_):
        global preparación
        with laboratorio.Ocupado([botón_preparar], estado, "Preparando datos", utils.html_estado) as ocupado:
            ids = IDS_DE_VALIDACIÓN if modo.value == "por_documento" else None
            preparación = preparar_datos(documentos, tokenizador, largo.value, modo.value, ids,
                                         PROPORCIÓN_DE_VALIDACIÓN, SEMILLA_DE_PARTICIÓN)
            bloque.value = 0
            dibujar()
        if ocupado.error is not None:
            resumen.value = utils.html_aviso(str(ocupado.error), "error")

    for control in (partición, bloque, posición):
        estación.observar(control, dibujar)
    estación.al_hacer_clic(botón_preparar, al_preparar)
    dibujar()

    explorar = widgets.VBox([resumen, laboratorio.fila_de_control(partición), laboratorio.fila_de_control(bloque),
                             laboratorio.fila_de_control(posición), vista_texto, vista_tokens, vista_par, vista_origen])
    modificar = widgets.VBox([laboratorio.fila_de_control(largo), laboratorio.fila_de_control(modo), botón_preparar, estado, código])
    return laboratorio.panel_de_tres_niveles(explorar, widgets.VBox([resumen, vista_origen]), modificar)

### Preparar los datos

Los paneles de las estaciones 3 a 5 parten de esta celda. Divide el corpus en **entrenamiento** (los textos con los que se ajusta el modelo) y **validación** (textos reservados, que el modelo no ve al ajustarse y sirven para medir con textos nuevos), convierte cada documento en tokens y arma los **bloques**.

| Parámetro | Qué es | Qué cambia si lo modificás |
|---|---|---|
| `LARGO_DE_BLOQUE` | Cuántos tokens tiene cada bloque (el máximo de este modelo es 1024) | Más largo: el modelo ve más contexto por ejemplo, pero cada paso cuesta más memoria y tiempo. Más corto: bloques más baratos, con menos contexto. 256 tokens son unas 190 palabras en español |
| `MODO_DE_PARTICIÓN` | `"por_documento"` o `"por_párrafo"` | Por documento: cada cuento queda entero de un lado. Por párrafo (para corpus muy chicos): el mismo cuento aparece de los dos lados y la validación sale más optimista |
| `IDS_DE_VALIDACIÓN` | Qué documentos se reservan | `adulto_07` e `infantil_04` se apartaron antes de mirar ningún resultado; con `None` se sortean |
| `PROPORCIÓN_DE_VALIDACIÓN` y `SEMILLA_DE_PARTICIÓN` | Cuántos reservar y con qué semilla, si se sortea | Solo importan si `IDS_DE_VALIDACIÓN` es `None` |

### ✎ Antes de ejecutar

Si el corpus tiene unos 90.000 tokens y los bloques son de 256, ¿cuántos bloques esperás aproximadamente?

In [ ]:
LARGO_DE_BLOQUE = 256
MODO_DE_PARTICIÓN = "por_documento"
IDS_DE_VALIDACIÓN = ["adulto_07", "infantil_04"]  # los cuentos reservados; con None se sortea
PROPORCIÓN_DE_VALIDACIÓN = 0.1                   # solo se usa si IDS_DE_VALIDACIÓN es None
SEMILLA_DE_PARTICIÓN = 42

preparación = preparar_datos(
    documentos=documentos,
    tokenizador=tokenizador,
    largo_de_bloque=LARGO_DE_BLOQUE,
    modo_de_partición=MODO_DE_PARTICIÓN,
    ids_de_validación=IDS_DE_VALIDACIÓN,
    proporción_de_validación=PROPORCIÓN_DE_VALIDACIÓN,
    semilla=SEMILLA_DE_PARTICIÓN,
)

for nombre in ("bloques_de_entrenamiento", "bloques_de_validación"):
    datos = preparación[nombre]
    print(f"{nombre}: {len(datos['bloques'])} bloques · tokens totales {utils.con_miles(datos['tokens_totales'])} · descartados {datos['tokens_descartados']}")
print("Verificación de fuga:", preparación["verificación"])

**Mirá el resultado.** Cada partición informa cuántos bloques tiene y cuántos tokens quedaron afuera: los que sobran al final y no completan un bloque se descartan, y son siempre menos que el largo de bloque. La validación tiene pocos bloques (dos cuentos), así que es probable que su pérdida sea una estimación ruidosa. Si `ventanas_compartidas` no es cero, mirá cuáles son antes de descartar el resultado: pueden ser frases corrientes del español y no una fuga.

### Panel: texto, tokens y bloques

Elegí el conjunto, el bloque y la posición `t`. Todo lo que se ve corresponde al mismo bloque: el **texto** (con espacios `·` y saltos de línea `↵` visibles), los **tokens** como fichas con su número, el par **contexto → objetivo** de la posición `t` y los **documentos de origen** del bloque.

1. Pasá de **entrenamiento** a **validación** y mirá cómo cambia el documento de origen.
2. Buscá un bloque cuyo origen tenga dos documentos: entre los dos cuentos hay un token especial, el fin de texto.
3. Mové `t` y fijate que el objetivo es siempre el token que sigue.
4. En **Modificar**, poné largo de bloque 64 y apretá **Preparar datos** (el único control que cambia los datos): ¿cuántos bloques hay ahora y cuántos tokens se descartan?

In [ ]:
display(armar_estación_3())

### Qué miraste en los bloques

- **La tarea.** En cada posición el modelo recibe lo que hay hasta `t` (el **contexto**) e intenta predecir el token `t+1` (el **objetivo**). Un bloque de N tokens produce N−1 pares: el primer token nunca es objetivo.
- **El fin de texto.** Entre un cuento y el siguiente se pega el token `<|endoftext|>`. Así el modelo ve dónde termina un texto y empieza otro; sin ese separador, el final de un cuento y el principio del siguiente parecerían una sola oración.
- **Partir antes de armar bloques.** Se hace en ese orden para que ningún tramo del mismo texto caiga en los dos lados.

### Cómo se mide el error: la pérdida

Para saber si el modelo predice bien, se compara su predicción en cada posición con el token que realmente vino. La **pérdida** (*loss*) es, en promedio, cuánto «le sorprendió» al modelo el token que realmente vino. Si le había asignado probabilidad 0,5, la sorpresa es 0,69; si le había asignado 0,01, es 4,6. Cada sorpresa se calcula como `−log p` (con `p` la probabilidad que el modelo le dio al token real), y la pérdida es el promedio de todas.

En la celda siguiente calculamos la pérdida de un bloque dos veces: con el modelo y a mano, con el desplazamiento a la vista. Al ajustar, `labels` e `input_ids` son los mismos números y **el modelo hace el desplazamiento por dentro**; si lo hiciéramos nosotros, quedaría aprendiendo a predecir dos tokens más adelante.

In [ ]:
bloque_de_prueba = preparación["bloques_de_entrenamiento"]["bloques"][0]
entrada = torch.tensor([bloque_de_prueba], device=DISPOSITIVO)

with torch.no_grad():
    salida = modelo_base(input_ids=entrada, labels=entrada)
    logits = modelo_base(input_ids=entrada).logits

# A mano: la predicción de la posición t se compara con el token t+1.
predicciones = logits[0, :-1, :]      # posiciones 0 a N-2: lo que el modelo dice del token siguiente
objetivos = entrada[0, 1:]            # tokens 1 a N-1: lo que realmente vino después
pérdida_a_mano = torch.nn.functional.cross_entropy(predicciones, objetivos)

print(f"Pérdida que devuelve el modelo (labels = input_ids): {salida.loss.item():.5f}")
print(f"Pérdida calculada a mano, desplazando acá:           {pérdida_a_mano.item():.5f}")
print(f"Diferencia: {abs(salida.loss.item() - pérdida_a_mano.item()):.2e}")

**Mirá el resultado.** Las dos pérdidas coinciden (la diferencia es del orden del redondeo), así que en este modelo y con esta versión de `transformers`, `labels=input_ids` ya incluye el desplazamiento. En otro motor el desplazamiento puede hacerse en otro lugar. Si aparece un aviso `loss_type=None`, es de `transformers` y no cambia el valor.

### ✎ Para pensar

Si hubiéramos pasado las `labels` ya desplazadas, ¿qué tarea estaría resolviendo el modelo?

<details>
<summary>Profundización: huellas que atan cada paso al anterior</summary>

Si cambia un texto del corpus, el tokenizador o la partición, los bloques que ya tenés dejan de corresponder. Para detectarlo, cada paso queda atado al anterior con una **huella**: un resumen del contenido (SHA-256) que cambia si cambia un solo carácter. Es como el número de lote de un producto: no dice si es bueno, dice de qué partida salió.

| Huella | Qué identifica | Sobre qué se calcula |
|---|---|---|
| `corpus` | Los textos incluidos en este momento | El texto de cada documento incluido |
| `partición` | Quién quedó en entrenamiento y quién en validación | Modo, semilla, e id y texto de cada documento de cada lado |
| `tokenizador` | Cómo se corta el texto en tokens | Vocabulario completo y token de fin de texto |
| `bloques_de_entrenamiento`, `bloques_de_validación` | Lo que realmente recibe el modelo | Los identificadores de tokens de cada bloque |
| `preparación` | Todo lo anterior junto | Las huellas de arriba y el largo de bloque |

`preparar_datos` las devuelve en `preparación["identidades"]`. Después, `crear_configuración`, `entrenar` y la reanudación las comprueban de nuevo: si el corpus o el tokenizador ya no coinciden con los bloques, se niegan a seguir y dicen cuál cambió. No se vuelve a tokenizar el corpus en cada comprobación: se verifica que lo registrado siga siendo lo mismo.

**Fuga entre particiones.** `buscar_fuga_entre_particiones` hace dos controles: los **bloques idénticos** en entrenamiento y validación (un error: el modelo vería lo que después se le evalúa) y las **ventanas compartidas** de 16 tokens (una señal, no un error: pueden ser frases hechas o un cuento citado dentro de otro).

**Partición por párrafo.** Es la alternativa para corpus muy chicos (partir por documento exige al menos 3 documentos). Es una evaluación más débil: el mismo cuento aparece de los dos lados y la pérdida de validación sale más optimista que con un texto realmente nuevo.

**Sobrantes.** `armar_bloques` corta la cadena de todos los documentos (cada uno seguido del fin de texto) en tramos de `largo_de_bloque` tokens y descarta lo que sobra al final: `tokens_descartados` dice cuánto fue. Guarda además el `origen` de cada bloque, porque un bloque puede contener el final de un cuento, el separador y el principio de otro.
</details>

---
## Estación 4. Configurá el ajuste

**Desafío.** Decidí cuánto, con qué datos y a qué ritmo vas a ajustar el modelo. Mové los controles y mirá cómo cambian los números: el ajuste no empieza hasta que lo lances en la estación 5.

**Preparación · estación 4.** Ejecutá la celda siguiente: define la configuración, la cuenta de pasos y épocas, la validación de decisiones y el panel. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
@dataclasses.dataclass(frozen=True)
class Configuración:
    """Todas las decisiones de un experimento. Es inmutable: no se puede modificar después de crearla."""

    # Identificación
    nombre: str
    hipótesis: str
    notas: str
    destino: str
    # Modelo y datos
    modelo_base: str
    revisión_del_modelo: str
    huella_del_corpus: str
    modo_de_partición: str
    ids_de_validación: tuple
    largo_de_bloque: int
    bloques_de_entrenamiento: int
    bloques_de_validación: int
    # Optimización
    tasa_de_aprendizaje: float
    tamaño_de_lote: int
    acumulación_de_gradientes: int
    pasos_de_calentamiento: int
    # Duración
    unidad_de_duración: str
    duración_pedida: int
    pasos_por_época: int
    pasos_totales: int
    # Seguimiento
    evaluar_cada: int
    guardar_cada: int
    comienzos_de_muestras: tuple
    # Azar y entorno
    semilla: int
    dispositivo: str
    precisión: str
    # Identidad de los datos (ver «Identidades verificables» en la estación 3)
    huella_de_la_preparación: str = ""
    huella_de_la_partición: str = ""
    huella_del_tokenizador: str = ""
    huella_de_los_bloques_de_entrenamiento: str = ""
    huella_de_los_bloques_de_validación: str = ""

    @property
    def lote_efectivo(self):
        return self.tamaño_de_lote * self.acumulación_de_gradientes

    @property
    def microlotes_totales(self):
        return self.pasos_totales * self.acumulación_de_gradientes

    @property
    def bloques_que_se_dejan_afuera_por_época(self):
        """Bloques que no entran en ningún paso de una época (no completan un lote efectivo)."""
        return self.bloques_de_entrenamiento - self.pasos_por_época * self.lote_efectivo

    @property
    def carpeta_del_experimento(self):
        return Path(self.destino) / utils.limpiar_para_nombre_de_archivo(self.nombre)

    def como_diccionario(self):
        datos = dataclasses.asdict(self)
        datos["lote_efectivo"] = self.lote_efectivo
        return datos

    def huella(self):
        """SHA-256 de la configuración. Dos configuraciones iguales tienen la misma huella."""
        texto = json.dumps(dataclasses.asdict(self), ensure_ascii=False, sort_keys=True)
        return laboratorio.huella_de_texto(texto)


def cuentas_de_la_época(cantidad_de_bloques, tamaño_de_lote, acumulación_de_gradientes):
    """Cuántos microlotes, pasos y bloques entran en una época. Lo que no completa un paso queda afuera."""
    microlotes_completos = cantidad_de_bloques // tamaño_de_lote
    pasos_por_época = microlotes_completos // acumulación_de_gradientes
    microlotes_usados = pasos_por_época * acumulación_de_gradientes
    bloques_usados = microlotes_usados * tamaño_de_lote
    return {
        "microlotes_completos": microlotes_completos,
        "pasos_por_época": pasos_por_época,
        "microlotes_usados": microlotes_usados,
        "bloques_usados": bloques_usados,
        "bloques_afuera": cantidad_de_bloques - bloques_usados,
    }


def crear_configuración(preparación, documentos, tokenizador, nombre, hipótesis="", notas="", destino=None,
                        tasa_de_aprendizaje=5e-5, tamaño_de_lote=4, acumulación_de_gradientes=2,
                        unidad_de_duración="pasos", duración=60, pasos_de_calentamiento=5,
                        evaluar_cada=20, guardar_cada=30, comienzos_de_muestras=None,
                        semilla=42, dispositivo=None, permitir_carpeta_existente=False):
    """Valida los datos y devuelve una Configuración congelada. Si algo no tiene sentido, levanta ValueError.

    Se niega si la preparación ya no corresponde al corpus o al tokenizador actuales.
    """
    exigir_datos_vigentes(preparación, documentos, tokenizador)
    destino = str(destino or RAÍZ_DE_EXPERIMENTOS)
    dispositivo = str(dispositivo or DISPOSITIVO)
    comienzos_de_muestras = tuple(comienzos_de_muestras or ("Una mañana de otoño,", "Esa noche"))
    bloques_de_entrenamiento = len(preparación["bloques_de_entrenamiento"]["bloques"])
    bloques_de_validación = len(preparación["bloques_de_validación"]["bloques"])

    if not nombre.strip():
        raise ValueError("Ponele un nombre al experimento.")
    if tasa_de_aprendizaje <= 0:
        raise ValueError("La tasa de aprendizaje tiene que ser mayor que 0.")
    if tamaño_de_lote < 1 or acumulación_de_gradientes < 1:
        raise ValueError("El tamaño de lote y la acumulación tienen que ser de al menos 1.")
    if tamaño_de_lote > bloques_de_entrenamiento:
        raise ValueError(f"El microlote ({tamaño_de_lote}) es mayor que la cantidad de bloques ({bloques_de_entrenamiento}).")
    if unidad_de_duración not in ("pasos", "épocas"):
        raise ValueError("La unidad de duración es 'pasos' o 'épocas'.")
    if duración < 1:
        raise ValueError("La duración tiene que ser de al menos 1.")

    cuentas = cuentas_de_la_época(bloques_de_entrenamiento, tamaño_de_lote, acumulación_de_gradientes)
    pasos_por_época = cuentas["pasos_por_época"]
    if pasos_por_época < 1:
        raise ValueError(
            f"Con microlote {tamaño_de_lote} y acumulación {acumulación_de_gradientes} no alcanza para un paso "
            f"completo por época ({bloques_de_entrenamiento} bloques). Bajá la acumulación o el microlote."
        )
    pasos_totales = duración if unidad_de_duración == "pasos" else duración * pasos_por_época
    if evaluar_cada < 1 or guardar_cada < 1:
        raise ValueError("La frecuencia de evaluación y la de guardado tienen que ser de al menos 1 paso.")
    if evaluar_cada > pasos_totales:
        raise ValueError(f"Evaluar cada {evaluar_cada} pasos no ocurre nunca en un ajuste de {pasos_totales} pasos.")
    if pasos_de_calentamiento >= pasos_totales:
        raise ValueError("El calentamiento tiene que ser menor que la cantidad total de pasos.")

    ids_de_validación = []
    for documento_de_validación in preparación["partición"]["validación"]:
        ids_de_validación.append(documento_de_validación["id"])

    configuración = Configuración(
        nombre=nombre.strip(), hipótesis=hipótesis, notas=notas, destino=destino,
        modelo_base=MODELO_BASE, revisión_del_modelo=REVISIÓN_DEL_MODELO, huella_del_corpus=preparación["identidades"]["corpus"],
        modo_de_partición=preparación["partición"]["modo"],
        ids_de_validación=tuple(ids_de_validación),
        largo_de_bloque=preparación["largo_de_bloque"],
        bloques_de_entrenamiento=bloques_de_entrenamiento, bloques_de_validación=bloques_de_validación,
        tasa_de_aprendizaje=float(tasa_de_aprendizaje), tamaño_de_lote=int(tamaño_de_lote),
        acumulación_de_gradientes=int(acumulación_de_gradientes), pasos_de_calentamiento=int(pasos_de_calentamiento),
        unidad_de_duración=unidad_de_duración, duración_pedida=int(duración), pasos_por_época=pasos_por_época,
        pasos_totales=int(pasos_totales), evaluar_cada=int(evaluar_cada), guardar_cada=int(guardar_cada),
        comienzos_de_muestras=comienzos_de_muestras, semilla=int(semilla), dispositivo=dispositivo,
        precisión="float32",
        huella_de_la_preparación=preparación["identidades"]["preparación"],
        huella_de_la_partición=preparación["identidades"]["partición"],
        huella_del_tokenizador=preparación["identidades"]["tokenizador"],
        huella_de_los_bloques_de_entrenamiento=preparación["identidades"]["bloques_de_entrenamiento"],
        huella_de_los_bloques_de_validación=preparación["identidades"]["bloques_de_validación"],
    )
    if configuración.carpeta_del_experimento.exists() and not permitir_carpeta_existente:
        raise ValueError(
            f"Ya existe {configuración.carpeta_del_experimento}. Elegí otro nombre: "
            "no se pisan experimentos guardados."
        )
    return configuración


def diagnóstico_de_hardware():
    """Qué máquina hay. Solo informa: ninguna celda del cuaderno decide nada a partir de esto."""
    import os
    import platform

    diagnóstico = {
        "dispositivo_en_uso": str(DISPOSITIVO),
        "procesador": platform.processor() or platform.machine(),
        "hilos_de_cpu": torch.get_num_threads(),
        "memoria_ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
        "cuda_disponible": bool(torch.cuda.is_available()),
        "mps_disponible": bool(torch.backends.mps.is_available()),
    }
    if torch.cuda.is_available():
        diagnóstico["gpu"] = torch.cuda.get_device_name(0)
        diagnóstico["memoria_gpu_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    return diagnóstico


configuración_consolidada = None  # la que usa la estación 5; solo cambia con el botón de abajo


def armar_estación_4():
    """Panel de diseño del experimento. Deja la configuración elegida en `configuración_consolidada`."""
    estación = laboratorio.nueva_estación("4-diseño")

    nombre = estación.agregar(widgets.Text(value="ensayo_quiroga_" + time.strftime("%Y%m%d_%H%M%S")))  # con fecha y hora, para no pisar otro
    hipótesis = estación.agregar(widgets.Textarea(value="", placeholder="Una hipótesis que se pueda desmentir con lo que vas a medir", rows=2))
    notas = estación.agregar(widgets.Textarea(value="", rows=2))
    tasa = estación.agregar(widgets.FloatLogSlider(value=5e-5, base=10, min=-6, max=-3, step=0.1, readout_format=".1e", continuous_update=False))
    lote = estación.agregar(widgets.IntSlider(value=4, min=1, max=16, continuous_update=False))
    acumulación = estación.agregar(widgets.IntSlider(value=2, min=1, max=16, continuous_update=False))
    unidad = estación.agregar(widgets.ToggleButtons(options=["pasos", "épocas"], value="pasos"))
    duración = estación.agregar(widgets.IntSlider(value=60, min=1, max=2000, continuous_update=False))
    evaluar = estación.agregar(widgets.IntSlider(value=20, min=1, max=500, continuous_update=False))
    guardar = estación.agregar(widgets.IntSlider(value=30, min=1, max=1000, continuous_update=False))
    semilla = estación.agregar(widgets.IntText(value=42))
    comienzos = estación.agregar(widgets.Textarea(value="Una mañana de otoño,\nEsa noche", rows=3))
    laboratorio.control(nombre, "Nombre del experimento", "nombre", tipo="experimento",
                        ayuda="Es el nombre de la carpeta donde queda todo. Nunca pisa un experimento guardado.",
                        prueba="dejá el que viene: ya lleva fecha y hora.")
    laboratorio.control(hipótesis, "Hipótesis", "hipótesis", tipo="experimento",
                        ayuda="Qué esperás que pase, escrito de modo que los datos puedan desmentirlo.",
                        prueba="por ejemplo: «la pérdida de validación baja».")
    laboratorio.control(notas, "Notas", "notas", tipo="experimento",
                        ayuda="Lo que quieras recordar de esta configuración. Se guarda con el experimento.")
    laboratorio.control(tasa, "Tasa de aprendizaje", "tasa_de_aprendizaje", tipo="experimento",
                        ayuda="Cuánto se mueven los pesos del modelo en cada paso.",
                        prueba="en ajustes cortos, comparé 5e-6, 5e-5 y 5e-4 y mirá las curvas de la estación 5.")
    laboratorio.control(lote, "Microlote (bloques por pasada)", "tamaño_de_lote", tipo="experimento",
                        ayuda="Cuántos bloques entran juntos al modelo en una pasada; lo limita la memoria.",
                        prueba="subilo y mirá cómo cambia el lote efectivo en la tabla.")
    laboratorio.control(acumulación, "Acumulación (microlotes por paso)", "acumulación_de_gradientes", tipo="experimento",
                        ayuda="Cuántos microlotes se suman antes de mover los pesos.",
                        prueba="duplicala y mirá cuántos pasos por época quedan.")
    laboratorio.control(unidad, "Medir la duración en", "unidad_de_duración", ancho="34em", tipo="experimento",
                        ayuda="Una época es una pasada completa por los bloques de entrenamiento; un paso, una actualización.",
                        prueba="elegí «épocas» y mirá cómo se recalculan los pasos totales.")
    laboratorio.control(duración, "Duración", "duración", tipo="experimento",
                        ayuda="Cuánto dura el ajuste, en la unidad elegida.",
                        prueba="empezá con valores chicos: alcanzan para ver el mecanismo.")
    laboratorio.control(evaluar, "Evaluar cada (pasos)", "evaluar_cada", tipo="experimento",
                        ayuda="Cada cuántos pasos se mide la pérdida en los textos reservados y se generan las muestras fijas.",
                        prueba="con 60 pasos y evaluar cada 20 vas a tener unos 4 puntos de validación.")
    laboratorio.control(guardar, "Guardar checkpoint cada (pasos)", "guardar_cada", tipo="experimento",
                        ayuda="Cada cuántos pasos se guarda un checkpoint: una copia de los pesos del modelo en ese momento.",
                        prueba="mirá en la estación 5 qué checkpoints quedaron.")
    laboratorio.control(semilla, "Semilla del ajuste", "semilla", tipo="experimento",
                        ayuda="Fija el orden en que se mezclan los bloques y el azar del ajuste.",
                        prueba="repetí un ajuste corto con otra semilla y compará las curvas.")
    laboratorio.control(comienzos, "Comienzos fijos (uno por línea)", "comienzos_de_muestras", tipo="experimento",
                        ayuda="Frases con las que el modelo escribe en cada evaluación, para ver el mismo ejercicio a lo largo del ajuste.",
                        prueba="sumá un comienzo que se parezca al autor y otro que no.")

    botón_consolidar = estación.agregar(laboratorio.botón("Consolidar configuración", "Crea la configuración inmutable; no entrena", "primary"))
    botón_restaurar = estación.agregar(laboratorio.botón("Volver a los valores iniciales", "Solo mueve los controles"))
    botón_anterior = estación.agregar(laboratorio.botón("Partir de la última configuración", "Pone en los controles lo que tenía la última configuración consolidada"))
    derivados = estación.agregar(widgets.HTML())
    resumen = estación.agregar(widgets.HTML(utils.html_aviso("Todavía no hay una configuración consolidada.")))
    inspección = estación.agregar(widgets.HTML())
    código = estación.agregar(widgets.HTML())

    estación.recordar_valores_iniciales(nombre, hipótesis, notas, tasa, lote, acumulación, unidad, duración, evaluar, guardar, semilla, comienzos)

    def parámetros():
        lista_de_comienzos = utils.líneas_con_texto(comienzos.value)
        return dict(nombre=nombre.value, hipótesis=hipótesis.value, notas=notas.value,
                    tasa_de_aprendizaje=tasa.value, tamaño_de_lote=lote.value, acumulación_de_gradientes=acumulación.value,
                    unidad_de_duración=unidad.value, duración=duración.value, evaluar_cada=evaluar.value,
                    guardar_cada=guardar.value, comienzos_de_muestras=lista_de_comienzos, semilla=semilla.value)

    def actualizar_derivados(_=None):
        bloques = len(preparación["bloques_de_entrenamiento"]["bloques"])
        cuentas = cuentas_de_la_época(bloques, lote.value, acumulación.value)
        por_época = cuentas["pasos_por_época"]
        total = duración.value if unidad.value == "pasos" else duración.value * por_época
        derivados.value = utils.html_tabla(
            ["lote efectivo", "pasos por época", "bloques que quedan afuera por época", "pasos totales", "microlotes totales",
             "bloques procesados"],
            [[str(lote.value * acumulación.value), str(por_época), str(cuentas["bloques_afuera"]), str(total),
              str(total * acumulación.value), str(total * lote.value * acumulación.value)]])
        llamada = laboratorio.texto_de_llamada("crear_configuración", dict(
            preparación=laboratorio.Código("preparación"), documentos=laboratorio.Código("documentos"),
            tokenizador=laboratorio.Código("tokenizador"), **parámetros()),
            "configuración")
        código.value = f"<pre style='font-family:{utils.FUENTE_CODIGO}'>{utils.escapar(llamada)}</pre>"

    def al_restaurar(_):
        estación.restaurar_valores_iniciales()
        actualizar_derivados()

    def al_partir_de_la_anterior(_):
        anterior = configuración_consolidada
        if anterior is None:
            resumen.value = utils.html_aviso("Todavía no consolidaste ninguna configuración.")
            return
        tasa.value = anterior.tasa_de_aprendizaje
        lote.value = anterior.tamaño_de_lote
        acumulación.value = anterior.acumulación_de_gradientes
        unidad.value = anterior.unidad_de_duración
        duración.value = anterior.duración_pedida
        evaluar.value = anterior.evaluar_cada
        guardar.value = anterior.guardar_cada
        semilla.value = anterior.semilla
        hipótesis.value = anterior.hipótesis
        notas.value = anterior.notas
        comienzos.value = "\n".join(anterior.comienzos_de_muestras)
        actualizar_derivados()

    def al_consolidar(_):
        global configuración_consolidada
        try:
            nueva = crear_configuración(preparación, documentos, tokenizador, **parámetros())
        except ValueError as error:
            resumen.value = utils.html_aviso(str(error), "error")
            return
        configuración_consolidada = nueva
        resumen.value = (utils.html_aviso(f"Configuración consolidada. Huella {nueva.huella()[:12]}. "
                                           f"Se guardará en {nueva.carpeta_del_experimento}.", "ok")
                         + utils.html_tabla(["campo", "valor"], utils.filas_de_diccionario(nueva.como_diccionario())))
        inspección.value = f"<pre style='font-family:{utils.FUENTE_CODIGO}'>{utils.escapar(json.dumps(nueva.como_diccionario(), ensure_ascii=False, indent=2))}</pre>"

    for control in (tasa, lote, acumulación, unidad, duración, evaluar, guardar, semilla, nombre, hipótesis, notas, comienzos):
        estación.observar(control, actualizar_derivados)
    estación.al_hacer_clic(botón_consolidar, al_consolidar)
    estación.al_hacer_clic(botón_restaurar, al_restaurar)
    estación.al_hacer_clic(botón_anterior, al_partir_de_la_anterior)
    actualizar_derivados()

    filas_de_controles = []
    for control_de_la_estación in (nombre, hipótesis, notas, tasa, lote, acumulación, unidad, duración, evaluar, guardar, semilla, comienzos):
        filas_de_controles.append(laboratorio.fila_de_control(control_de_la_estación))
    explorar = widgets.VBox(filas_de_controles + [derivados, widgets.HBox([botón_consolidar, botón_restaurar, botón_anterior]), resumen])
    return laboratorio.panel_de_tres_niveles(explorar, widgets.VBox([inspección]), widgets.VBox([código]))

### Panel: configurar y consolidar

Los números de la tabla de abajo (lote efectivo, pasos por época, pasos totales) se actualizan al mover los controles, sin tocar el modelo. Solo el botón **Consolidar configuración** crea la configuración que va a usar la estación 5, y tampoco entrena.

1. Mové **Microlote** y **Acumulación** y mirá cómo cambian el lote efectivo y los pasos por época.
2. Cambiá la unidad a **épocas**: los pasos totales se recalculan.
3. Poné **Evaluar cada** en 1000 con 60 de duración y apretá **Consolidar**: el panel responde con un error en español, no con un fallo.
4. Consolidá con un nombre nuevo. La estación 5 usa **esa** configuración.

Para volver atrás: **Volver a los valores iniciales** o **Partir de la última configuración**.

In [ ]:
display(armar_estación_4())

### Qué significa cada número

Un ajuste repite una rueda: el modelo mira unos bloques, predice, mide su error (la pérdida) y se corrige un poco. Un ejemplo con los valores iniciales (346 bloques de entrenamiento, microlote 4 y acumulación 2):

- Cada pasada lleva 4 bloques al modelo: es un **microlote**. Con 346 bloques hay 86 microlotes completos (sobran 2 bloques).
- Cada microlote deja, para cada peso del modelo, una indicación de hacia dónde conviene moverlo para que la pérdida baje un poco: el *gradiente*. Con una **acumulación** de 2, esas indicaciones de dos microlotes se suman y recién entonces el modelo se mueve. Ese movimiento es un **paso**, y el **lote efectivo** es de 4 × 2 = 8 bloques.
- Una **época** es una pasada por todos los bloques de entrenamiento. Acá tiene 86 ÷ 2 = 43 pasos.

| Parámetro | Qué controla | Qué podés examinar | Qué no significa |
|---|---|---|---|
| **Microlote** (`tamaño_de_lote`) | Cuántos bloques entran juntos al modelo en una pasada; lo limita la memoria | El lote efectivo y los pasos por época de la tabla | No es cuánto aprende el modelo por paso |
| **Acumulación** (`acumulación_de_gradientes`) | Cuántos microlotes se suman antes de mover los pesos | Cómo bajan los pasos por época cuando la subís | No agrega datos: usa los mismos bloques |
| **Tasa de aprendizaje** (`tasa_de_aprendizaje`) | Cuánto se mueven los pesos en cada paso | La curva de la pérdida en la estación 5 | No es la velocidad de ejecución: una tasa más alta no hace que el ajuste termine antes |
| **Evaluar cada** (`evaluar_cada`) | Cada cuántos pasos se mide la pérdida en los textos reservados y se generan las muestras fijas | Cuántos puntos tiene la curva de validación | No cambia lo que aprende el modelo |
| **Guardar cada** (`guardar_cada`) | Cada cuántos pasos se guarda un *checkpoint* (una copia de los pesos) | Qué checkpoints quedan para comparar en la estación 6 | No es un respaldo del corpus |

Con una tasa muy alta la pérdida puede saltar o subir; con una muy baja puede casi no cambiar en pocos pasos. Es lo que vas a poder comprobar en la estación 5.

### Ahora probá otra cosa

Duplicá el microlote y partí la acumulación a la mitad: ¿qué queda igual (el lote efectivo) y qué cambia? Si la memoria no alcanza, esa es la razón de ser de la acumulación: lograr un lote efectivo grande sin pedir más memoria.

<details>
<summary>Profundización: la configuración, las épocas y lo que se valida</summary>

**La configuración es un objeto congelado.** `Configuración` agrupa todas las decisiones (identificación, modelo y datos, optimización, duración, seguimiento, azar y entorno). Con `frozen=True`, cambiar un campo después de crearla da un error: la configuración de una corrida no puede cambiar mientras corre. Para probar una variante se usa `dataclasses.replace(configuración, tasa_de_aprendizaje=1e-3)`, que devuelve una configuración nueva. La precisión es siempre `float32`, y las cinco huellas del final registran con qué corpus, partición, tokenizador y bloques se creó.

**Qué es una época cuando hay acumulación.** Un paso nunca mezcla microlotes de dos épocas: una época tiene `(bloques // microlote) // acumulación` pasos (`//` es la división que descarta el resto), y los microlotes que no completan un paso se dejan afuera de esa época. En la época siguiente el orden se vuelve a mezclar, así que no quedan siempre los mismos bloques afuera. Con 346 bloques, microlote 4 y acumulación 4, por ejemplo: 86 microlotes completos, 21 pasos por época y 10 bloques afuera. `cuentas_de_la_época` hace esa cuenta en un solo lugar y la usan la configuración, el panel y el bucle de entrenamiento.

**Qué revisa `crear_configuración`** antes de construir nada:

- un microlote más grande que el corpus, una evaluación que nunca ocurriría, un calentamiento más largo que todo el ajuste;
- datos desactualizados: si el corpus o el tokenizador ya no coinciden con los bloques preparados, se niega y dice cuál cambió;
- un nombre que ya existe: nunca se pisa un experimento guardado.

El *calentamiento* son los pasos iniciales en que la tasa de aprendizaje sube de 0 hasta su valor; el panel lo deja en 5 pasos.
</details>

### Vía con código (alternativa al panel)

Armar una configuración, mirar sus números derivados y consolidarla. Los parámetros están en MAYÚSCULAS.

| Parámetro | Qué significa | Qué cambia si lo modificás |
|---|---|---|
| `TASA_DE_APRENDIZAJE` | Cuánto se mueven los pesos por paso; `5e-5` es un valor habitual para ajustar GPT-2 | Con una tasa muy alta la pérdida puede saltar; con una muy baja, casi no cambiar en pocos pasos |
| `TAMAÑO_DE_LOTE` | Bloques por microlote | Más grande: más memoria por pasada |
| `ACUMULACIÓN` | Microlotes por paso | Más acumulación: menos pasos por época |
| `DURACIÓN` y `UNIDAD` | Cuánto dura el ajuste | En épocas, el total depende de cuántos bloques hay |

Con 60 pasos de lote efectivo 8, el ajuste ve unos 480 bloques: algo más de una pasada por el corpus. Es un ensayo corto: sirve para observar el mecanismo, no para obtener un modelo bueno.

### ✎ Antes de ejecutar

Con lote efectivo 8 y 346 bloques de entrenamiento, ¿cuántos pasos tiene una época? ¿Sobra algún bloque, y qué pasa con él?

In [ ]:
NOMBRE_DEL_EXPERIMENTO = "ensayo_quiroga_" + time.strftime("%Y%m%d_%H%M%S")  # con fecha y hora, para no pisar otro
HIPÓTESIS = "Después del ajuste, las continuaciones van a incorporar vocabulario y giros de Quiroga."
TASA_DE_APRENDIZAJE = 5e-5
TAMAÑO_DE_LOTE = 4
ACUMULACIÓN = 2
UNIDAD = "pasos"
DURACIÓN = 60
EVALUAR_CADA = 20
GUARDAR_CADA = 30
SEMILLA_DEL_AJUSTE = 42

configuración = crear_configuración(
    preparación=preparación,
    documentos=documentos,
    tokenizador=tokenizador,
    nombre=NOMBRE_DEL_EXPERIMENTO,
    hipótesis=HIPÓTESIS,
    tasa_de_aprendizaje=TASA_DE_APRENDIZAJE,
    tamaño_de_lote=TAMAÑO_DE_LOTE,
    acumulación_de_gradientes=ACUMULACIÓN,
    unidad_de_duración=UNIDAD,
    duración=DURACIÓN,
    evaluar_cada=EVALUAR_CADA,
    guardar_cada=GUARDAR_CADA,
    semilla=SEMILLA_DEL_AJUSTE,
)

print(f"Lote efectivo: {configuración.lote_efectivo} bloques ({configuración.tamaño_de_lote} por microlote × {configuración.acumulación_de_gradientes})")
print(f"Pasos por época: {configuración.pasos_por_época} · pasos totales: {configuración.pasos_totales} · microlotes totales: {configuración.microlotes_totales}")
print(f"Bloques que quedan afuera en cada época: {configuración.bloques_que_se_dejan_afuera_por_época} de {configuración.bloques_de_entrenamiento}")
print(f"Huella de la configuración: {configuración.huella()[:16]}…")
print(f"Se guardaría en: {configuración.carpeta_del_experimento}")

**Mirá el resultado.** Fijate en la diferencia entre **microlotes** (pasadas por el modelo) y **pasos** (actualizaciones): un paso son `ACUMULACIÓN` microlotes. Los pasos son lo que aparece en el eje horizontal de las curvas. Los pasos por época salen de dividir los bloques por el microlote y por la acumulación, descartando el resto de cada división; lo que no completa un paso queda afuera de esa época. La huella identifica la configuración, no dice si es buena.

Probá en una celda propia `configuración.tasa_de_aprendizaje = 1e-3`: tiene que dar un error (`FrozenInstanceError`).

### Preparación del ajuste y ensayo de tiempo

**Preparación · piezas del ajuste.** La celda siguiente define lo que usa el bucle de entrenamiento: cargar una copia fresca del modelo (el modelo base que usás para comparar **nunca se entrena**), decidir qué bloques van en cada microlote, medir la pérdida sobre un conjunto fijo, crear el optimizador y liberar memoria. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def cargar_modelo_para_entrenar(configuración, desde=None):
    """Carga una copia del modelo (la base, o un checkpoint si se indica `desde`) lista para entrenar."""
    origen = desde or configuración.modelo_base
    revisión = None if desde else configuración.revisión_del_modelo
    modelo = AutoModelForCausalLM.from_pretrained(origen, revision=revisión, dtype=torch.float32)
    modelo = modelo.to(torch.device(configuración.dispositivo))
    modelo.train()  # modo de entrenamiento: el dropout está activo
    return modelo


def orden_de_los_microlotes(cantidad_de_bloques, tamaño_de_lote, época, semilla):
    """Lista de microlotes (cada uno, una lista de números de bloque) para una época.

    Se mezclan los bloques con una semilla que depende de la época. Si sobran bloques que no
    completan un microlote, se dejan afuera de esa época. Para repartirlos en pasos, ver `microlotes_del_paso`.
    """
    generador = torch.Generator()
    generador.manual_seed(semilla * 1000 + época)
    mezclados = torch.randperm(cantidad_de_bloques, generator=generador).tolist()
    microlotes = []
    for inicio in range(0, cantidad_de_bloques - tamaño_de_lote + 1, tamaño_de_lote):
        microlotes.append(mezclados[inicio:inicio + tamaño_de_lote])
    return microlotes


def microlotes_del_paso(paso, cantidad_de_bloques, tamaño_de_lote, acumulación_de_gradientes, semilla):
    """Los microlotes que alimentan el paso número `paso` (el primero es el 0); cada uno es una lista de números de bloque.

    Un paso nunca mezcla microlotes de dos épocas. Los microlotes que sobran al final de cada época
    (no completan un paso) se dejan afuera; el orden se vuelve a mezclar en la época siguiente.
    """
    cuentas = cuentas_de_la_época(cantidad_de_bloques, tamaño_de_lote, acumulación_de_gradientes)
    if cuentas["pasos_por_época"] < 1:
        raise ValueError("Con ese microlote y esa acumulación no alcanza para un paso completo por época.")
    época = paso // cuentas["pasos_por_época"]
    paso_dentro_de_la_época = paso % cuentas["pasos_por_época"]
    orden = orden_de_los_microlotes(cantidad_de_bloques, tamaño_de_lote, época, semilla)
    desde = paso_dentro_de_la_época * acumulación_de_gradientes
    return orden[desde:desde + acumulación_de_gradientes]


def tensor_del_microlote(bloques, números_de_bloque, dispositivo):
    """Arma el tensor (microlote × largo de bloque) con los bloques elegidos."""
    filas = []
    for número in números_de_bloque:
        filas.append(bloques[número])
    return torch.tensor(filas, device=dispositivo)


def pérdida_en_bloques(modelo, bloques, tamaño_de_lote=8):
    """Pérdida media por token (en nats) y perplejidad del modelo sobre `bloques`, sin entrenar.

    Cada bloque aporta (largo − 1) predicciones: la primera posición no es objetivo de ninguna.
    Se promedia sobre todas las predicciones, así que los bloques pesan igual porque tienen el mismo largo.
    La perplejidad es exp(pérdida). Su mínimo teórico es 1.
    """
    exigir_dispositivo_libre()
    mayor_identificador = 0
    for bloque in bloques:
        mayor_identificador = max(mayor_identificador, max(bloque))
    if mayor_identificador >= modelo.config.vocab_size:
        raise ValueError(
            f"Los bloques usan el token {mayor_identificador}, pero el modelo solo conoce {modelo.config.vocab_size}: "
            "fueron armados con otro tokenizador. No se calcula la pérdida."
        )
    estaba_entrenando = modelo.training
    modelo.eval()
    suma_de_pérdidas = 0.0
    predicciones = 0
    with torch.no_grad():
        for inicio in range(0, len(bloques), tamaño_de_lote):
            lote = tensor_del_microlote(bloques, range(inicio, min(inicio + tamaño_de_lote, len(bloques))), modelo.device)
            pérdida_del_lote = modelo(input_ids=lote, labels=lote).loss.item()
            cantidad = lote.shape[0] * (lote.shape[1] - 1)
            suma_de_pérdidas += pérdida_del_lote * cantidad
            predicciones += cantidad
    if estaba_entrenando:
        modelo.train()
    pérdida = suma_de_pérdidas / predicciones
    return {"pérdida": pérdida, "perplejidad": math.exp(pérdida), "predicciones": predicciones, "bloques": len(bloques)}


def liberar_memoria_del_dispositivo():
    """Pide a Python y al dispositivo que devuelvan la memoria de lo que ya no se usa."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


def crear_optimizador_y_programador(modelo, configuración):
    """AdamW con calentamiento lineal y descenso lineal de la tasa de aprendizaje hasta 0 al final."""
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=configuración.tasa_de_aprendizaje, weight_decay=0.0)
    programador = transformers.get_linear_schedule_with_warmup(
        optimizador,
        num_warmup_steps=configuración.pasos_de_calentamiento,
        num_training_steps=configuración.pasos_totales,
    )
    return optimizador, programador


def ensayo_de_tiempo(configuración, preparación, microlotes_de_prueba=3):
    """Mide cuánto tarda un microlote completo (adelante, atrás y actualización) y estima el total.

    Usa una copia desechable del modelo; el modelo base y los experimentos guardados no se tocan.
    La estimación es APROXIMADA: sale de una medición corta en esta máquina, con la carga de este
    momento, y no incluye evaluaciones ni guardados.
    """
    exigir_dispositivo_libre()
    modelo = cargar_modelo_para_entrenar(configuración)
    optimizador, _ = crear_optimizador_y_programador(modelo, configuración)
    bloques = preparación["bloques_de_entrenamiento"]["bloques"]
    dispositivo = torch.device(configuración.dispositivo)
    tiempos = []
    for número in range(microlotes_de_prueba + 1):
        inicio = time.perf_counter()
        lote = tensor_del_microlote(bloques, range(configuración.tamaño_de_lote), dispositivo)
        pérdida = modelo(input_ids=lote, labels=lote).loss
        pérdida.backward()
        optimizador.step()
        optimizador.zero_grad()
        if dispositivo.type == "mps":
            torch.mps.synchronize()
        if dispositivo.type == "cuda":
            torch.cuda.synchronize()
        if número > 0:  # el primero incluye la preparación y no cuenta
            tiempos.append(time.perf_counter() - inicio)
    del modelo, optimizador
    liberar_memoria_del_dispositivo()
    segundos_por_microlote = sum(tiempos) / len(tiempos)
    return {
        "segundos_por_microlote": segundos_por_microlote,
        "segundos_por_paso": segundos_por_microlote * configuración.acumulación_de_gradientes,
        "estimación_total_segundos": segundos_por_microlote * configuración.microlotes_totales,
        "mediciones": len(tiempos),
    }

**Ensayo de tiempo (opcional).** Mide cuánto tarda tu máquina y estima el total de la configuración de arriba, con cuatro microlotes y una copia desechable del modelo. **No corre con «Ejecutar todo»**: poné `PERMITIR_ENSAYO_DE_TIEMPO = True`. La estimación es aproximada: sale de una medición corta, con la carga actual de la máquina, y no incluye evaluaciones ni guardados.

In [ ]:
display(pd.DataFrame([diagnóstico_de_hardware()]).T.rename(columns={0: "valor"}))

if PERMITIR_ENSAYO_DE_TIEMPO:
    ensayo = ensayo_de_tiempo(configuración, preparación, microlotes_de_prueba=3)
    print(f"≈ {ensayo['segundos_por_microlote']:.2f} s por microlote · ≈ {ensayo['segundos_por_paso']:.2f} s por paso (medido con {ensayo['mediciones']} microlotes)")
    print(f"Estimación aproximada del ajuste completo: ≈ {ensayo['estimación_total_segundos'] / 60:.1f} minutos, sin evaluaciones ni guardados.")
else:
    print("Ensayo de tiempo omitido. Poné PERMITIR_ENSAYO_DE_TIEMPO = True en los parámetros del cuaderno para medirlo.")

**Mirá el resultado.** Primero aparece el diagnóstico de hardware (solo informa) y, si habilitaste el ensayo, el tiempo medido por microlote y por paso. Si en CPU el total supera lo que querés esperar, bajá la duración o el largo de bloque; otra carga de la máquina, otro largo de bloque u otro dispositivo cambian la medición.

---
## Estación 5. Observá el ajuste

**Desafío.** Lanzá el ajuste con la configuración que consolidaste y mirá dos cosas a la vez: cómo cambia la pérdida y cómo cambia lo que escribe el modelo en el camino.

**Advertencia:** el panel de abajo **entrena de verdad** (minutos; los checkpoints ocupan unos 2 GB en disco). Antes de iniciar, consolidá la configuración en la estación 4. Si el botón responde con un aviso rojo, leé el motivo: se niega por una razón. Mientras el ajuste corre no podés generar ni comparar con otro modelo (la GPU no tolera dos usos a la vez): esperá o apretá **Detener**.

**Preparación · estación 5 (guardado).** La celda siguiente define cómo se guardan los ingredientes del experimento, el azar y los checkpoints. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def guardar_ingredientes_del_experimento(configuración, preparación, documentos):
    """Escribe en la carpeta del experimento todo lo que hace falta para entender y repetir la corrida.

    Guarda: la configuración y su huella, el manifiesto del corpus (con huellas de originales y de
    textos procesados), los textos usados, la partición y el entorno. Los textos originales se
    guardan aparte de los procesados para conservarlos.
    """
    carpeta = configuración.carpeta_del_experimento
    carpeta.mkdir(parents=True, exist_ok=False)
    datos = configuración.como_diccionario()
    datos["huella_de_la_configuración"] = configuración.huella()
    datos["creada"] = laboratorio.ahora()
    laboratorio.escribir_json(carpeta / "configuración.json", datos)

    manifiesto = {"huella": configuración.huella_del_corpus, "documentos": []}
    for documento in documentos:
        ficha = {}
        for campo in ("id", "título", "autoría", "fecha", "género", "fuente", "condiciones_de_uso", "grupo", "incluido", "advertencias"):
            ficha[campo] = documento[campo]
        ficha["huella_del_original"] = laboratorio.huella_de_texto(documento["texto_original"])
        ficha["huella_del_texto"] = laboratorio.huella_de_texto(documento["texto"])
        ficha["transformaciones"] = documento["transformaciones"]
        manifiesto["documentos"].append(ficha)
        if documento["incluido"]:
            (carpeta / "corpus").mkdir(exist_ok=True)
            (carpeta / "corpus" / "originales").mkdir(exist_ok=True)
            (carpeta / "corpus" / f"{documento['id']}.txt").write_text(documento["texto"], encoding="utf-8")
            (carpeta / "corpus" / "originales" / f"{documento['id']}.txt").write_text(documento["texto_original"], encoding="utf-8")
    laboratorio.escribir_json(carpeta / "manifiesto_del_corpus.json", manifiesto)

    ids_de_entrenamiento = []
    for documento_de_entrenamiento in preparación["partición"]["entrenamiento"]:
        ids_de_entrenamiento.append(documento_de_entrenamiento["id"])
    ids_de_validación = []
    for documento_de_validación in preparación["partición"]["validación"]:
        ids_de_validación.append(documento_de_validación["id"])
    laboratorio.escribir_json(carpeta / "partición.json", {
        "modo": preparación["partición"]["modo"],
        "ids_de_entrenamiento": ids_de_entrenamiento,
        "ids_de_validación": ids_de_validación,
        "advertencias": preparación["partición"]["advertencias"],
        "verificación_de_fuga": preparación["verificación"],
        "identidades": preparación["identidades"],
    })
    laboratorio.escribir_json(carpeta / "entorno.json", laboratorio.información_del_entorno())
    return carpeta


import contextlib


def guardar_estados_de_azar():
    """Copia el generador de números al azar de cada dispositivo disponible y devuelve {dispositivo: estado}.

    Cada dispositivo tiene el suyo. El dropout del modelo sortea en el dispositivo donde está el modelo,
    así que para retomar un ajuste hay que guardar el de ese dispositivo, no solo el de la CPU.
    """
    estados = {"cpu": torch.get_rng_state()}
    if torch.cuda.is_available():
        estados["cuda"] = torch.cuda.get_rng_state_all()
    if torch.backends.mps.is_available():
        estados["mps"] = torch.mps.get_rng_state()
    return estados


def restaurar_estados_de_azar(estados):
    """Devuelve cada generador a lo guardado. Responde con la lista de dispositivos que sí se restauraron."""
    restaurados = []
    if "cpu" in estados:
        torch.set_rng_state(estados["cpu"])
        restaurados.append("cpu")
    if "cuda" in estados and torch.cuda.is_available() and len(estados["cuda"]) == torch.cuda.device_count():
        torch.cuda.set_rng_state_all(estados["cuda"])
        restaurados.append("cuda")
    if "mps" in estados and torch.backends.mps.is_available():
        torch.mps.set_rng_state(estados["mps"])
        restaurados.append("mps")
    return restaurados


@contextlib.contextmanager
def conservar_el_azar():
    """Guarda el azar de todos los dispositivos al entrar y lo restaura al salir.

    Lo usamos al generar las muestras periódicas: `generar` fija su propia semilla con `torch.manual_seed`,
    que reinicia el azar de la CPU y de los aceleradores. Sin esta protección, retomar un ajuste
    dependería de cuántas muestras se hicieron antes.
    """
    estados = guardar_estados_de_azar()
    try:
        yield
    finally:
        restaurar_estados_de_azar(estados)


def identidad_de_los_datos_de(configuración):
    """Las huellas de datos que una configuración registra; un checkpoint las copia para poder verificarlas al reanudar."""
    return {
        "huella_del_corpus": configuración.huella_del_corpus,
        "huella_de_la_preparación": configuración.huella_de_la_preparación,
        "huella_de_la_partición": configuración.huella_de_la_partición,
        "huella_del_tokenizador": configuración.huella_del_tokenizador,
        "huella_de_los_bloques_de_entrenamiento": configuración.huella_de_los_bloques_de_entrenamiento,
        "huella_de_los_bloques_de_validación": configuración.huella_de_los_bloques_de_validación,
    }


def guardar_checkpoint(modelo, tokenizador, optimizador, programador, carpeta, paso, con_estado_de_reanudación,
                       identidad_de_los_datos):
    """Guarda pesos y tokenizador; si `con_estado_de_reanudación`, también optimizador, programador y azar.

    Con solo los pesos se puede GENERAR. Para REANUDAR el ajuste hace falta además el estado de reanudación
    (optimizador, programador y el generador de azar de cada dispositivo). Siempre se anota con qué datos se ajustó.
    """
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    modelo.save_pretrained(carpeta)
    tokenizador.save_pretrained(carpeta)
    estados_de_azar = {}
    if con_estado_de_reanudación:
        estados_de_azar = guardar_estados_de_azar()
        torch.save({
            "paso": paso,
            "optimizador": optimizador.state_dict(),
            "programador": programador.state_dict(),
            "azar": estados_de_azar,
        }, carpeta / "estado_de_entrenamiento.pt")
    laboratorio.escribir_json(carpeta / "checkpoint.json", {
        "paso": paso,
        "contiene_pesos": True,
        "contiene_estado_de_optimizador_y_programador": con_estado_de_reanudación,
        "se_puede_reanudar_el_ajuste": con_estado_de_reanudación,
        "estados_de_azar_guardados": list(estados_de_azar),
        "dispositivo_del_modelo": str(modelo.device),
        "identidad_de_los_datos": identidad_de_los_datos,
        "guardado": laboratorio.ahora(),
    })
    return carpeta


def muestras_fijas(modelo, tokenizador, configuración, paso):
    """Genera una muestra por cada comienzo fijo, con la misma configuración de muestreo y la misma semilla."""
    modelo.eval()
    muestras = []
    with conservar_el_azar():
        for comienzo in configuración.comienzos_de_muestras:
            resultado = generar(modelo, tokenizador, comienzo, n_muestras=1, max_new_tokens=40, modo="muestreo",
                                temperatura=0.8, top_k=50, top_p=0.95, semilla=configuración.semilla,
                                procedencia=f"{configuración.nombre}@paso{paso}")
            muestras.append({"paso": paso, "comienzo": comienzo, "continuación": resultado[0]["continuación"],
                             "configuración": resultado[0]["configuración"]})
    modelo.train()
    return muestras

**Preparación · estación 5 (bucle y panel).** La celda siguiente define la comprobación de datos, el bucle `entrenar`, la lectura del historial y el panel. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def verificar_datos_del_ajuste(configuración, preparación, documentos, tokenizador, carpeta_del_checkpoint=None):
    """Levanta un ValueError si los datos que se entregan no son los de la configuración (ni los del checkpoint).

    Comprueba, en este orden: que la preparación siga siendo la del corpus y el tokenizador actuales, que sea la
    que usó la configuración y, al reanudar, que sea la que usó el checkpoint.
    """
    exigir_datos_vigentes(preparación, documentos, tokenizador)
    if configuración.huella_de_la_preparación != preparación["identidades"]["preparación"]:
        raise ValueError(
            "Los datos preparados no son los de esta configuración: cambió el corpus, la partición, el tokenizador "
            "o los bloques desde que se la consolidó. Volvé a consolidar la configuración (estación 4)."
        )
    if carpeta_del_checkpoint is not None:
        ficha = laboratorio.leer_json(Path(carpeta_del_checkpoint) / "checkpoint.json")
        del_checkpoint = ficha.get("identidad_de_los_datos")
        if not del_checkpoint:
            raise ValueError("Este checkpoint no registra con qué datos se ajustó, así que no se puede verificar. "
                             "Sirve para generar, pero no para reanudar.")
        esperada = identidad_de_los_datos_de(configuración)
        distintas = []
        for clave, valor in esperada.items():
            if del_checkpoint.get(clave) != valor:
                distintas.append(clave)
        if distintas:
            raise ValueError("El checkpoint se hizo con otros datos (" + ", ".join(distintas) + "). No se reanuda.")


def entrenar(configuración, preparación, tokenizador, documentos, parar=None, al_paso=None,
             al_evaluar=None, al_muestra=None, al_guardar=None, reanudar_desde=None):
    """Ajusta el modelo según la configuración y devuelve un diccionario con el resultado.

    - parar: un threading.Event; si se activa, termina el paso en curso, guarda con estado de reanudación y sale.
    - al_paso, al_evaluar, al_muestra, al_guardar: funciones que se llaman con cada registro (para mostrar progreso).
    - reanudar_desde: carpeta de un checkpoint con estado de reanudación, para continuar un ajuste detenido.
    Antes de empezar comprueba que los datos sean los de la configuración (y los del checkpoint, si se reanuda).
    Todo lo que ocurre queda escrito en la carpeta del experimento (métricas.jsonl, muestras.jsonl, checkpoints).
    """
    verificar_datos_del_ajuste(configuración, preparación, documentos, tokenizador, reanudar_desde)
    carpeta = configuración.carpeta_del_experimento
    if reanudar_desde is None:
        guardar_ingredientes_del_experimento(configuración, preparación, documentos)
    métricas = laboratorio.RegistroJSONL(carpeta / "métricas.jsonl")
    registro_de_muestras = laboratorio.RegistroJSONL(carpeta / "muestras.jsonl")
    identidad_de_los_datos = identidad_de_los_datos_de(configuración)
    inicio_total = time.perf_counter()
    estado_final = "completado"
    avisos = []

    torch.manual_seed(configuración.semilla)
    modelo = cargar_modelo_para_entrenar(configuración, desde=reanudar_desde)
    optimizador, programador = crear_optimizador_y_programador(modelo, configuración)
    paso = 0
    if reanudar_desde is not None:
        estado = torch.load(Path(reanudar_desde) / "estado_de_entrenamiento.pt", weights_only=True)
        optimizador.load_state_dict(estado["optimizador"])
        programador.load_state_dict(estado["programador"])
        restaurados = restaurar_estados_de_azar(estado["azar"])
        tipo_de_dispositivo = torch.device(configuración.dispositivo).type
        if tipo_de_dispositivo not in restaurados:
            avisos.append(f"No se restauró el azar de '{tipo_de_dispositivo}' (el checkpoint guardó {list(estado['azar'])}): "
                          "el dropout de esta continuación no retoma el sorteo del ajuste original.")
        paso = estado["paso"]

    bloques = preparación["bloques_de_entrenamiento"]["bloques"]
    bloques_de_validación = preparación["bloques_de_validación"]["bloques"]
    dispositivo = torch.device(configuración.dispositivo)

    def evaluar_y_mostrar(paso_actual):
        medida = pérdida_en_bloques(modelo, bloques_de_validación)
        registro = métricas.agregar({"tipo": "validación", "paso": paso_actual, "pérdida": medida["pérdida"],
                                      "perplejidad": medida["perplejidad"], "segundos": time.perf_counter() - inicio_total})
        nuevas = muestras_fijas(modelo, tokenizador, configuración, paso_actual)
        for muestra in nuevas:
            registro_de_muestras.agregar(muestra)
        if al_evaluar:
            al_evaluar(registro)
        if al_muestra:
            al_muestra(nuevas)

    def guardar(paso_actual, con_estado_de_reanudación):
        destino = carpeta / "checkpoints" / f"paso_{paso_actual:05d}"
        guardar_checkpoint(modelo, tokenizador, optimizador, programador, destino, paso_actual,
                           con_estado_de_reanudación, identidad_de_los_datos)
        if al_guardar:
            al_guardar({"paso": paso_actual, "carpeta": str(destino), "con_estado_de_reanudación": con_estado_de_reanudación})
        return destino

    try:
        if reanudar_desde is None:
            evaluar_y_mostrar(0)  # punto de partida: cómo está el modelo antes de ajustar
        while paso < configuración.pasos_totales:
            if parar is not None and parar.is_set():
                estado_final = "detenido"
                break
            inicio_del_paso = time.perf_counter()
            pérdidas_del_paso = []
            microlotes = microlotes_del_paso(paso, len(bloques), configuración.tamaño_de_lote,
                                              configuración.acumulación_de_gradientes, configuración.semilla)
            for números in microlotes:
                lote = tensor_del_microlote(bloques, números, dispositivo)
                pérdida = modelo(input_ids=lote, labels=lote).loss
                if not torch.isfinite(pérdida):
                    raise RuntimeError(f"La pérdida dejó de ser un número ({pérdida.item()}) en el paso {paso + 1}. "
                                       "Probá bajar la tasa de aprendizaje.")
                (pérdida / configuración.acumulación_de_gradientes).backward()
                pérdidas_del_paso.append(pérdida.item())
            norma = torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
            optimizador.step()
            programador.step()
            optimizador.zero_grad(set_to_none=True)
            paso += 1

            registro = métricas.agregar({
                "tipo": "entrenamiento", "paso": paso, "pérdida": sum(pérdidas_del_paso) / len(pérdidas_del_paso),
                "época": (paso - 1) // configuración.pasos_por_época + 1,
                "tasa_de_aprendizaje": programador.get_last_lr()[0], "norma_del_gradiente": float(norma),
                "segundos_del_paso": time.perf_counter() - inicio_del_paso, "segundos": time.perf_counter() - inicio_total,
            })
            if al_paso:
                al_paso(registro)
            if paso % configuración.evaluar_cada == 0 and paso < configuración.pasos_totales:
                evaluar_y_mostrar(paso)
            if paso % configuración.guardar_cada == 0 and paso < configuración.pasos_totales:
                guardar(paso, con_estado_de_reanudación=False)
    except KeyboardInterrupt:
        estado_final = "detenido"  # el kernel se interrumpió: los pesos son los del último paso completo

    if estado_final == "completado":
        evaluar_y_mostrar(paso)
    checkpoint_final = guardar(paso, con_estado_de_reanudación=True)
    resultado = {"estado": estado_final, "paso_final": paso, "segundos": time.perf_counter() - inicio_total,
                 "checkpoint_final": str(checkpoint_final), "terminó": laboratorio.ahora(), "avisos": avisos}
    laboratorio.escribir_json(carpeta / "resultado.json", resultado)
    del modelo, optimizador, programador
    liberar_memoria_del_dispositivo()
    return resultado


def columna(registros, clave):
    """La lista de valores de un campo en una lista de registros (por ejemplo, todos los pasos)."""
    valores = []
    for registro in registros:
        valores.append(registro[clave])
    return valores


def leer_historial(carpeta):
    """Reconstruye las métricas y las muestras de un experimento desde sus archivos (sin usar widgets)."""
    carpeta = Path(carpeta)
    métricas = laboratorio.RegistroJSONL(carpeta / "métricas.jsonl").leer()
    muestras = laboratorio.RegistroJSONL(carpeta / "muestras.jsonl").leer()
    entrenamiento = []
    validación = []
    for medida in métricas:
        if medida["tipo"] == "entrenamiento":
            entrenamiento.append(medida)
        else:
            validación.append(medida)
    return {"entrenamiento": entrenamiento, "validación": validación, "muestras": muestras}


def carpeta_del_experimento_activo(nombre=None):
    """Carpeta del experimento con el que trabajan las vías con código, o None si todavía no hay ninguno.

    Con `nombre` busca esa carpeta. Sin nombre toma la más reciente que tenga resultado.json,
    es decir, un ajuste que terminó o se detuvo.
    """
    for carpeta in laboratorio.listar_experimentos(RAÍZ_DE_EXPERIMENTOS):
        if nombre is not None and carpeta.name == nombre:
            return carpeta
        if nombre is None and (carpeta / "resultado.json").exists():
            return carpeta
    return None


def iniciar_entrenamiento_en_segundo_plano(configuración, preparación, tokenizador, documentos, **funciones_de_aviso):
    """Lanza `entrenar` en un hilo y devuelve enseguida. Se niega si ya hay un ajuste en curso."""
    hilo_anterior = ejecución_actual["hilo"]
    if hilo_anterior is not None and hilo_anterior.is_alive():
        raise RuntimeError("Ya hay un ajuste en curso. Esperá a que termine o detenelo antes de iniciar otro.")
    ejecución_actual["parar"] = threading.Event()
    ejecución_actual.update(estado="ejecutando", resultado=None, error=None)

    def trabajo():
        try:
            resultado = entrenar(configuración, preparación, tokenizador, documentos,
                                 parar=ejecución_actual["parar"], **funciones_de_aviso)
            ejecución_actual["resultado"] = resultado
            ejecución_actual["estado"] = resultado["estado"]
        except Exception as error:  # cualquier falla queda registrada y visible, no se pierde en el hilo
            ejecución_actual["error"] = f"{type(error).__name__}: {error}"
            ejecución_actual["estado"] = "falló"

    ejecución_actual["hilo"] = threading.Thread(target=trabajo, name="ajuste", daemon=True)
    ejecución_actual["hilo"].start()
    return ejecución_actual


def esperar_entrenamiento(segundos=None):
    """Espera a que termine el hilo (útil en scripts y pruebas). Devuelve el estado final."""
    if ejecución_actual["hilo"] is not None:
        ejecución_actual["hilo"].join(timeout=segundos)
    return ejecución_actual["estado"]


def configuración_vigente(configuración_a_revisar):
    """Devuelve (True, "") si la configuración sigue valiendo, o (False, motivo) si cambió algo desde que se consolidó."""
    if configuración_a_revisar is None:
        return False, "Todavía no hay una configuración consolidada (estación 4)."
    try:
        verificar_datos_del_ajuste(configuración_a_revisar, preparación, documentos, tokenizador)
    except ValueError as error:
        return False, str(error)
    return True, ""


def armar_estación_5():
    """Panel de entrenamiento. Usa `configuración_consolidada`; el ajuste corre en un hilo aparte."""
    estación = laboratorio.nueva_estación("5-entrenamiento")

    botón_iniciar = estación.agregar(laboratorio.botón("Iniciar ajuste", "Entrena de verdad con la configuración consolidada", "danger"))
    botón_detener = estación.agregar(laboratorio.botón("Detener", "Termina el paso en curso, guarda y sale", "warning"))
    botón_detener.disabled = True
    suavizado = estación.agregar(widgets.IntSlider(value=5, min=1, max=30, continuous_update=False))
    laboratorio.control(suavizado, "Suavizado (media móvil, 1 = sin suavizar)", "ventana", tipo="vista",
                        ayuda="Alisa la curva para ver la tendencia. Solo cambia el dibujo, no los datos ni el ajuste.",
                        prueba="movelo mientras corre: el trazo fino queda igual.")
    progreso = estación.agregar(widgets.IntProgress(value=0, min=0, max=1, description="Pasos"))
    estado = estación.agregar(widgets.HTML(utils.html_estado("pendiente")))
    números = estación.agregar(widgets.HTML())
    gráfico = estación.agregar(widgets.Image(format="png", layout=widgets.Layout(max_width="100%")))
    muestras_vista = estación.agregar(widgets.HTML())
    checkpoints_vista = estación.agregar(widgets.HTML("<i>Todavía no hay checkpoints.</i>"))
    aviso = estación.agregar(widgets.HTML())

    memoria = {"pasos": [], "pérdidas": [], "pasos_validación": [], "pérdidas_validación": [], "muestras": [], "checkpoints": [],
               "última_vez": 0.0}

    def dibujar_curvas(forzar=False):
        if not forzar and time.perf_counter() - memoria["última_vez"] < 0.7:
            return
        memoria["última_vez"] = time.perf_counter()
        figura = utils.figura_de_pérdidas(memoria["pasos"], memoria["pérdidas"], memoria["pasos_validación"],
                                          memoria["pérdidas_validación"], ventana=suavizado.value)
        gráfico.value = utils.figura_a_png(figura)

    def al_paso(registro):
        memoria["pasos"].append(registro["paso"])
        memoria["pérdidas"].append(registro["pérdida"])
        progreso.value = registro["paso"]
        números.value = utils.html_tabla(
            ["paso", "pérdida de entrenamiento", "tasa de aprendizaje", "tiempo transcurrido"],
            [[f"{registro['paso']} de {progreso.max}", utils.con_coma(registro["pérdida"], 3),
              f"{registro['tasa_de_aprendizaje']:.2e}", f"{registro['segundos']:.0f} s"]])
        dibujar_curvas()

    def al_evaluar(registro):
        memoria["pasos_validación"].append(registro["paso"])
        memoria["pérdidas_validación"].append(registro["pérdida"])
        dibujar_curvas(forzar=True)

    def al_mostrar_muestras(nuevas):
        memoria["muestras"] = list(nuevas) + memoria["muestras"]
        tarjetas = []
        for muestra in memoria["muestras"][:6]:
            tarjetas.append(f"<div style='font-size:.85em;color:{utils.TINTA_SUAVE}'>paso {muestra['paso']}</div>"
                            + utils.html_texto_con_continuación(muestra["comienzo"], muestra["continuación"], "ajustado"))
        muestras_vista.value = "".join(tarjetas)

    def al_guardar(información):
        memoria["checkpoints"].append(información)
        filas = []
        for ítem in memoria["checkpoints"]:
            tipo = "pesos y estado de reanudación (se puede reanudar)" if ítem["con_estado_de_reanudación"] else "solo pesos (sirve para generar)"
            filas.append([str(ítem["paso"]), tipo, ítem["carpeta"]])
        checkpoints_vista.value = utils.html_tabla(["paso", "contenido", "carpeta"], filas)

    def al_iniciar(_):
        vigente, motivo = configuración_vigente(configuración_consolidada)
        if not vigente:
            aviso.value = utils.html_aviso(motivo, "error")
            return
        for lista in ("pasos", "pérdidas", "pasos_validación", "pérdidas_validación", "muestras", "checkpoints"):
            memoria[lista] = []
        progreso.max = configuración_consolidada.pasos_totales
        progreso.value = 0
        try:
            iniciar_entrenamiento_en_segundo_plano(
                configuración_consolidada, preparación, tokenizador, documentos,
                al_paso=al_paso, al_evaluar=al_evaluar, al_muestra=al_mostrar_muestras, al_guardar=al_guardar)
        except (RuntimeError, FileExistsError) as error:
            aviso.value = utils.html_aviso(str(error), "error")
            return
        aviso.value = ""
        botón_iniciar.disabled = True
        botón_detener.disabled = False
        estado.value = utils.html_estado("ejecutando", f"experimento {configuración_consolidada.nombre}")
        vigilante = threading.Thread(target=vigilar, daemon=True)
        vigilante.start()

    def vigilar():
        """Espera el final del hilo del ajuste y actualiza el estado visible."""
        esperar_entrenamiento()
        botón_iniciar.disabled = False
        botón_detener.disabled = True
        if ejecución_actual["estado"] == "falló":
            estado.value = utils.html_estado("falló", ejecución_actual["error"])
        else:
            estado.value = utils.html_estado("completado" if ejecución_actual["estado"] == "completado" else "detenido",
                                             f"paso {ejecución_actual['resultado']['paso_final']}")
        dibujar_curvas(forzar=True)

    def al_detener(_):
        ejecución_actual["parar"].set()
        botón_detener.disabled = True
        estado.value = utils.html_estado("ejecutando", "deteniendo: termina el paso en curso y guarda")

    def al_cambiar_suavizado(_):
        dibujar_curvas(forzar=True)

    estación.al_hacer_clic(botón_iniciar, al_iniciar)
    estación.al_hacer_clic(botón_detener, al_detener)
    estación.observar(suavizado, al_cambiar_suavizado)
    dibujar_curvas(forzar=True)

    explorar = widgets.VBox([widgets.HBox([botón_iniciar, botón_detener]), aviso, progreso, estado, números,
                             laboratorio.fila_de_control(suavizado), gráfico, muestras_vista])
    return laboratorio.panel_de_tres_niveles(explorar, widgets.VBox([checkpoints_vista]), widgets.HTML(
        "<pre>iniciar_entrenamiento_en_segundo_plano(configuración, preparación, tokenizador, documentos)\n"
        "ejecución_actual['parar'].set()   # pedir que se detenga\n"
        "esperar_entrenamiento()           # bloquear hasta que termine</pre>"))

### Panel del ajuste

1. Apretá **Iniciar ajuste**. Mirá el estado (pendiente, ejecutando, completado, detenido o falló: cambia con texto, no solo con color), el progreso y las curvas.
2. Mientras corre, movés el **Suavizado**: solo cambia el dibujo, no los datos.
3. Mirá las **muestras**: son siempre los mismos comienzos, así que podés comparar el paso 0 con el último.
4. **Detener** espera a que termine el paso en curso, guarda un checkpoint y termina. En **Inspeccionar** ves los checkpoints guardados.

### ✎ Antes de ejecutar

¿La pérdida de validación del último paso va a ser menor, igual o mayor que la del paso 0?

In [ ]:
display(armar_estación_5())

### Cómo leer la curva

- **Entrenamiento y validación.** La curva de entrenamiento mide el error sobre los bloques con los que el modelo se ajusta, y suele bajar. La de validación, sobre los textos reservados, muestra si lo aprendido se sostiene con textos nuevos. Si la de validación empieza a subir mientras la otra baja, puede ser una señal de memorización, pero habría que mirar más evidencia para afirmarlo.
- **Trazo fino y trazo grueso.** El fino es el valor real de cada paso (ruidoso); el grueso es una media móvil que ayuda a ver la tendencia pero esconde los saltos. El gráfico dibuja los dos.
- **Pérdida y perplejidad.** La *perplejidad* es la exponencial de la pérdida (`exp(pérdida)`). Se lee como entre cuántos tokens igual de probables estaría «dudando» el modelo, en promedio. Su mínimo teórico es 1, no 0. Una pérdida menor significa que el modelo le asigna más probabilidad al texto de ese conjunto, no que el texto sea mejor literatura.

### Ahora compará el paso 0 con el último

Mirá la muestra del paso 0 y la última: ¿qué cambió y qué no? Separá lo que ves de lo que explicás.

### ✎ Práctica

Dejá un registro más desarrollado. Anotá una **observación** (lo que ves en la curva o en las muestras) y, aparte, una **interpretación** (lo que creés que lo explica). Si querés, sumá una **limitación** y una **hipótesis pendiente**.

In [ ]:
display(armar_bitácora_visual(estación_inicial="5-entrenamiento", lugar="5"))

<details>
<summary>Profundización: qué hace <code>entrenar</code>, paso por paso</summary>

0. **Verifica los datos.** La preparación tiene que seguir siendo la del corpus y el tokenizador actuales, y la misma con la que se creó la configuración (y, al reanudar, la del checkpoint). Si no, levanta un error antes de tocar nada.
1. Si no se reanuda, guarda los ingredientes del experimento (configuración, corpus, partición, entorno) y fija la semilla.
2. Carga una **copia nueva** del modelo y crea el optimizador y el programador de la tasa de aprendizaje. Si reanuda, les devuelve su estado y el azar de cada dispositivo guardado.
3. Mide la **validación en el paso 0**: el punto de partida contra el que se compara todo.
4. Por cada paso: pide los microlotes de ese paso (nunca cruzan de una época a la otra), acumula sus gradientes (`pérdida / acumulación`, para que el promedio sea correcto), **recorta la norma del gradiente** a 1,0 (una protección contra saltos) y actualiza.
5. Cada `evaluar_cada` pasos mide la validación y genera las muestras fijas; cada `guardar_cada` guarda pesos.
6. Si el evento `parar` se activa o el kernel se interrumpe, termina el paso y guarda con estado de reanudación.
7. Si la pérdida deja de ser un número (`nan` o infinito), se detiene con un mensaje que lo explica.

Cada paso queda en `métricas.jsonl` (pérdida, época, tasa de aprendizaje, norma del gradiente, tiempos), así que el historial sobrevive aunque cierres el cuaderno. El ajuste corre en un **hilo aparte**: por eso el cuaderno responde mientras corre, y por eso se niega a lanzar un segundo ajuste. `leer_historial` rearma las curvas y las muestras desde los archivos, sin depender de ningún control. El optimizador usa `weight_decay=0.0` y `modelo.train()` activa el *dropout* (se apagan neuronas al azar para regularizar); la validación lo apaga con `modelo.eval()`.
</details>

<details>
<summary>Profundización: checkpoints, azar y reanudación</summary>

`guardar_checkpoint` distingue dos cosas que suelen confundirse:

| Qué se guardó | Para qué sirve |
|---|---|
| Solo pesos y tokenizador | **Cargar el modelo y generar texto.** No alcanza para continuar el ajuste |
| Pesos y también el estado de reanudación: optimizador, programador y el generador de azar de cada dispositivo | **Reanudar** el ajuste desde ese punto |

Cada checkpoint lleva un `checkpoint.json` que dice cuál de los dos es. En nuestras corridas, un checkpoint intermedio solo con pesos pesó unos 500 MB; con el estado del optimizador, unos 1,5 GB. Por eso los intermedios guardan solo pesos y el último (o el de una detención) guarda también el estado de reanudación.

**Qué azar se guarda, y por dispositivo.** PyTorch lleva un generador de números al azar **por dispositivo**: uno para la CPU, otro para cada GPU de NVIDIA (`cuda`) y otro para la de Mac (`mps`). El dropout sortea en el dispositivo donde está el modelo, por eso se guarda el de la CPU y el de cada acelerador disponible, y `checkpoint.json` anota cuáles guardó. **No se guardan** el módulo `random` de Python ni NumPy (el bucle no los usa). El orden de los bloques no se guarda: se reconstruye con la semilla y el número de paso.

**Cuidado con la palabra «exacto».** Reanudar retoma pesos, optimizador, programador, el azar guardado de cada dispositivo y la posición en los datos. Eso no garantiza una continuación idéntica bit a bit a un ajuste sin interrupciones. En CPU, en este entorno, las pérdidas de un ajuste reanudado coincidieron con las de uno continuo dentro de 1e-4; en `mps` y en `cuda` no se midió. Un checkpoint hecho en un dispositivo y reanudado en otro no restaura el azar del segundo, y el ajuste lo registra como aviso. Al reanudar, `entrenar` también compara las huellas de datos del checkpoint con las que le pasás, y se niega si no coinciden.

**Muestras fijas.** Las muestras periódicas usan siempre los mismos comienzos, la misma temperatura, el mismo `top_k` y `top_p` y la misma semilla. Eso evita que cambie el sorteo entre un paso y otro, pero no elimina su efecto: con pesos distintos el mismo sorteo puede dar continuaciones distintas. Sirven para mirar y comparar con cautela, no como un experimento controlado.
</details>

### Vía con código (alternativa al panel): un ajuste breve

Lanzar el ajuste de la configuración de la estación 4 en un hilo y esperar a que termine. Es el camino sin controles, una alternativa al panel y no un paso previo.

**Advertencia:** esto **entrena de verdad** (60 pasos, lote efectivo 8) y **no corre con «Ejecutar todo»**: poné `PERMITIR_AJUSTE_POR_CÓDIGO = True`. En la medición que hicimos (una Mac M2 con `mps`) llevó unos 2 minutos, con evaluaciones y guardados; en otra máquina puede variar mucho, y en CPU suele tardar más (el ensayo de la estación 4 ayuda a estimarlo). Para una prueba rápida, bajá `DURACIÓN` a 10 y volvé a ejecutar la celda de configuración de la estación 4.

Antes de ejecutar, repetí la apuesta del panel y sumá otra: ¿la perplejidad de validación va a quedar por debajo de 1?

In [ ]:
if PERMITIR_AJUSTE_POR_CÓDIGO:
    ejecución = iniciar_entrenamiento_en_segundo_plano(
        configuración=configuración,
        preparación=preparación,
        tokenizador=tokenizador,
        documentos=documentos,
    )
    estado_final = esperar_entrenamiento()
    print(f"Estado final: {estado_final}")
    if ejecución["error"]:
        print("Error:", ejecución["error"])
else:
    print("Ajuste por código omitido. Para entrenar desde acá poné PERMITIR_AJUSTE_POR_CÓDIGO = True en los parámetros del cuaderno; "
          "o usá el panel de abajo.")

**Mirá el resultado.** El estado final dice si el ajuste terminó, se detuvo o falló (con el error). Las métricas que siguen no se leen de la memoria de la sesión sino de los archivos del experimento activo: el que indica `EXPERIMENTO_A_USAR` o, si es `None`, el más reciente que terminó.

In [ ]:
carpeta_activa = carpeta_del_experimento_activo(EXPERIMENTO_A_USAR)
if carpeta_activa is None:
    print("Todavía no hay un experimento ajustado. Entrená con el panel de la estación 5 o con la vía con código.")
else:
    historial = leer_historial(carpeta_activa)
    validación = pd.DataFrame(historial["validación"])[["paso", "pérdida", "perplejidad"]]
    display(validación)

**Mirá la tabla.** Una fila por evaluación; la primera (paso 0) es el modelo base, antes de ajustar.

- **Observación verificable:** compará la `pérdida` del paso 0 con la del último paso, y fijate que la perplejidad es siempre mayor que 1.
- **Interpretación posible:** si la pérdida de validación bajó, el modelo le asigna más probabilidad a los dos cuentos reservados que antes. Es compatible con que haya aprendido rasgos de la prosa de Quiroga, pero también con que haya aprendido algo más general del formato del texto.
- **Límite:** la validación son solo dos cuentos (24 bloques): la estimación es ruidosa, y una perplejidad menor no dice si los textos generados son mejores.

### ✎ Para pensar

¿Por qué la perplejidad no puede ser menor que 1, aunque el modelo prediga perfecto?

In [ ]:
if carpeta_activa is not None:
    figura = utils.figura_de_pérdidas(
        pasos_entrenamiento=columna(historial["entrenamiento"], "paso"),
        pérdidas_entrenamiento=columna(historial["entrenamiento"], "pérdida"),
        pasos_validación=columna(historial["validación"], "paso"),
        pérdidas_validación=columna(historial["validación"], "pérdida"),
        ventana=5,
    )
    display(Image(data=utils.figura_a_png(figura)))
else:
    print('Sin curvas que dibujar: todavía no hay un experimento ajustado.')

**Mirá el gráfico.** El trazo fino rojo son los valores originales de cada paso; el grueso, la media móvil de 5 pasos. Los círculos azules son las evaluaciones sobre el conjunto fijo de validación.

- **Observación verificable:** en el trazo fino se ven saltos de un paso a otro que el suavizado esconde.
- **Interpretación posible:** un descenso en ambas curvas es compatible con que el ajuste esté funcionando; que la validación quede quieta mientras la de entrenamiento baja sería una señal para mirar más pasos.
- **Límite:** pocas evaluaciones (una cada 20 pasos) y un conjunto de validación chico. Cualquier lectura fina de estas curvas sería sobreinterpretar.

---
## Estación 6. Compará y examiná la evidencia

**Desafío.** Ahora la pregunta del laboratorio: ¿el modelo ajustado escribe un poco más como el autor? Ponelos lado a lado, mirá por dentro cómo elige cada uno el siguiente token y fijate qué evidencia tenés.

Si todavía no ajustaste ningún modelo, solo vas a poder usar el modelo base: volvé a la estación 5. En todas las vistas, el modelo base es verde petróleo con ● y el ajustado es terracota con ▲; los rótulos acompañan siempre al color.

### 6A. Compará continuaciones

**Desafío.** Mismos comienzos, misma configuración, dos modelos: ¿qué diferencias ves entre lo que escriben?

**Preparación · estación 6A.** La celda siguiente define cómo se listan y se cargan los checkpoints (como máximo `MÁXIMO_DE_MODELOS_EN_MEMORIA`, 2, a la vez: cada uno ocupa unos 500 MB), cómo se comparan y el panel. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def listar_modelos_disponibles(raíz=None):
    """Devuelve un diccionario {etiqueta: carpeta} con 'base' y cada checkpoint de cada experimento guardado."""
    raíz = Path(raíz or RAÍZ_DE_EXPERIMENTOS)
    modelos = {"base": None}
    for experimento in sorted(raíz.glob("*/checkpoints")):
        for checkpoint in sorted(experimento.glob("paso_*")):
            if (checkpoint / "config.json").exists():
                modelos[f"{experimento.parent.name} · {checkpoint.name}"] = checkpoint
    return modelos


def etiqueta_del_último_checkpoint(nombre_del_experimento, modelos):
    """Etiqueta del checkpoint más avanzado de un experimento (None si no tiene ninguno). Las etiquetas vienen ordenadas por paso."""
    última = None
    for etiqueta in modelos:
        if etiqueta.startswith(f"{nombre_del_experimento} · "):
            última = etiqueta
    return última


MÁXIMO_DE_MODELOS_EN_MEMORIA = 2  # cada modelo ajustado ocupa unos 500 MB; el modelo base no cuenta
modelos_en_memoria = {}  # etiqueta → {"modificación", "modelo", "tokenizador"}; el menos usado, primero


def modelo_para_generar(etiqueta, carpeta):
    """Devuelve (modelo, tokenizador) listo para generar. 'base' es el modelo de la estación 1.

    Guarda en memoria como máximo MÁXIMO_DE_MODELOS_EN_MEMORIA modelos ajustados: al cargar uno nuevo
    suelta el que hace más tiempo que no se usa. Si la carpeta cambió desde la última carga, lo recarga.
    """
    if etiqueta == "base":
        return modelo_base, tokenizador
    exigir_dispositivo_libre()
    modificación = (Path(carpeta) / "config.json").stat().st_mtime
    guardado = modelos_en_memoria.pop(etiqueta, None)  # lo sacamos para volver a ponerlo al final: pasa a ser el más reciente
    if guardado is None or guardado["modificación"] != modificación:
        tokenizador_guardado = AutoTokenizer.from_pretrained(carpeta)
        modelo = AutoModelForCausalLM.from_pretrained(carpeta, dtype=torch.float32).to(DISPOSITIVO)
        modelo.eval()
        guardado = {"modificación": modificación, "modelo": modelo, "tokenizador": tokenizador_guardado}
    modelos_en_memoria[etiqueta] = guardado
    soltó_alguno = False
    while len(modelos_en_memoria) > MÁXIMO_DE_MODELOS_EN_MEMORIA:
        etiquetas_en_memoria = list(modelos_en_memoria)
        del modelos_en_memoria[etiquetas_en_memoria[0]]
        soltó_alguno = True
    if soltó_alguno:
        liberar_memoria_del_dispositivo()
    return guardado["modelo"], guardado["tokenizador"]


def liberar_modelos_en_memoria():
    """Suelta los modelos ajustados que el comparador tenía en memoria (el modelo base no se toca). Devuelve cuántos soltó."""
    cantidad = len(modelos_en_memoria)
    modelos_en_memoria.clear()
    liberar_memoria_del_dispositivo()
    return cantidad


def tokenizadores_compatibles(un_tokenizador, tokenizador_de_referencia):
    """Devuelve (True, "") si los dos tokenizadores son idénticos, o (False, motivo) si sus números no son comparables."""
    if huella_del_tokenizador(un_tokenizador) != huella_del_tokenizador(tokenizador_de_referencia):
        return False, "otro tokenizador (otro vocabulario o otro fin de texto)"
    return True, ""


def comparar_generaciones(etiquetas_y_carpetas, comienzos, n_muestras=3, max_new_tokens=40, modo="muestreo",
                          temperatura=0.8, top_k=50, top_p=0.95, semilla=42):
    """Genera con CADA modelo, para CADA comienzo, con la misma configuración y las mismas semillas.

    Devuelve una lista de resultados (uno por comienzo y modelo) con su procedencia: de qué carpeta
    viene el modelo, qué huella de corpus y de configuración tenía, y cuándo se generó.
    Recorre un modelo por vez (así la caché no recarga el mismo modelo una vez por comienzo).
    """
    huella_del_corpus_vigente = tomar_versión(documentos, "comparación")["huella"]
    resultados_por_modelo = {}
    for etiqueta, carpeta in etiquetas_y_carpetas.items():
        modelo, su_tokenizador = modelo_para_generar(etiqueta, carpeta)
        resultados_de_este_modelo = {}
        for comienzo in comienzos:
            muestras = generar(modelo, su_tokenizador, comienzo, n_muestras=n_muestras, max_new_tokens=max_new_tokens,
                               modo=modo, temperatura=temperatura, top_k=top_k, top_p=top_p, semilla=semilla,
                               procedencia=etiqueta)
            procedencia = {"modelo": etiqueta, "carpeta": str(carpeta) if carpeta else MODELO_BASE,
                           "huella_del_tokenizador": huella_del_tokenizador(su_tokenizador)[:16],
                           "huella_del_corpus_vigente": huella_del_corpus_vigente,
                           "hora": laboratorio.ahora()}
            if carpeta is not None:
                procedencia["modificación_del_modelo"] = (Path(carpeta) / "config.json").stat().st_mtime
                archivo_de_configuración = Path(carpeta).parent.parent / "configuración.json"
                if archivo_de_configuración.exists():
                    procedencia["huella_de_la_configuración"] = laboratorio.leer_json(archivo_de_configuración).get("huella_de_la_configuración")
                    procedencia["huella_del_corpus_del_experimento"] = laboratorio.leer_json(archivo_de_configuración).get("huella_del_corpus")
            resultados_de_este_modelo[comienzo] = {"comienzo": comienzo, "etiqueta": etiqueta, "muestras": muestras,
                                                   "procedencia": procedencia}
        resultados_por_modelo[etiqueta] = resultados_de_este_modelo
    resultados = []
    for comienzo in comienzos:
        for etiqueta in etiquetas_y_carpetas:
            resultados.append(resultados_por_modelo[etiqueta][comienzo])
    return resultados


def vigencia_de(resultado):
    """Dice si un resultado guardado sigue siendo vigente. Devuelve (True/False, lista de motivos).

    Queda desactualizado si cambió el corpus de trabajo respecto del que usó el experimento, si el
    modelo de esa carpeta fue modificado o ya no existe.
    """
    procedencia = resultado["procedencia"]
    motivos = []
    carpeta = procedencia["carpeta"]
    if procedencia["modelo"] != "base":
        config_json = Path(carpeta) / "config.json"
        if not config_json.exists():
            motivos.append("el checkpoint ya no existe")
        elif config_json.stat().st_mtime != procedencia.get("modificación_del_modelo"):
            motivos.append("el modelo de esa carpeta cambió")
        corpus_del_experimento = procedencia.get("huella_del_corpus_del_experimento")
        if corpus_del_experimento and corpus_del_experimento != tomar_versión(documentos, "vigencia")["huella"]:
            motivos.append("el corpus de trabajo cambió desde que se ajustó este modelo")
    return len(motivos) == 0, motivos


bitácora_de_comparaciones = laboratorio.Bitácora(RAÍZ_DE_EXPERIMENTOS / "comparaciones" / "anotaciones.jsonl")


def experimentos_de_una_comparación(resultados):
    """Nombres de las carpetas de experimento de los modelos de una comparación (el modelo base no pertenece a ninguno)."""
    nombres = []
    for resultado in resultados:
        if resultado["procedencia"]["modelo"] != "base":
            nombres.append(Path(resultado["procedencia"]["carpeta"]).parent.parent.name)
    return sorted(set(nombres))


def armar_estación_6a():
    """Comparador de generación con resultados que se conservan y anotaciones."""
    estación = laboratorio.nueva_estación("6a-comparador")
    bitácora = bitácora_de_comparaciones
    registro = laboratorio.RegistroJSONL(RAÍZ_DE_EXPERIMENTOS / "comparaciones" / "comparaciones.jsonl")
    lista_actual = {"modelos": listar_modelos_disponibles(), "comparaciones": []}

    selector = estación.agregar(widgets.SelectMultiple(options=list(lista_actual["modelos"]), value=("base",), rows=5))
    laboratorio.control(selector, "Modelos (Ctrl/Cmd para varios)", "MODELOS_A_COMPARAR", tipo="experimento",
                        ayuda="«base» es el modelo sin ajustar; los demás son checkpoints de tus experimentos.",
                        prueba="elegí «base» y el último checkpoint de tu experimento.")
    selector.layout.max_width = "90em"  # los nombres de checkpoint son largos y no tienen que quedar cortados
    comienzos = estación.agregar(widgets.Textarea(value="Una mañana de otoño,\nEsa noche", rows=3, layout=widgets.Layout(width="100%")))
    laboratorio.control(comienzos, "Comienzos (uno por línea)", "comienzos", tipo="experimento",
                        ayuda="Las frases con las que escribe cada modelo. Todos reciben los mismos.",
                        prueba="usá un comienzo que se parezca al autor y otro que no.")
    temperatura = estación.agregar(widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.05, continuous_update=False))
    laboratorio.control(temperatura, "Temperatura", "temperatura", tipo="experimento",
                        ayuda="Igual para todos los modelos. Reparte o concentra las probabilidades antes de sortear.",
                        prueba="dejala fija entre comparaciones para que la diferencia venga del modelo.")
    top_k = estación.agregar(widgets.IntSlider(value=50, min=0, max=200, continuous_update=False))
    laboratorio.control(top_k, "top_k (0 = sin límite)", "top_k", tipo="experimento",
                        ayuda="Deja en juego solo a los k tokens más probables.")
    top_p = estación.agregar(widgets.FloatSlider(value=0.95, min=0.05, max=1.0, step=0.05, continuous_update=False))
    laboratorio.control(top_p, "top_p", "top_p", tipo="experimento",
                        ayuda="Deja en juego al grupo más chico de candidatos cuya probabilidad suma p.")
    máximo = estación.agregar(widgets.IntSlider(value=40, min=5, max=150, continuous_update=False))
    laboratorio.control(máximo, "Tokens nuevos como máximo", "max_new_tokens", tipo="experimento",
                        ayuda="Cuánto escribe cada modelo.")
    cantidad = estación.agregar(widgets.IntSlider(value=3, min=1, max=5, continuous_update=False))
    laboratorio.control(cantidad, "Muestras por modelo y comienzo", "n_muestras", tipo="experimento",
                        ayuda="Con una sola muestra no se distingue el efecto del modelo del azar del sorteo.",
                        prueba="pedí 5 y buscá qué diferencia se repite.")
    semilla = estación.agregar(widgets.IntText(value=7))
    laboratorio.control(semilla, "Semilla", "semilla", tipo="experimento",
                        ayuda="Fija el sorteo. Con pesos distintos, la misma semilla no recorre el mismo camino.",
                        prueba="cambiala y mirá cuánto varía un mismo modelo.")
    botón_actualizar = estación.agregar(laboratorio.botón("Actualizar lista de modelos", "Busca checkpoints nuevos"))
    botón_comparar = estación.agregar(laboratorio.botón("Comparar", "Genera con cada modelo seleccionado", "primary"))
    botón_vigencia = estación.agregar(laboratorio.botón("Revisar vigencia", "Marca resultados desactualizados"))
    botón_liberar = estación.agregar(laboratorio.botón("Liberar memoria", "Suelta los checkpoints cargados; el modelo base queda"))
    botón_restaurar = estación.agregar(laboratorio.botón("Volver a los valores iniciales", "Solo mueve los controles; los resultados quedan"))
    estado = estación.agregar(widgets.HTML(utils.html_estado("pendiente")))
    resultados_vista = estación.agregar(widgets.VBox([]))
    cuál = estación.agregar(widgets.Dropdown(options=[]))
    laboratorio.control(cuál, "Comparación a anotar", "número", tipo="registro",
                        ayuda="Cuál de las comparaciones de esta sesión estás anotando.")
    tipo = estación.agregar(widgets.Dropdown(options=list(laboratorio.TIPOS_DE_ANOTACIÓN), value="observado"))
    laboratorio.control(tipo, "Tipo de anotación", "tipo", tipo="registro",
                        ayuda="Observado: lo que ves. Interpretación: lo que creés que lo explica. Limitación: lo que lo debilita. Hipótesis pendiente: lo que habría que probar.")
    texto = estación.agregar(widgets.Textarea(rows=3, layout=widgets.Layout(width="100%")))
    texto.description = "Anotación"
    botón_anotar = estación.agregar(laboratorio.botón("Guardar anotación", "Queda en anotaciones.jsonl", "success"))
    anotaciones_vista = estación.agregar(widgets.HTML())
    procedencia_vista = estación.agregar(widgets.HTML("<i>Después de comparar, acá vas a ver de dónde viene cada resultado: la carpeta del modelo, la huella de su tokenizador y la hora.</i>"))

    def tarjetas_de(número, resultados):
        columnas = []
        for resultado in resultados:
            actor = "base" if resultado["etiqueta"] == "base" else "ajustado"
            vigente, motivos = vigencia_de(resultado)
            insignia = "" if vigente else utils.html_aviso("DESACTUALIZADO: " + "; ".join(motivos))
            cuerpo = ""
            for muestra in resultado["muestras"]:
                cuerpo += utils.html_texto_con_continuación(muestra["entrada"], muestra["continuación"], actor)
            columnas.append(widgets.HTML(
                f"<div style='min-width:18em'><b>{utils.escapar(utils.etiqueta_de_actor(actor))}</b> · {utils.escapar(resultado['etiqueta'])}"
                f"{insignia}{cuerpo}</div>"))
        return widgets.VBox([widgets.HTML(f"<h4>Comparación {número}: comienzo {utils.escapar(resultados[0]['comienzo'])!s}</h4>"),
                             widgets.HBox(columnas, layout=widgets.Layout(flex_flow="row wrap"))])

    def redibujar():
        bloques = []
        for número, resultados_de_comienzos in lista_actual["comparaciones"]:
            por_comienzo = {}
            for resultado in resultados_de_comienzos:
                por_comienzo.setdefault(resultado["comienzo"], []).append(resultado)
            for resultados_del_comienzo in por_comienzo.values():
                bloques.append(tarjetas_de(número, resultados_del_comienzo))
        resultados_vista.children = tuple(reversed(bloques))
        números_de_comparación = []
        for número, _ in lista_actual["comparaciones"]:
            números_de_comparación.append(número)
        cuál.options = números_de_comparación
        if números_de_comparación:
            cuál.value = números_de_comparación[-1]  # por defecto, la comparación más nueva

    estación.recordar_valores_iniciales(selector, comienzos, temperatura, top_k, top_p, máximo, cantidad, semilla)

    def dibujar_procedencia(resultados):
        filas = []
        for resultado in resultados:
            procedencia = resultado["procedencia"]
            filas.append([resultado["etiqueta"], procedencia["carpeta"], procedencia["huella_del_tokenizador"], procedencia["hora"]])
        procedencia_vista.value = utils.html_tabla(["modelo", "carpeta", "huella del tokenizador", "hora"], filas)

    def al_restaurar(_):
        estación.restaurar_valores_iniciales()

    def al_actualizar(_):
        lista_actual["modelos"] = listar_modelos_disponibles()
        selector.options = list(lista_actual["modelos"])

    def al_comparar(_):
        with laboratorio.Ocupado([botón_comparar, botón_actualizar, botón_vigencia, botón_liberar], estado, "Comparando", utils.html_estado) as ocupado:
            elegidos = {}
            for etiqueta in selector.value:
                elegidos[etiqueta] = lista_actual["modelos"][etiqueta]
            if not elegidos:
                raise ValueError("Elegí al menos un modelo.")
            lista_de_comienzos = utils.líneas_con_texto(comienzos.value)
            resultados = comparar_generaciones(elegidos, lista_de_comienzos, cantidad.value, máximo.value, "muestreo",
                                               temperatura.value, top_k.value, top_p.value, semilla.value)
            número = len(lista_actual["comparaciones"]) + 1
            lista_actual["comparaciones"].append((número, resultados))
            registro.agregar({"número": número, "resultados": resultados})
            redibujar()
            dibujar_procedencia(resultados)
        if ocupado.error is not None:
            resultados_vista.children = (widgets.HTML(utils.html_aviso(str(ocupado.error), "error")),) + tuple(resultados_vista.children)

    def al_anotar(_):
        if cuál.value is None:
            anotaciones_vista.value = utils.html_aviso("Primero hacé una comparación y elegí cuál anotás.", "error")
            return
        resultados_anotados = []
        for número_guardado, resultados_de_ese_número in lista_actual["comparaciones"]:
            if número_guardado == cuál.value:
                resultados_anotados = resultados_de_ese_número
        modelos = []
        for resultado in resultados_anotados:
            modelos.append(resultado["etiqueta"])
        try:
            bitácora.anotar("6a-comparador", tipo.value, texto.value,
                            contexto={"comparación": cuál.value, "modelos": sorted(set(modelos))},
                            experimentos=experimentos_de_una_comparación(resultados_anotados))
        except ValueError as error:
            anotaciones_vista.value = utils.html_aviso(str(error), "error")
            return
        texto.value = ""
        filas = []
        for entrada in bitácora.leer():
            filas.append([str(entrada["contexto"].get("comparación")), entrada["tipo"], entrada["texto"]])
        anotaciones_vista.value = utils.html_tabla(["comparación", "tipo", "anotación"], filas)

    def al_revisar(_):
        redibujar()

    def al_liberar(_):
        cantidad = liberar_modelos_en_memoria()
        estado.value = utils.html_estado("completado", f"{cantidad} modelos ajustados liberados de la memoria")

    estación.al_hacer_clic(botón_actualizar, al_actualizar)
    estación.al_hacer_clic(botón_comparar, al_comparar)
    estación.al_hacer_clic(botón_vigencia, al_revisar)
    estación.al_hacer_clic(botón_liberar, al_liberar)
    estación.al_hacer_clic(botón_anotar, al_anotar)
    estación.al_hacer_clic(botón_restaurar, al_restaurar)

    filas_de_parámetros = []
    for control_de_generación in (temperatura, top_k, top_p, máximo, cantidad, semilla):
        filas_de_parámetros.append(laboratorio.fila_de_control(control_de_generación))
    parámetros_de_generación = laboratorio.sección_plegable(
        "Parámetros de generación (los mismos para todos los modelos)", widgets.VBox(filas_de_parámetros), abierta=True)
    explorar = widgets.VBox([laboratorio.fila_de_control(selector), laboratorio.fila_de_control(comienzos), parámetros_de_generación,
                             widgets.HBox([botón_actualizar, botón_comparar, botón_vigencia, botón_liberar, botón_restaurar]),
                             estado, resultados_vista])
    inspeccionar = widgets.VBox([procedencia_vista])
    anotar = widgets.VBox([laboratorio.fila_de_control(cuál), laboratorio.fila_de_control(tipo), texto, botón_anotar, anotaciones_vista])
    return laboratorio.panel_de_tres_niveles(explorar, inspeccionar, anotar)

#### Panel: comparar modelos

Elegí `base` y el último checkpoint de tu experimento, dejá los comienzos que vienen y apretá **Comparar**.

1. Cada comparación se guarda y se apila arriba: las anteriores no desaparecen.
2. Cambiá **un** parámetro de generación (por ejemplo, la temperatura) y compará de nuevo; o dejá los parámetros y cambiá el comienzo.
3. **Revisar vigencia** marca como desactualizado un resultado cuyo modelo o corpus cambió después. **Liberar memoria** suelta los checkpoints cargados.
4. En **Inspeccionar** ves de dónde viene cada resultado. En **Modificar** anotás: elegí la comparación, el tipo de anotación y escribí.

Cada anotación se guarda con un identificador propio y con los experimentos de los modelos que anotás; así la estación 7 la asocia a su experimento por esos datos y no por la fecha.

### ✎ Antes de ejecutar

¿Qué esperás que cambie entre el modelo base y el ajustado: el vocabulario, los nombres propios, el largo de las frases?

In [ ]:
display(armar_estación_6a())

### Cómo leer la comparación

Con la misma semilla, dos modelos con pesos distintos **no** recorren el mismo camino aleatorio: el sorteo es el mismo, pero las probabilidades cambian. Por eso conviene pedir más de una muestra y buscar diferencias que se repitan. Si algo aparece en varias muestras del ajustado y en ninguna del base, es compatible con un efecto del ajuste; si también aparece entre dos semillas del mismo modelo, no hace falta el ajuste para explicarlo.

Una precaución más: que un texto «suene» a un autor no nos dice si ese autor estaba entre los datos con que se entrenó el modelo base. No conocemos sus fuentes concretas (ver `docs/MODELO.md`).

### ✎ Para pensar

¿Qué haría falta para convertir una impresión («suena más a Quiroga») en una afirmación con evidencia?

### 6A. Vía con código (alternativa al panel)

Comparar el modelo base con el último checkpoint del experimento activo (`EXPERIMENTO_A_USAR`, o el más reciente que terminó), con dos comienzos y tres muestras cada uno. Carga checkpoints, así que necesita `PERMITIR_ANALIZAR_CHECKPOINTS = True`.

In [ ]:
COMIENZOS_A_COMPARAR = ["Una mañana de otoño,", "Esa noche"]

carpeta_activa = carpeta_del_experimento_activo(EXPERIMENTO_A_USAR)
if not PERMITIR_ANALIZAR_CHECKPOINTS:
    print('Omitido: esta celda carga checkpoints (unos 500 MB cada uno) y puede escribir en la carpeta del experimento. Poné PERMITIR_ANALIZAR_CHECKPOINTS = True en los parámetros del cuaderno para ejecutarla.')
elif carpeta_activa is None:
    print("Todavía no hay un experimento ajustado: no hay un checkpoint para comparar con el modelo base.")
else:
    disponibles = listar_modelos_disponibles()
    MODELOS_A_COMPARAR = ["base", etiqueta_del_último_checkpoint(carpeta_activa.name, disponibles)]
    elegidos = {}
    for etiqueta in MODELOS_A_COMPARAR:
        elegidos[etiqueta] = disponibles[etiqueta]

    comparación_a = comparar_generaciones(
        etiquetas_y_carpetas=elegidos,
        comienzos=COMIENZOS_A_COMPARAR,
        n_muestras=3,
        max_new_tokens=40,
        semilla=7,
    )

    for resultado in comparación_a:
        actor = "base" if resultado["etiqueta"] == "base" else "ajustado"
        print(f"— {utils.etiqueta_de_actor(actor)} · {resultado['etiqueta']} · comienzo: {resultado['comienzo']!r}")
        for muestra in resultado["muestras"]:
            print("    ", muestra["continuación"][:110].replace("\n", " "))

**Mirá el resultado.** Cada bloque agrupa las tres muestras de un modelo para un comienzo.

- **Observación verificable:** copiá una diferencia concreta (una palabra, un nombre, una construcción) y anotá en qué muestra aparece.
- **Interpretación posible:** si esa diferencia aparece en varias muestras del ajustado y en ninguna del base, es compatible con un efecto del ajuste.
- **Límite:** tres muestras por celda, un solo ajuste corto, un corpus chico y semillas que no recorren el mismo camino. Es un indicio, no una prueba.

---
### 6B. Mirá el siguiente token

**Desafío.** En un punto concreto del texto, ¿qué candidatos tiene cada modelo para seguir y cuánta probabilidad le da a cada uno? ¿Cambia eso entre el base y el ajustado? Y si elegís vos el token, ¿cómo cambia el camino?

**Preparación · estación 6B.** La celda siguiente define `transformar_logits` (la temperatura, `top_k` y `top_p` a la vista), el cálculo de candidatos y el panel. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def transformar_logits(logits, temperatura, top_k, top_p):
    """Aplica temperatura, top_k y top_p a los logits del siguiente token, en el mismo orden que `generate`.

    Devuelve los logits transformados: los tokens descartados quedan en -infinito.
    """
    transformados = logits.clone().float()
    transformados = transformados / temperatura                      # 1. temperatura
    if top_k > 0:
        k = min(top_k, transformados.shape[-1])
        valor_de_corte = torch.topk(transformados, k).values[-1]
        transformados[transformados < valor_de_corte] = -float("inf")     # 2. top_k
    if top_p < 1.0:
        ordenados, posiciones = torch.sort(transformados, descending=False)
        acumuladas = torch.softmax(ordenados, dim=-1).cumsum(dim=-1)
        a_quitar = acumuladas <= (1 - top_p)                          # 3. top_p
        a_quitar[-1] = False  # el más probable siempre se queda
        transformados[posiciones[a_quitar]] = -float("inf")
    return transformados


def distribución_del_siguiente_token(modelo, identificadores, temperatura=0.8, top_k=50, top_p=0.95):
    """Devuelve las probabilidades del siguiente token: la original y la transformada (dos vectores de 50.257)."""
    exigir_dispositivo_libre()
    entrada = torch.tensor([identificadores], device=modelo.device)
    with torch.no_grad():
        logits = modelo(input_ids=entrada).logits[0, -1, :].float().cpu()
    original = torch.softmax(logits, dim=-1)
    transformada = torch.softmax(transformar_logits(logits, temperatura, top_k, top_p), dim=-1)
    return original, transformada


def candidatos_principales(original, transformada, un_tokenizador, cantidad=12):
    """Los `cantidad` tokens más probables según la distribución original, con las dos probabilidades y la masa que cubren."""
    valores, posiciones = torch.topk(original, cantidad)
    candidatos = []
    masa_original = 0.0
    masa_transformada = 0.0
    valores_de_la_lista = valores.tolist()
    posiciones_de_la_lista = posiciones.tolist()
    for orden in range(cantidad):
        valor = valores_de_la_lista[orden]
        posición = posiciones_de_la_lista[orden]
        probabilidad_transformada = transformada[posición].item()
        candidatos.append((un_tokenizador.decode([posición]), posición, valor, probabilidad_transformada))
        masa_original += valor
        masa_transformada += probabilidad_transformada
    return {
        "candidatos": candidatos,
        "masa_mostrada_original": masa_original,
        "masa_mostrada_transformada": masa_transformada,
        "tokens_con_probabilidad_transformada": int((transformada > 0).sum().item()),
    }


def elegir_token(transformada, criterio, semilla):
    """Elige un token de la distribución transformada: el más probable ('voraz') o por sorteo ('muestreo')."""
    if criterio == "voraz":
        return int(torch.argmax(transformada).item())
    if criterio == "muestreo":
        generador = torch.Generator()
        generador.manual_seed(semilla)
        return int(torch.multinomial(transformada, 1, generator=generador).item())
    raise ValueError(f"Criterio desconocido: {criterio!r}. Usá 'voraz' o 'muestreo'.")


def armar_estación_6b():
    """Inspector del siguiente token: compara dos modelos sobre un mismo contexto y avanza de a un token."""
    estación = laboratorio.nueva_estación("6b-siguiente-token")
    modelos = listar_modelos_disponibles()
    etiquetas_de_ajustados = []
    for etiqueta_disponible in modelos:
        if etiqueta_disponible != "base":
            etiquetas_de_ajustados.append(etiqueta_disponible)
    registro_de_ramas = laboratorio.RegistroJSONL(RAÍZ_DE_EXPERIMENTOS / "comparaciones" / "ramas.jsonl")
    contexto_inicial = "La casa estaba vacía y el perro"
    estado_de_la_rama = {"ids": tokenizador(contexto_inicial)["input_ids"], "pasos": []}

    contexto = estación.agregar(widgets.Textarea(value=contexto_inicial, rows=2, layout=widgets.Layout(width="100%")))
    laboratorio.control(contexto, "Contexto", "contexto", tipo="experimento",
                        ayuda="El texto desde el que se calculan los candidatos del siguiente token.",
                        prueba="escribí una frase de Quiroga y comparala con una tuya.")
    ajustado = estación.agregar(widgets.Dropdown(options=etiquetas_de_ajustados or ["(no hay checkpoints)"]))
    laboratorio.control(ajustado, "Modelo ajustado", "checkpoint", tipo="experimento",
                        ayuda="Qué checkpoint se compara con el modelo base.",
                        prueba="elegí uno intermedio y uno final y comparalos.")
    temperatura = estación.agregar(widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.05, continuous_update=False))
    laboratorio.control(temperatura, "Temperatura", "temperatura", tipo="experimento",
                        ayuda="Cambia la distribución transformada (la que se sortea), no la original del modelo.",
                        prueba="subila y recalculá: la barra clara no cambia, la rayada sí.")
    top_k = estación.agregar(widgets.IntSlider(value=50, min=0, max=200, continuous_update=False))
    laboratorio.control(top_k, "top_k (0 = sin límite)", "top_k", tipo="experimento",
                        ayuda="Deja en juego solo a los k tokens más probables.")
    top_p = estación.agregar(widgets.FloatSlider(value=0.95, min=0.05, max=1.0, step=0.05, continuous_update=False))
    laboratorio.control(top_p, "top_p", "top_p", tipo="experimento",
                        ayuda="Deja en juego al grupo más chico de candidatos cuya probabilidad suma p.")
    cuántos = estación.agregar(widgets.IntSlider(value=10, min=3, max=25, continuous_update=False))
    laboratorio.control(cuántos, "Candidatos a mostrar", "cantidad", tipo="vista",
                        ayuda="Cuántos candidatos se dibujan. Cada gráfico dice qué parte de la probabilidad no se ve.",
                        prueba="subilo a 25 y mirá cuánta probabilidad queda sin mostrar.")
    criterio = estación.agregar(widgets.ToggleButtons(options=["muestreo", "voraz"], value="muestreo"))
    laboratorio.control(criterio, "Cómo avanza", "criterio", ancho="34em", tipo="experimento",
                        ayuda="«muestreo» sortea según la distribución transformada; «voraz» toma el token más probable.",
                        prueba="avanzá varias veces en cada modo y compará los caminos.")
    botón_calcular = estación.agregar(laboratorio.botón("Calcular candidatos", "Una pasada por cada modelo", "primary"))
    botón_base = estación.agregar(laboratorio.botón("Avanzar con el base", "Agrega un token elegido por el modelo base"))
    botón_ajustado = estación.agregar(laboratorio.botón("Avanzar con el ajustado", "Agrega un token elegido por el ajustado"))
    manual = estación.agregar(widgets.Dropdown(options=[]))
    laboratorio.control(manual, "Token elegido por vos", "id", tipo="experimento",
                        ayuda="Un token que elegís vos en lugar de dejar que elija el modelo; el paso queda marcado como intervención humana.",
                        prueba="elegí uno poco probable y mirá cómo cambia el camino.")
    botón_manual = estación.agregar(laboratorio.botón("Agregar token elegido (intervención humana)", "Queda marcado como ◆"))
    botón_registrar = estación.agregar(laboratorio.botón("Registrar rama", "Guarda el contexto y los pasos", "success"))
    botón_restaurar = estación.agregar(laboratorio.botón("Volver a los valores iniciales", "Solo mueve los controles"))
    estado = estación.agregar(widgets.HTML(utils.html_estado("pendiente")))
    gráfico = estación.agregar(widgets.Image(format="png", layout=widgets.Layout(max_width="100%", display="none")))  # oculto hasta que haya un gráfico
    rama_vista = estación.agregar(widgets.HTML())
    candidatos_vista = estación.agregar(widgets.HTML("<i>Calculá los candidatos para ver acá los números detrás del gráfico.</i>"))
    memoria = {"candidatos": {}}
    estación.recordar_valores_iniciales(contexto, ajustado, temperatura, top_k, top_p, cuántos, criterio)

    def cargar_contexto(_=None):
        estado_de_la_rama["ids"] = tokenizador(contexto.value)["input_ids"] if contexto.value else [tokenizador.eos_token_id]
        estado_de_la_rama["pasos"] = []
        dibujar_rama()

    def dibujar_rama():
        piezas = piezas_de_los_tokens(estado_de_la_rama["ids"], tokenizador)
        marcas = ""
        for paso_de_la_rama in estado_de_la_rama["pasos"]:
            if "humana" in paso_de_la_rama["origen"]:
                actor = "humano"
            elif "ajustado" in paso_de_la_rama["origen"]:
                actor = "ajustado"
            else:
                actor = "base"
            marcas += (f"<span style='margin-right:6px'>{utils.escapar(utils.etiqueta_de_actor(actor))}: "
                       f"{utils.escapar(utils.pieza_visible(paso_de_la_rama['token']))}</span>")
        rama_vista.value = (utils.html_tokens(piezas, estado_de_la_rama["ids"])
                            + f"<div>Pasos agregados: {marcas or 'ninguno'}</div>")

    def calcular(_=None):
        with laboratorio.Ocupado([botón_calcular, botón_base, botón_ajustado, botón_manual], estado, "Calculando candidatos", utils.html_estado) as ocupado:
            paneles = []
            avisos = []
            memoria["candidatos"] = {}
            for actor, etiqueta in (("base", "base"), ("ajustado", ajustado.value)):
                if etiqueta not in modelos:
                    continue
                modelo, su_tokenizador = modelo_para_generar(etiqueta, modelos[etiqueta])
                compatible, motivo = tokenizadores_compatibles(su_tokenizador, tokenizador)
                if not compatible:
                    avisos.append(f"{etiqueta}: {motivo}. Sus probabilidades no se calculan, porque el contexto está en los tokens del modelo base.")
                    continue
                original, transformada = distribución_del_siguiente_token(modelo, estado_de_la_rama["ids"], temperatura.value, top_k.value, top_p.value)
                principales = candidatos_principales(original, transformada, su_tokenizador, cuántos.value)
                memoria["candidatos"][actor] = (original, transformada, principales)
                candidatos_para_graficar = []
                for candidato in principales["candidatos"]:
                    candidatos_para_graficar.append((candidato[0], candidato[2], candidato[3]))
                paneles.append({"actor": actor, "candidatos": candidatos_para_graficar,
                                "masa_mostrada_original": principales["masa_mostrada_original"],
                                "masa_mostrada_transformada": principales["masa_mostrada_transformada"]})
            gráfico.value = utils.figura_a_png(utils.figura_de_candidatos(paneles))
            filas_de_candidatos = []
            for actor_calculado in memoria["candidatos"]:
                principales_del_actor = memoria["candidatos"][actor_calculado][2]
                for texto_, identificador, p_original, p_transformada in principales_del_actor["candidatos"]:
                    filas_de_candidatos.append([utils.etiqueta_de_actor(actor_calculado), utils.pieza_visible(texto_), str(identificador),
                                                utils.con_coma(p_original, 3), utils.con_coma(p_transformada, 3)])
            candidatos_vista.value = utils.html_tabla(["modelo", "token", "id", "probabilidad original", "probabilidad transformada"], filas_de_candidatos)
            gráfico.layout.display = ""
            unión = {}
            for _, _, principales in memoria["candidatos"].values():
                for texto_, identificador, _, _ in principales["candidatos"]:
                    unión[f"{utils.pieza_visible(texto_)}  (id {identificador})"] = identificador
            manual.options = list(unión.items())
        if ocupado.error is not None:
            rama_vista.value = utils.html_aviso(str(ocupado.error), "error")
        elif avisos:
            dibujar_rama()
            rama_vista.value = rama_vista.value + utils.html_avisos(avisos)

    def agregar_token(identificador, origen, actor):
        original, transformada, _ = memoria["candidatos"][actor if actor in memoria["candidatos"] else "base"]
        estado_de_la_rama["ids"] = estado_de_la_rama["ids"] + [identificador]
        estado_de_la_rama["pasos"].append({
            "token": tokenizador.decode([identificador]), "id": identificador, "origen": origen,
            "p_original": original[identificador].item(), "p_transformada": transformada[identificador].item()})
        dibujar_rama()
        calcular()

    def avanzar_con(actor):
        if actor not in memoria["candidatos"]:
            calcular()
        if actor not in memoria["candidatos"]:
            return  # calcular() dejó a la vista por qué no hay candidatos de este modelo
        _, transformada, _ = memoria["candidatos"][actor]
        identificador = elegir_token(transformada, criterio.value, 1000 + len(estado_de_la_rama["pasos"]))
        agregar_token(identificador, f"modelo {actor} ({criterio.value})", actor)

    def al_manual(_):
        if manual.value is None or not memoria["candidatos"]:
            rama_vista.value = utils.html_aviso("Calculá los candidatos primero.", "error")
            return
        agregar_token(manual.value, "intervención humana", "base")

    def al_restaurar(_):
        estación.restaurar_valores_iniciales()

    def al_registrar(_):
        registro_de_ramas.agregar({"contexto_inicial": contexto.value, "pasos": estado_de_la_rama["pasos"],
                                   "texto_final": tokenizador.decode(estado_de_la_rama["ids"]),
                                   "parámetros": {"temperatura": temperatura.value, "top_k": top_k.value, "top_p": top_p.value}})
        estado.value = utils.html_estado("completado", "rama registrada")

    estación.al_hacer_clic(botón_calcular, calcular)
    def al_avanzar_con_el_base(_):
        avanzar_con("base")

    def al_avanzar_con_el_ajustado(_):
        avanzar_con("ajustado")

    estación.al_hacer_clic(botón_base, al_avanzar_con_el_base)
    estación.al_hacer_clic(botón_ajustado, al_avanzar_con_el_ajustado)
    estación.al_hacer_clic(botón_manual, al_manual)
    estación.al_hacer_clic(botón_registrar, al_registrar)
    estación.al_hacer_clic(botón_restaurar, al_restaurar)
    estación.observar(contexto, cargar_contexto)
    dibujar_rama()

    filas_de_parámetros = []
    for control_del_inspector in (ajustado, temperatura, top_k, top_p, cuántos, criterio):
        filas_de_parámetros.append(laboratorio.fila_de_control(control_del_inspector))
    parámetros_del_inspector = laboratorio.sección_plegable("Parámetros", widgets.VBox(filas_de_parámetros), abierta=True)
    explorar = widgets.VBox([laboratorio.fila_de_control(contexto), parámetros_del_inspector,
                             widgets.HBox([botón_calcular, botón_registrar, botón_restaurar]), estado, gráfico,
                             widgets.HBox([botón_base, botón_ajustado]), laboratorio.fila_de_control(manual), botón_manual, rama_vista])
    inspeccionar = widgets.VBox([candidatos_vista])
    return laboratorio.panel_de_tres_niveles(explorar, inspeccionar, widgets.HTML(
        "<pre>original, transformada = distribución_del_siguiente_token(modelo, identificadores, temperatura, top_k, top_p)\n"
        "principales = candidatos_principales(original, transformada, tokenizador)\n"
        "elegido = elegir_token(transformada, criterio='muestreo', semilla=1)</pre>"))

#### Panel: candidatos del siguiente token

Escribí un contexto (o dejá el inicial) y apretá **Calcular candidatos**: hace una pasada por cada modelo, y nada se calcula al mover un control.

1. Mirá cuánta probabilidad cubren los candidatos que se dibujan: el título de cada gráfico lo dice, y el resto está repartido entre miles de tokens que no se ven.
2. Subí la temperatura y recalculá. Cada candidato tiene dos barras: la clara es la distribución **original** del modelo (la softmax de los puntajes); la rayada es la **transformada**, la que realmente se sortea después de aplicar temperatura, `top_k` y `top_p`. ¿Cuál cambia?
3. Apretá **Avanzar con el base** y después **con el ajustado**: el texto crece con la elección de cada uno (● y ▲).
4. Elegí vos un token con **Agregar token elegido**: el paso queda marcado como ◆, intervención humana.
5. **Registrar rama** guarda el contexto y los pasos. La pestaña **Inspeccionar** tiene los números detrás del gráfico.

### ✎ Antes de ejecutar

Para «La casa estaba vacía y el perro», ¿cuánta probabilidad esperás que reúna el candidato más probable? ¿Más o menos que la mitad?

In [ ]:
display(armar_estación_6b())

### Qué miraste en los candidatos

Las dos distribuciones no se confunden: la **original** suma 1 sobre los 50.257 tokens; la **transformada** se recalcula solo sobre los tokens que sobreviven a `top_k` y `top_p`, y con una temperatura menor que 1 suele darle **más** probabilidad al candidato más probable. Esa diferencia no significa que el modelo «dude» más o menos: es el efecto de los filtros. Si elegís vos el token, el paso queda registrado distinto de los que eligió el modelo.

### ✎ Para pensar

Si el ajustado le da más probabilidad que el base a una palabra del corpus, ¿eso prueba que «aprendió el estilo» o solo que aprendió esa palabra?

### 6B. Vía con código (alternativa al panel)

Calcular los candidatos del siguiente token para un contexto con el modelo base y avanzar tres tokens con muestreo. Cada paso registra de dónde vino el token.

In [ ]:
CONTEXTO = "La casa estaba vacía y el perro"
TEMPERATURA_DEL_INSPECTOR = 0.8
TOP_K_DEL_INSPECTOR = 50
TOP_P_DEL_INSPECTOR = 0.95

identificadores = tokenizador(CONTEXTO)["input_ids"]
rama = []  # registro de pasos: qué token se agregó, quién lo eligió y con qué probabilidades

for número_de_paso in range(3):
    original, transformada = distribución_del_siguiente_token(
        modelo_base, identificadores, TEMPERATURA_DEL_INSPECTOR, TOP_K_DEL_INSPECTOR, TOP_P_DEL_INSPECTOR)
    token_elegido = elegir_token(transformada, criterio="muestreo", semilla=100 + número_de_paso)
    rama.append({"token": tokenizador.decode([token_elegido]), "id": token_elegido, "origen": "modelo base (muestreo)",
                 "p_original": original[token_elegido].item(), "p_transformada": transformada[token_elegido].item()})
    identificadores = identificadores + [token_elegido]

principales = candidatos_principales(original, transformada, tokenizador)
print(f"Texto después de tres pasos: {tokenizador.decode(identificadores)!r}")
print(f"Candidatos mostrados: {len(principales['candidatos'])} · masa original {principales['masa_mostrada_original']:.3f} · masa transformada {principales['masa_mostrada_transformada']:.3f}")
display(pd.DataFrame(rama))

**Mirá el resultado.** La tabla muestra, para cada token agregado, su probabilidad antes y después de los filtros. Compará `p_original` con `p_transformada`: con `top_k` y `top_p` activos, la transformada suele ser mayor, porque se reparte entre menos candidatos. Una rama es un camino entre muchos posibles; no representa lo que el modelo «piensa» de ese texto.

---
### 6C. ¿Qué evidencia tenemos?

**Desafío.** Leé una generación con criterios explícitos, medí con números y buscá si lo que escribió el modelo aparece idéntico en el corpus: tres miradas distintas sobre la misma pregunta.

**Preparación · estación 6C.** La celda siguiente define la evaluación, la búsqueda de coincidencias y la ficha de lectura. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def evaluar_modelos(etiquetas_y_carpetas, bloques_de_validación, tokenizador_de_referencia):
    """Evalúa cada modelo sobre el mismo conjunto fijo. Si el tokenizador no coincide, NO calcula la métrica."""
    filas = []
    huella_de_los_bloques = laboratorio.huella_de_json(bloques_de_validación)[:12]
    for etiqueta, carpeta in etiquetas_y_carpetas.items():
        modelo, su_tokenizador = modelo_para_generar(etiqueta, carpeta)
        fila = {"modelo": etiqueta, "pérdida": None, "perplejidad": None, "predicciones": None,
                "bloques": len(bloques_de_validación), "conjunto_fijo": huella_de_los_bloques,
                "comparable_con_la_referencia": "sí"}
        compatible, motivo = tokenizadores_compatibles(su_tokenizador, tokenizador_de_referencia)
        if compatible:
            medida = pérdida_en_bloques(modelo, bloques_de_validación)
            fila["pérdida"] = medida["pérdida"]
            fila["perplejidad"] = medida["perplejidad"]
            fila["predicciones"] = medida["predicciones"]
        else:
            fila["comparable_con_la_referencia"] = f"NO: {motivo}; no se calculó"
        filas.append(fila)
    return pd.DataFrame(filas)


def construir_índice_de_ngramas(documentos_a_indexar, un_tokenizador, n):
    """Diccionario {n-grama de tokens: [(id del documento, posición)]} con todos los documentos incluidos."""
    índice = {}
    for documento in documentos_a_indexar:
        if not documento["incluido"]:
            continue
        identificadores = un_tokenizador(documento["texto"])["input_ids"]
        for posición in range(len(identificadores) - n + 1):
            clave = tuple(identificadores[posición:posición + n])
            índice.setdefault(clave, []).append((documento["id"], posición))
    return índice


def coincidencias_con_el_corpus(texto_generado, un_tokenizador, índice, n):
    """Tramos del texto generado que aparecen idénticos (en tokens) en el corpus.

    Devuelve un diccionario con la lista de tramos y la proporción de tokens del texto que cubren.
    """
    identificadores = un_tokenizador(texto_generado)["input_ids"]
    cubiertos = [False] * len(identificadores)
    orígenes_por_posición = {}
    for posición in range(len(identificadores) - n + 1):
        clave = tuple(identificadores[posición:posición + n])
        if clave in índice:
            orígenes_por_posición[posición] = índice[clave]
            for k in range(posición, posición + n):
                cubiertos[k] = True
    tramos = []
    posición = 0
    while posición < len(identificadores):
        if cubiertos[posición]:
            fin = posición
            while fin < len(identificadores) and cubiertos[fin]:
                fin += 1
            documentos_del_tramo = set()
            primera = None
            for inicio_de_coincidencia in orígenes_por_posición:
                if posición <= inicio_de_coincidencia < fin:
                    if primera is None or inicio_de_coincidencia < primera:
                        primera = inicio_de_coincidencia
                    for origen in orígenes_por_posición[inicio_de_coincidencia]:
                        documentos_del_tramo.add(origen[0])
            tramos.append({
                "texto": un_tokenizador.decode(identificadores[posición:fin]),
                "tokens": fin - posición,
                "documentos": sorted(documentos_del_tramo),
                "ejemplo_de_origen": orígenes_por_posición[primera][0],
            })
            posición = fin
        else:
            posición += 1
    cantidad_cubierta = sum(cubiertos)
    return {"tramos": tramos, "proporción_cubierta": cantidad_cubierta / len(identificadores) if identificadores else 0.0,
            "tokens": len(identificadores), "n": n}


CRITERIOS_DE_LECTURA = [
    "Mantiene el tema del comienzo",
    "Repite frases o estructuras",
    "Usa vocabulario que reconozco del corpus",
    "Rompe la gramática o la concordancia",
    "Introduce nombres propios o lugares",
    "Termina abruptamente o se corta",
]


def armar_ficha_de_lectura():
    """Casillas de lectura cualitativa y un campo libre. Guarda en la bitácora con el tipo 'observado'."""
    estación = laboratorio.nueva_estación("6c-ficha")
    bitácora = bitácora_de_comparaciones
    texto_leído = estación.agregar(widgets.Textarea(rows=4, layout=widgets.Layout(width="100%")))
    texto_leído.description = "Texto leído"
    texto_leído.style = {"description_width": "initial"}
    opciones_de_experimento = [("(ninguno)", None)]
    for nombre_del_experimento in nombres_de_experimentos():
        opciones_de_experimento.append((nombre_del_experimento, nombre_del_experimento))
    experimento = estación.agregar(widgets.Dropdown(options=opciones_de_experimento, value=None))
    laboratorio.control(experimento, "Experimento al que se refiere", "experimento", tipo="registro",
                        ayuda="Si elegís uno, las dos entradas quedan asociadas a él y la estación 7 las copia a su carpeta.")
    casillas = []
    for criterio in CRITERIOS_DE_LECTURA:
        casillas.append(estación.agregar(widgets.Checkbox(value=False, description=criterio, indent=False)))
    libre = estación.agregar(widgets.Textarea(rows=3, layout=widgets.Layout(width="100%")))
    libre.description = "En mis palabras"
    interpretación = estación.agregar(widgets.Textarea(rows=3, layout=widgets.Layout(width="100%")))
    interpretación.description = "Interpretación"
    botón_guardar = estación.agregar(laboratorio.botón("Guardar ficha", "Guarda observación e interpretación por separado", "success"))
    aviso = estación.agregar(widgets.HTML())

    def al_guardar(_):
        marcadas = []
        for casilla in casillas:
            if casilla.value:
                marcadas.append(casilla.description)
        referidos = []
        if experimento.value:
            referidos.append(experimento.value)
        try:
            bitácora.anotar("6c-ficha", "observado", libre.value,
                            contexto={"texto_leído": texto_leído.value, "criterios_marcados": marcadas},
                            experimentos=referidos)
            if interpretación.value.strip():
                bitácora.anotar("6c-ficha", "interpretación", interpretación.value,
                                contexto={"texto_leído": texto_leído.value}, experimentos=referidos)
        except ValueError as error:
            aviso.value = utils.html_aviso(str(error), "error")
            return
        aviso.value = utils.html_aviso("Ficha guardada.", "ok")

    estación.al_hacer_clic(botón_guardar, al_guardar)
    return widgets.VBox([texto_leído, laboratorio.fila_de_control(experimento)] + casillas + [libre, interpretación, botón_guardar, aviso])

#### Ficha de lectura

Pegá un texto que haya escrito un modelo y marcá lo que ves: las casillas **describen**, no puntúan. Después escribí con tus palabras una **observación** y, aparte, una **interpretación**. Esa separación entre lo que ves y lo que creés que lo explica es la que después te deja revisar si tus conclusiones tenían evidencia. Si elegís un experimento, las dos entradas quedan asociadas a él por su nombre y su identificador, y la estación 7 las copia a su carpeta.

In [ ]:
display(armar_ficha_de_lectura())

#### Medir con números

La **pérdida** y la **perplejidad** (ver estación 5) se calculan sobre un conjunto fijo: los bloques de validación de la estación 3. Para cada modelo, se predice cada token a partir de los anteriores; la pérdida es el promedio de `−log p(token real)` y la perplejidad es `exp(pérdida)`. Comparar modelos sirve **solo si** tienen el mismo tokenizador, miden el mismo conjunto de bloques y usan el mismo largo de bloque. Por eso `evaluar_modelos` compara primero los tokenizadores y, si no coinciden, **no calcula nada**: mostrar un número que no se puede comparar es peor que no mostrarlo.

Lo que estos números no permiten concluir: no miden calidad literaria; con dos cuentos de validación son una estimación ruidosa; la perplejidad tiene un mínimo de 1, no de 0; y si cambia el tokenizador, la cantidad de tokens cambia y los números no son equivalentes.

### 6C. Vía con código: evaluación

Evaluar el modelo base y todos los checkpoints del experimento activo sobre los bloques de validación. Carga checkpoints, así que necesita `PERMITIR_ANALIZAR_CHECKPOINTS = True`.

### ✎ Antes de ejecutar

¿Qué orden esperás entre el modelo base y los checkpoints de los pasos 30 y 60? ¿Y entre estos dos últimos?

In [ ]:
carpeta_activa = carpeta_del_experimento_activo(EXPERIMENTO_A_USAR)
if PERMITIR_ANALIZAR_CHECKPOINTS and carpeta_activa is not None:
    disponibles = listar_modelos_disponibles()
    a_evaluar = {"base": None}
    for etiqueta, carpeta in disponibles.items():
        if etiqueta.startswith(carpeta_activa.name):
            a_evaluar[etiqueta] = carpeta

    tabla_de_evaluación = evaluar_modelos(a_evaluar, preparación["bloques_de_validación"]["bloques"], tokenizador)
    display(tabla_de_evaluación)
elif not PERMITIR_ANALIZAR_CHECKPOINTS:
    print('Omitido: esta celda carga checkpoints (unos 500 MB cada uno) y puede escribir en la carpeta del experimento. Poné PERMITIR_ANALIZAR_CHECKPOINTS = True en los parámetros del cuaderno para ejecutarla.')
else:
    print('Todavía no hay un experimento ajustado: no hay checkpoints para evaluar.')

**Mirá el resultado.** Una fila por modelo, todas sobre el mismo conjunto fijo.

- **Observación verificable:** la pérdida y la perplejidad de cada modelo; la perplejidad siempre es mayor que 1.
- **Interpretación posible:** si bajan a medida que avanza el ajuste, el modelo les asigna más probabilidad a esos dos cuentos reservados.
- **Límite:** es una medida sobre dos cuentos de un mismo autor. No dice nada sobre cuán «buenos» son los textos generados ni sobre otros autores.

**Buscar coincidencias con el corpus.** ¿Hay fragmentos de lo que generó el modelo que aparecen idénticos en el corpus? El método: se tokeniza el corpus y se arma un índice de n-gramas (secuencias de `n` tokens seguidos y dónde aparecen); después se mira cada secuencia de `n` tokens del texto generado y se agrupan en tramos continuos las que están en el índice. `n` es la longitud mínima de coincidencia, en **tokens**, y cambia el resultado de manera decisiva, por eso está a la vista.

- Una coincidencia **no es plagio**: puede ser una frase corriente del español. Con `n` pequeño (4) coinciden frases comunes; con `n` grande (12 o más) empiezan a ser sospechosas de memorización.
- La **ausencia** de coincidencias **no prueba originalidad**: solo detecta repeticiones exactas de tokens, no paráfrasis.
- Solo busca en el corpus de trabajo; no sabe nada de los textos con que se entrenó el modelo base.
- Los cuentos **reservados** (`adulto_07`, `infantil_04`) están en el índice, pero el ajuste no los vio: una coincidencia con ellos no puede venir del ajuste.

### 6C. Vía con código: coincidencias

Generar con el último checkpoint del experimento activo y buscar sus coincidencias con el corpus, con `n = 8`. Necesita `PERMITIR_ANALIZAR_CHECKPOINTS = True`.

### ✎ Antes de ejecutar

¿Esperás coincidencias en una generación de 120 tokens? ¿Cambia tu respuesta si `n` sube a 16?

In [ ]:
N_DE_COINCIDENCIA = 8
carpeta_activa = carpeta_del_experimento_activo(EXPERIMENTO_A_USAR)
if PERMITIR_ANALIZAR_CHECKPOINTS and carpeta_activa is not None:
    disponibles = listar_modelos_disponibles()
    etiqueta_del_ajustado = etiqueta_del_último_checkpoint(carpeta_activa.name, disponibles)
    modelo_ajustado, tokenizador_ajustado = modelo_para_generar(etiqueta_del_ajustado, disponibles[etiqueta_del_ajustado])

    generadas = generar(modelo_ajustado, tokenizador_ajustado, "Una mañana de otoño,", n_muestras=3, max_new_tokens=120, semilla=11)
    índice = construir_índice_de_ngramas(documentos, tokenizador, N_DE_COINCIDENCIA)

    for muestra in generadas:
        informe = coincidencias_con_el_corpus(muestra["continuación"], tokenizador, índice, N_DE_COINCIDENCIA)
        print(f"Tramos: {len(informe['tramos'])} · cubren {utils.en_porcentaje(informe['proporción_cubierta'])} de {informe['tokens']} tokens")
        for tramo in informe["tramos"]:
            print(f"   {tramo['tokens']} tokens, en {tramo['documentos']}: {tramo['texto'][:90]!r}")
elif not PERMITIR_ANALIZAR_CHECKPOINTS:
    print('Omitido: esta celda carga checkpoints (unos 500 MB cada uno) y puede escribir en la carpeta del experimento. Poné PERMITIR_ANALIZAR_CHECKPOINTS = True en los parámetros del cuaderno para ejecutarla.')
else:
    print('Todavía no hay un experimento ajustado: no hay un modelo ajustado para buscar coincidencias.')

**Mirá el resultado.** Si no hay tramos, no se encontró ninguna secuencia de 8 tokens idéntica a una del corpus. Si hay, mirá si es una frase corriente o un pasaje reconocible.

- **Observación verificable:** cantidad de tramos y porcentaje del texto que cubren.
- **Interpretación posible:** pocas coincidencias con `n = 8` indican que el modelo recombina más de lo que copia, en estas tres muestras.
- **Límite:** no detecta paráfrasis ni dice nada de textos fuera del corpus. Que no haya coincidencias no demuestra originalidad, y que las haya no prueba plagio.

### ✎ Para pensar

Si el modelo reproduce 30 tokens seguidos de un cuento, ¿qué hipótesis sobre el ajuste (pocos datos, muchos pasos, tasa alta) te parece más plausible y cómo la pondrías a prueba?

### ✎ Práctica

Dejá el registro final de la comparación: una **observación**, una **interpretación**, una **limitación** y, si te quedó algo por probar, una **hipótesis pendiente**.

In [ ]:
display(armar_bitácora_visual(estación_inicial="6c-evaluación", lugar="6"))

---
## Estación 7. Guardá el experimento

**Desafío.** Si cerrás el cuaderno hoy, ¿qué hace falta para que otra persona (o vos en un mes) entienda qué se hizo, vuelva a cargar el modelo y reconstruya la comparación?

Todo se guarda en la carpeta del experimento, en formatos abiertos y legibles:

| Archivo o carpeta | Qué contiene |
|---|---|
| `configuración.json` | Todas las decisiones, la huella de la configuración y la hora |
| `manifiesto_del_corpus.json` | Metadatos y huellas de cada documento, originales y procesados |
| `corpus/` y `corpus/originales/` | Los textos usados y los originales sin tocar |
| `partición.json` | Qué documentos fueron entrenamiento y cuáles validación |
| `métricas.jsonl` | Un renglón por paso de entrenamiento y por evaluación |
| `muestras.jsonl` | Las muestras con comienzos fijos durante el ajuste |
| `checkpoints/paso_XXXXX/` | Pesos y tokenizador; solo algunos con estado de optimizador |
| `anotaciones.jsonl` | Tus anotaciones, con su tipo |
| `entorno.json` | Versiones de Python y librerías, sistema y dispositivos |
| `resultado.json` | Cómo terminó el ajuste |
| `FICHA.md` | Un resumen legible de todo lo anterior |

**Cargar para generar y reanudar no son lo mismo.** Para generar texto sirve cualquier checkpoint. Para reanudar el ajuste hace falta uno con el estado del optimizador y del programador: los intermedios no lo tienen; el último y el de una detención, sí. Nada de lo guardado depende de que los controles existan.

**Preparación · estación 7.** La celda siguiente define cómo se lee un experimento guardado, cómo se reanuda y cómo se arma la ficha. Su código aparece plegado si tu entorno lo permite; no hace falta leerlo.

In [ ]:
def describir_checkpoint(carpeta):
    """Dice qué contiene un checkpoint y si sirve para generar, para reanudar o para ambas cosas."""
    ficha = laboratorio.leer_json(Path(carpeta) / "checkpoint.json")
    return {
        "paso": ficha["paso"],
        "sirve_para_generar": ficha["contiene_pesos"],
        "sirve_para_reanudar": ficha["se_puede_reanudar_el_ajuste"],
        "estados_de_azar_guardados": ficha.get("estados_de_azar_guardados", []),
        "dispositivo_del_modelo": ficha.get("dispositivo_del_modelo", "no registrado"),
        "carpeta": str(carpeta),
    }


def configuración_desde_archivo(carpeta_del_experimento):
    """Reconstruye la Configuración de un experimento guardado a partir de su configuración.json."""
    datos = laboratorio.leer_json(Path(carpeta_del_experimento) / "configuración.json")
    argumentos = {}
    for campo in dataclasses.fields(Configuración):
        nombre = campo.name
        if nombre not in datos and campo.default is not dataclasses.MISSING:
            valor = campo.default  # experimentos viejos, anteriores a las huellas de datos
        else:
            valor = datos[nombre]
        argumentos[nombre] = tuple(valor) if isinstance(valor, list) else valor
    return Configuración(**argumentos)


def cargar_experimento(carpeta_del_experimento):
    """Lee un experimento guardado SIN usar widgets ni el estado de la sesión.

    Devuelve un diccionario con la configuración, el resultado, el historial, las anotaciones,
    los checkpoints (con lo que contiene cada uno) y el manifiesto del corpus.
    """
    carpeta = Path(carpeta_del_experimento)
    if not (carpeta / "configuración.json").exists():
        raise FileNotFoundError(f"{carpeta} no parece un experimento guardado (falta configuración.json).")
    checkpoints = []
    for subcarpeta in sorted((carpeta / "checkpoints").glob("paso_*")):
        if (subcarpeta / "checkpoint.json").exists():
            checkpoints.append(describir_checkpoint(subcarpeta))
    resultado = None
    if (carpeta / "resultado.json").exists():
        resultado = laboratorio.leer_json(carpeta / "resultado.json")
    return {
        "configuración": configuración_desde_archivo(carpeta),
        "resultado": resultado,
        "historial": leer_historial(carpeta),
        "anotaciones": laboratorio.RegistroJSONL(carpeta / "anotaciones.jsonl").leer(),
        "checkpoints": checkpoints,
        "manifiesto_del_corpus": laboratorio.leer_json(carpeta / "manifiesto_del_corpus.json"),
        "entorno": laboratorio.leer_json(carpeta / "entorno.json"),
    }


def reanudar_experimento(carpeta_del_experimento, preparación, tokenizador, documentos, **funciones_de_aviso):
    """Continúa un ajuste detenido desde su último checkpoint CON estado de reanudación.

    Retoma pesos, optimizador, programador, el azar de cada dispositivo guardado y la posición en los
    datos. Termina cuando llega a los pasos totales de la configuración original. Antes de seguir,
    `entrenar` comprueba que los datos sean los de la configuración y los del checkpoint. No promete una
    continuación idéntica bit a bit a un ajuste sin interrupciones.
    """
    experimento = cargar_experimento(carpeta_del_experimento)
    reanudables = []
    for checkpoint in experimento["checkpoints"]:
        if checkpoint["sirve_para_reanudar"]:
            reanudables.append(checkpoint)
    if not reanudables:
        raise ValueError("Este experimento no tiene ningún checkpoint con estado de optimizador: solo se puede cargar para generar.")
    último = reanudables[-1]
    configuración_guardada = experimento["configuración"]
    if último["paso"] >= configuración_guardada.pasos_totales:
        raise ValueError(f"El ajuste ya llegó a su último paso ({configuración_guardada.pasos_totales}): no hay nada que reanudar.")
    return entrenar(configuración_guardada, preparación, tokenizador, documentos,
                    reanudar_desde=último["carpeta"], **funciones_de_aviso)


def texto_de_la_ficha(carpeta_del_experimento):
    """Arma el texto de FICHA.md a partir de los archivos del experimento."""
    experimento = cargar_experimento(carpeta_del_experimento)
    config = experimento["configuración"]
    validación = experimento["historial"]["validación"]
    manifiesto = experimento["manifiesto_del_corpus"]
    líneas = [f"# Ficha del experimento: {config.nombre}", "",
              f"- Creada: {laboratorio.leer_json(Path(carpeta_del_experimento) / 'configuración.json').get('creada')}",
              f"- Huella de la configuración: `{config.huella()}`",
              f"- Estado: {experimento['resultado']['estado'] if experimento['resultado'] else 'sin resultado'}", "",
              "## Hipótesis y notas", "", config.hipótesis or "(sin hipótesis)", "", config.notas or "", "",
              "## Modelo base", "",
              f"- Identificación: `{config.modelo_base}`, revisión `{config.revisión_del_modelo}`.",
              "- Licencia declarada: MIT. Tamaño: unos 124 millones de parámetros. Ver `docs/MODELO.md`.", "",
              "## Corpus", "", f"- Huella: `{config.huella_del_corpus}`", f"- Partición: {config.modo_de_partición}; validación: {', '.join(config.ids_de_validación)}",
              f"- Huellas de los datos: preparación `{config.huella_de_la_preparación[:16] or 'no registrada'}`, "
              f"tokenizador `{config.huella_del_tokenizador[:16] or 'no registrada'}`, "
              f"bloques de entrenamiento `{config.huella_de_los_bloques_de_entrenamiento[:16] or 'no registrada'}`, "
              f"bloques de validación `{config.huella_de_los_bloques_de_validación[:16] or 'no registrada'}`.", ""]
    for ficha in manifiesto["documentos"]:
        if ficha["incluido"]:
            líneas.append(f"- `{ficha['id']}` · {ficha['título']} ({ficha['autoría']}) · uso: {ficha['condiciones_de_uso']}")
    líneas += ["", "## Parámetros", "",
               f"- Tasa de aprendizaje {config.tasa_de_aprendizaje}; microlote {config.tamaño_de_lote} × acumulación {config.acumulación_de_gradientes} = lote efectivo {config.lote_efectivo}.",
               f"- {config.pasos_totales} pasos ({config.pasos_por_época} por época; en cada época quedan afuera {config.bloques_que_se_dejan_afuera_por_época} de {config.bloques_de_entrenamiento} bloques); largo de bloque {config.largo_de_bloque}; semilla {config.semilla}.", "",
               "## Resultados observados", ""]
    if validación:
        primera, última = validación[0], validación[-1]
        líneas.append(f"- Pérdida de validación: {primera['pérdida']:.3f} (paso {primera['paso']}) → {última['pérdida']:.3f} (paso {última['paso']}).")
        líneas.append(f"- Perplejidad de validación: {primera['perplejidad']:.1f} → {última['perplejidad']:.1f}.")
    líneas += ["", "## Anotaciones", ""]
    for tipo in laboratorio.TIPOS_DE_ANOTACIÓN:
        líneas.append(f"### {tipo.capitalize()}")
        hubo_alguna = False
        for anotación in experimento["anotaciones"]:
            if anotación["tipo"] == tipo:
                líneas.append(f"- {anotación['texto']}")
                hubo_alguna = True
        if not hubo_alguna:
            líneas.append("- (ninguna)")
        líneas.append("")
    líneas += ["## Límites", "",
               "- La validación es un conjunto chico; la pérdida es ruidosa y no mide calidad literaria.",
               "- Un ajuste corto con poco texto muestra el mecanismo, no un modelo útil.",
               "- Las coincidencias textuales con el corpus no prueban plagio ni su ausencia prueba originalidad.",
               "- El determinismo no está garantizado entre entornos; ver `entorno.json`.", ""]
    return "\n".join(líneas)


def exportar_experimento(carpeta_del_experimento, bitácoras=None):
    """Completa la carpeta del experimento: copia sus anotaciones y escribe FICHA.md. Devuelve la lista de archivos.

    `bitácoras` es una lista de laboratorio.Bitácora. Se copia una anotación si declara, en `experimentos`,
    el nombre de esta carpeta. Se la reconoce por su `id`: exportar dos veces no la duplica, y dos
    anotaciones con el mismo texto o la misma hora no se confunden. Las anotaciones viejas, sin `id`, no se copian.
    """
    carpeta = Path(carpeta_del_experimento)
    if bitácoras is not None:
        anotaciones_del_experimento = laboratorio.RegistroJSONL(carpeta / "anotaciones.jsonl")
        ids_ya_copiados = set()
        for existente in anotaciones_del_experimento.leer():
            ids_ya_copiados.add(existente.get("id"))
        for bitácora in bitácoras:
            for entrada in bitácora.de_experimento(carpeta.name):
                if entrada.get("id") not in ids_ya_copiados:
                    anotaciones_del_experimento.agregar(entrada)
                    ids_ya_copiados.add(entrada["id"])
    (carpeta / "FICHA.md").write_text(texto_de_la_ficha(carpeta), encoding="utf-8")
    archivos = []
    for ruta in sorted(carpeta.rglob("*")):
        if ruta.is_file() and "originales" not in ruta.parts and "corpus" not in ruta.parts:
            archivos.append((str(ruta.relative_to(carpeta)), ruta.stat().st_size))
    return archivos

### Bitácora de la sesión

Un lugar único donde registrar lo que pensás mientras trabajás. Cada entrada tiene una **estación**, un **tipo** (observado, interpretación, limitación o hipótesis pendiente), tu **predicción** (si la hiciste) y el texto. Se guarda en `experimentos/bitácora.jsonl`, y funciona con y sin controles. Cada anotación recibe un identificador propio y, si indicás un experimento, queda asociada a él: no se asocia por la hora ni por el texto.

Revisá lo que anotaste en las estaciones anteriores y sumá lo que falte.

In [ ]:
display(armar_bitácora_visual(estación_inicial="7-guardado", lugar="7"))

### Vía con código (alternativa al panel): recargar y reconstruir la comparación

Leer el experimento que acabás de ajustar desde su carpeta (como si abrieras el cuaderno otro día) y volver a comparar el modelo base con su checkpoint final, sin ningún control. La segunda celda necesita `PERMITIR_ANALIZAR_CHECKPOINTS = True`.

Antes de ejecutar, apostá: ¿qué va a cambiar respecto de la comparación de la estación 6 si usás otra semilla? ¿Y si usás la misma?

In [ ]:
carpeta_activa = carpeta_del_experimento_activo(EXPERIMENTO_A_USAR)
if carpeta_activa is not None:
    experimento = cargar_experimento(carpeta_activa)

    print(f"Experimento: {experimento['configuración'].nombre} · huella {experimento['configuración'].huella()[:12]}")
    print(f"Estado: {experimento['resultado']['estado']} en el paso {experimento['resultado']['paso_final']}")
    for checkpoint in experimento["checkpoints"]:
        print(f"  paso {checkpoint['paso']:>3}: generar={checkpoint['sirve_para_generar']}, reanudar={checkpoint['sirve_para_reanudar']}")
else:
    print('Todavía no hay un experimento ajustado para recargar.')

**Mirá el resultado.** Cada checkpoint informa por separado si sirve para generar y si sirve para reanudar. Ahora se carga el último desde su carpeta.

In [ ]:
if PERMITIR_ANALIZAR_CHECKPOINTS and carpeta_activa is not None:
    último_checkpoint = experimento["checkpoints"][-1]["carpeta"]
    etiqueta_recargada = f"{experimento['configuración'].nombre} · recargado"

    comparación_recargada = comparar_generaciones(
        etiquetas_y_carpetas={"base": None, etiqueta_recargada: Path(último_checkpoint)},
        comienzos=["Una mañana de otoño,"],
        n_muestras=2,
        max_new_tokens=40,
        semilla=7,
    )
    for resultado in comparación_recargada:
        print(f"— {resultado['etiqueta']}")
        for muestra in resultado["muestras"]:
            print("    ", muestra["continuación"][:100].replace("\n", " "))
elif not PERMITIR_ANALIZAR_CHECKPOINTS:
    print('Omitido: esta celda carga checkpoints (unos 500 MB cada uno) y puede escribir en la carpeta del experimento. Poné PERMITIR_ANALIZAR_CHECKPOINTS = True en los parámetros del cuaderno para ejecutarla.')
else:
    print('Todavía no hay un experimento ajustado para recargar.')

**Mirá el resultado.** El modelo se cargó desde el disco, no desde la memoria de la sesión.

- **Observación verificable:** con las mismas semillas que usaste en la estación 6A, el modelo recargado debería dar las mismas continuaciones en esta máquina.
- **Interpretación posible:** si coinciden, los pesos guardados son los del modelo que evaluaste; si no, algo cambió (dispositivo, versión, semillas).
- **Límite:** la coincidencia vale en este entorno; no hay garantía en otra máquina.

<details>
<summary>Profundización: reanudar un ajuste</summary>

`reanudar_experimento` exige un checkpoint con `sirve_para_reanudar`; si no lo hay, el error lo dice. Usa la **configuración guardada** (no la de la sesión) y retoma la corrida que se interrumpió, **si los datos que le pasás son los mismos**: `entrenar` compara las huellas de la configuración y del checkpoint con las de tu `preparación` y tus `documentos`, y se niega, nombrando qué difiere, si cambió el corpus, la partición, el tokenizador o los bloques. Para extender un ajuste más allá de sus pasos originales no se reanuda: se crea una configuración nueva que parta de los pesos guardados.
</details>

### ✎ Para pensar

¿Qué parte de `entorno.json` te parece más importante para explicar una diferencia entre dos máquinas?

### Exportar y documentar

Completar la carpeta del experimento activo con las anotaciones que se refieren a él y con la ficha, y listar su contenido. Escribe archivos en esa carpeta, así que necesita `PERMITIR_ANALIZAR_CHECKPOINTS = True`.

La ficha junta datos de varios archivos en un resumen legible y **no inventa nada**: cada línea sale de un archivo que ya existe. Separa lo observado, las anotaciones por tipo y los límites. Las anotaciones se copian por su `id`, así que exportar dos veces no duplica nada.

**Advertencia:** la carpeta del experimento contiene una copia del corpus. Si usaste textos con derechos vigentes (`datos/textos`), no la publiques ni la subas a un servicio externo. Nada de este cuaderno publica ni sube archivos por su cuenta.

In [ ]:
if PERMITIR_ANALIZAR_CHECKPOINTS and carpeta_activa is not None:
    archivos = exportar_experimento(carpeta_activa, [bitácora_general, bitácora_de_comparaciones])
    for nombre, tamaño in archivos:
        print(f"{tamaño / 1e6:9.1f} MB  {nombre}")
    print()
    print((carpeta_activa / "FICHA.md").read_text(encoding="utf-8")[:900])
elif not PERMITIR_ANALIZAR_CHECKPOINTS:
    print('Omitido: esta celda carga checkpoints (unos 500 MB cada uno) y puede escribir en la carpeta del experimento. Poné PERMITIR_ANALIZAR_CHECKPOINTS = True en los parámetros del cuaderno para ejecutarla.')
else:
    print('Todavía no hay un experimento ajustado para exportar.')

**Mirá el resultado.** La lista muestra qué ocupa lugar: los checkpoints son casi todo el peso, y los que tienen estado de optimizador pesan unas tres veces lo que uno solo de pesos. La ficha resume el experimento con sus huellas, sus condiciones de uso y la separación entre lo observado y lo anotado, pero no reemplaza los archivos: el experimento se reconstruye desde `configuración.json`, el manifiesto y los checkpoints.

---
## Cierre

### Volver al desafío

*¿Podemos hacer que un modelo escriba un poco más como el autor que elegimos?* Con lo que hiciste tenés material para responder con cuidado, y también para reconocer hasta dónde llega la respuesta:

| Podés afirmar (con el registro a mano) | No podés afirmar |
|---|---|
| Que la pérdida de validación de un conjunto fijo cambió entre dos modelos | Que el modelo ajustado escribe «mejor» |
| Qué diferencias concretas viste en las muestras comparadas | Que esas diferencias valen en general, con otros comienzos o semillas |
| Qué tokens candidatos propone cada modelo en un punto dado | Qué «entiende» o «quiere decir» el modelo |
| Qué fragmentos del texto generado coinciden con el corpus | Que no haya copia (ni que la haya) |

### Limitaciones del método

- **Corpus chico y de un solo autor.** El ajuste muestra el mecanismo; un modelo realmente adaptado necesitaría mucho más texto y más pasos.
- **Validación mínima.** Dos cuentos reservados dan una estimación ruidosa.
- **Determinismo acotado.** Las semillas reproducen resultados en el mismo entorno, no necesariamente en otro.
- **Modelo base.** Se entrenó con unos 11,5 GB de texto no filtrado, cuyas fuentes concretas no conocemos.

### Conexión con lo que sigue

Los modelos de esta unidad son pequeños y en nuestras pruebas se ajustaron en minutos. Los asistentes actuales parten de la misma idea (modelo causal, tokens, pérdida) con escalas enormes y etapas adicionales de entrenamiento. Conviene volver una vez más a la pregunta de siempre: **¿qué evidencia tenemos de lo que cambió?**